In [ ]:
# Paths, relative to the folder containing this notebook (code/).
# Inputs are read from ../data/, figure data are written to ../figure_data/,
# and all other outputs (figures, peak catalogs) to ../outputs/.
import os
from pathlib import Path

_HERE = Path.cwd()
DB_PATH = str((_HERE / "../data/blog_comments_deidentified.db").resolve())
GENDER_CSV_PATH = str((_HERE / "../data/author_inferred_gender.csv").resolve())
FIGURE_DATA_DIR = str((_HERE / "../figure_data").resolve())
_OUT = (_HERE / "../outputs").resolve()
_OUT.mkdir(exist_ok=True)
os.chdir(_OUT)


# De-duplicated copy of `moggablogg_fixed_final (copy2).ipynb`

Changes (search for `FIX (dedup copy)`):
1. `run_pipeline()` no longer merges blog topics onto `comments`. The merge copied each comment once per blog topic (~2.9x), inflating every count and weighting comments by their blog's number of topics.
2. Empty comments (6,097) are dropped in `load_comments()` (toggle `DROP_EMPTY_COMMENTS`).
3. `build_gen_from_specs()` keeps one row per (comment, bucket).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns  # or plotnine if you want ggplot-style
from matplotlib.ticker import PercentFormatter
from matplotlib import cm  # for viridis
from statsmodels.nonparametric.smoothers_lowess import lowess
from patsy import dmatrix, cr

import re
import sqlite3
from datetime import datetime

import statsmodels.api as sm
import statsmodels.formula.api as smf

# For Excel
import openpyxl

# For categorical factors
import numpy as np

from matplotlib.cm import get_cmap
from matplotlib import patheffects as pe

In [ ]:
from __future__ import annotations

import sqlite3
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess


# =========================
# Section 1 — Core pipeline
# =========================

# FIX (dedup copy): set to False to keep empty comments
DROP_EMPTY_COMMENTS = True

def connect_sqlite(filename: str) -> sqlite3.Connection:
    return sqlite3.connect(filename)

def list_tables(conn: sqlite3.Connection) -> List[str]:
    q = "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
    return pd.read_sql(q, conn)["name"].tolist()

def load_comments(conn: sqlite3.Connection) -> pd.DataFrame:
    comments = pd.read_sql("SELECT * FROM comments;", conn)

    # FIX (dedup copy): drop comments with empty/whitespace-only text (6,097 rows).
    # They carry model scores but contain nothing to annotate.
    if DROP_EMPTY_COMMENTS:
        empty = comments["comment_text"].fillna("").str.strip().eq("")
        print(f"Dropping {int(empty.sum())} empty comments")
        comments = comments.loc[~empty].copy()

    # 👇 Normalize the key once and for all
    if "comment_id" not in comments.columns and "id" in comments.columns:
        comments = comments.rename(columns={"id": "comment_id"})
    # (optional) if both exist and are identical, drop the duplicate
    if "comment_id" in comments.columns and "id" in comments.columns:
        try:
            if (comments["comment_id"] == comments["id"]).all():
                comments = comments.drop(columns=["id"])
        except Exception:
            pass  # if types differ, just keep both

    # ... rest of your existing parsing (dt, dt_num, ranks, month/weekday) ...
    comments["dt"] = pd.to_datetime(
        comments["comment_datetime"], format="%Y-%m-%dT%H:%M:%S", errors="coerce"
    )
    ts_ns = comments["dt"].view("int64")
    comments["dt_num"] = (ts_ns / 1e9).astype("float")
    mn = comments["dt_num"].min(skipna=True); mx = comments["dt_num"].max(skipna=True)
    comments["dt_num"] = (comments["dt_num"] - mn) / (mx - mn) if (pd.notna(mn) and pd.notna(mx) and mx > mn) else 0.0
    comments["comment_rank"] = comments.groupby("author_name")["dt_num"].rank(method="average", na_option="keep")
    comments["n.comment"] = comments.groupby("author_name")["dt_num"].transform("size")
    comments["month"] = comments["dt"].dt.month_name()
    comments["weekday"] = comments["dt"].dt.day_name()
    return comments


In [ ]:
def single_comment_firsttox(comments: pd.DataFrame) -> pd.DataFrame:
    if "toxicity" not in comments.columns:
        raise KeyError("`toxicity` column not found in `comments`.")
    if "n.comment" not in comments.columns:
        raise KeyError("`n.comment` column not found; call `load_comments()` first.")
    return (
        comments.loc[comments["n.comment"] == 1, ["author_name", "toxicity"]]
        .rename(columns={"toxicity": "firsttox"})
        .drop_duplicates()
        .reset_index(drop=True)
    )

def attach_gender(comments: pd.DataFrame, gender_csv_path: str) -> pd.DataFrame:
    # author_inferred_gender.csv has exactly two columns: author_name, gender
    gender = pd.read_csv(gender_csv_path, header=None, names=["author_name", "newgender"])
    out = comments.merge(gender, on="author_name", how="left")
    out["newgender"] = out["newgender"].fillna("unknown")
    cats = ["male"] + sorted([c for c in out["newgender"].dropna().unique() if c != "male"])
    out["newgender"] = pd.Categorical(out["newgender"], categories=cats, ordered=True)
    return out

def build_blog_topic_names(blog_topics: pd.DataFrame, topics: pd.DataFrame, top_n: int = 9) -> pd.DataFrame:
    if "blog_id" not in blog_topics.columns or "topic_id" not in blog_topics.columns:
        raise KeyError("`blog_topics` must have columns: blog_id, topic_id.")
    if "id" not in topics.columns or "name" not in topics.columns:
        raise KeyError("`topics` must have columns: id, name.")

    top_ids = blog_topics["topic_id"].value_counts().nlargest(top_n).index.tolist()

    bt = blog_topics.copy()
    bt["id"] = np.where(bt["topic_id"].isin(top_ids), bt["topic_id"], np.nan)

    bt_named = bt.merge(topics[["id", "name"]], on="id", how="left")
    bt_named["name"] = bt_named["name"].fillna("Other")
    return bt_named[["blog_id", "name"]].drop_duplicates().reset_index(drop=True)

def attach_topics(comments: pd.DataFrame, conn: sqlite3.Connection, top_n: int = 9) -> pd.DataFrame:
    blog_topics = pd.read_sql("SELECT * FROM blog_topics;", conn)
    topics = pd.read_sql("SELECT * FROM topics;", conn)
    bt_named = build_blog_topic_names(blog_topics, topics, top_n=top_n)
    return comments.merge(bt_named, on="blog_id", how="left")

def run_pipeline(
    db_path: str = DB_PATH,
    gender_csv_path: str = GENDER_CSV_PATH,
    top_n_topics: int = 9,
) -> Tuple[pd.DataFrame, pd.DataFrame, List[str]]:
    conn = connect_sqlite(db_path)
    try:
        table_names = list_tables(conn)
        comments = load_comments(conn)
        firsttox_df = single_comment_firsttox(comments)
        comments = attach_gender(comments, gender_csv_path)
        # FIX (dedup copy): attach_topics() merged every comment with each of its blog's
        # topics (top 9 + "Other"), copying each comment ~2.9x (804,437 -> 2,355,528 rows).
        # Nothing downstream uses the topic column, so it is no longer merged here.
        # comments = attach_topics(comments, conn, top_n=top_n_topics)
        assert comments["comment_id"].is_unique, "comments must have one row per comment"
    finally:
        conn.close()
    return comments, firsttox_df, table_names


In [ ]:
comments, firsttox, tables = run_pipeline(
    db_path=DB_PATH,
    gender_csv_path=GENDER_CSV_PATH,
    top_n_topics=9,
)
print("Tables:", tables)
print("Comments shape:", comments.shape)
print("Firsttox shape:", firsttox.shape)

In [ ]:
# ===========================================
# Section 2 — Timetrend, weekly agg, plotting
# ===========================================

import pandas as pd
import matplotlib.pyplot as plt

def build_timetrend(comments: pd.DataFrame) -> pd.DataFrame:
    """
    Original function: builds a long/melted timetrend per 'emotion'/task.
    We keep this AS IS so your other plots don't break.
    """
    id_cols = ["dt", "author_name"]
    value_cols = [
        "politeness",
        "emotion_anger",
        "emotion_joy",
        "emotion_fear",
        "toxicity",
        "group_generalization_presence",
        "hate_speech_presence",
    ]

    for col in id_cols + value_cols:
        if col not in comments.columns:
            raise KeyError(f"Required column `{col}` missing from `comments`.")

    tt = (
        comments[id_cols + value_cols]
        .melt(
            id_vars=id_cols,
            value_vars=value_cols,
            var_name="emotion",
            value_name="outcome",
        )
        .copy()
    )

    grp = tt.groupby(["author_name", "emotion"], group_keys=False)
    tt["av"] = grp["outcome"].transform("mean")
    tt["outcome_std"] = tt["outcome"] - grp["outcome"].transform("mean")

    # Date-only + week floor (Monday-start). If you need Sunday-start, use "W-SUN".
    tt["date"] = pd.to_datetime(tt["dt"]).dt.date
    tt["vika"] = pd.to_datetime(tt["date"]).dt.to_period("W-MON").dt.start_time
    return tt


def weekly_agg(timetrend: pd.DataFrame) -> pd.DataFrame:
    """
    Original weekly agg per 'emotion'. We keep this too.
    """
    out = (
        timetrend
        .assign(
            date=pd.to_datetime(timetrend["dt"]).dt.date,
            vika=pd.to_datetime(timetrend["date"]).dt.to_period("W-MON").dt.start_time,
        )
        .groupby(["vika", "emotion"], as_index=False)
        .agg(
            n_comments=("outcome_std", "size"),
            av=("outcome_std", "mean"),
        )
    )
    return out


# -------- NEW: overall weekly counts, ignoring 'emotion'/task --------
def weekly_all_comments(comments: pd.DataFrame) -> pd.DataFrame:
    """
    Count ALL comments per week (no melt, no task/emotion dimension).
    This is what you said you wanted to see.
    """
    df = comments.copy()
    df["dt"] = pd.to_datetime(df["dt"], errors="coerce")
    df["date"] = df["dt"].dt.date
    df["vika"] = pd.to_datetime(df["date"]).dt.to_period("W-MON").dt.start_time
    weekly = (
        df.groupby("vika", as_index=False)
          .agg(n_comments=("dt", "size"))
          .sort_values("vika")
    )
    return weekly

# -------- NEW: plotting the overall weekly counts --------
def plot_all_weekly(weekly_df: pd.DataFrame, outfile: str = "number_of_comments_all.png") -> None:
    if weekly_df.empty:
        print("No rows for overall weekly comments; skipping plot.")
        return

    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

    # just the points
    ax.scatter(
        weekly_df["vika"],
        weekly_df["n_comments"],
        s=10,
        alpha=0.6
    )

    ax.set_xlabel("Time of the comment")
    ax.set_ylabel("Number of comments")
    ax.set_title("Weekly number of comments")
    ax.set_facecolor("white")
    fig.patch.set_facecolor("white")

    plt.tight_layout()
    plt.savefig(outfile, format="png", facecolor="white")
    plt.close(fig)
    print(f"Saved: {outfile}")


# ===== run =====
timetrend = build_timetrend(comments)
tt = weekly_agg(timetrend)

# NEW: overall
weekly_all = weekly_all_comments(comments)
plot_all_weekly(weekly_all, outfile="number_of_comments_all.png")


In [ ]:
import matplotlib.colors as mcolors

def weekly_summary(timetrend: pd.DataFrame) -> pd.DataFrame:
    # expects: vika (datetime-like), emotion, outcome_std
    wk = (
        timetrend
        .groupby(["vika", "emotion"], as_index=False)
        .agg(mu=("outcome_std", "mean"),
             n=("outcome_std", "size"))
    )
    wk["vika"] = pd.to_datetime(wk["vika"])
    return wk

def _blend_with_white(color, alpha=0.20):
    """Simulate transparency by blending color with white."""
    r, g, b, _ = mcolors.to_rgba(color)
    return (
        1 - alpha * (1 - r),
        1 - alpha * (1 - g),
        1 - alpha * (1 - b),
        1.0
    )

def plot_timetrend_spline_mean_ci(
    weekly_df: pd.DataFrame,
    outfile: str = "fig3d.e",
    df_spline: int = 8,
    grid_n: int = 300,
    ci_level: float = 0.95,
    label_map: dict | None = None,
    legend_title: str = "Examined factors",
    palette: str | list | tuple = "tab10",
    color_map: dict | None = None,   # NEW: map emotion→color, e.g. {"anger":"#b2182b"}
):
    required = {"vika", "emotion", "mu", "n"}
    missing = required - set(weekly_df.columns)
    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    emotions = sorted(weekly_df["emotion"].unique())

    # build fallback palette
    if isinstance(palette, str):
        cmap = get_cmap(palette)
        fallback_colors = [cmap(i % cmap.N) for i in range(len(emotions))]
    else:
        pl = list(palette)
        if not pl:
            raise ValueError("Custom palette list is empty.")
        fallback_colors = [pl[i % len(pl)] for i in range(len(emotions))]

    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

    for i, emo_raw in enumerate(emotions):
        df = weekly_df.loc[weekly_df["emotion"] == emo_raw, ["vika", "mu", "n"]].sort_values("vika").copy()

        # stable x
        x_days = (df["vika"] - df["vika"].min()).dt.total_seconds() / 86400.0
        x_mean, x_std = x_days.mean(), x_days.std(ddof=0) or 1.0
        x = (x_days - x_mean) / x_std

        y = df["mu"].to_numpy(float)
        w = df["n"].to_numpy(float)

        X = dmatrix(f"cr(x, df={df_spline})", {"x": x}, return_type="dataframe")
        model = sm.WLS(y, X, weights=w).fit()

        xg = np.linspace(x.min(), x.max(), grid_n)
        Xg = dmatrix(f"cr(x, df={df_spline})", {"x": xg}, return_type="dataframe")
        pred = model.get_prediction(Xg).summary_frame(alpha=1 - ci_level)

        y_hat = pred["mean"].to_numpy()
        y_lo  = pred["mean_ci_lower"].to_numpy()
        y_hi  = pred["mean_ci_upper"].to_numpy()

        xg_days = xg * x_std + x_mean
        x0 = df["vika"].min()
        xg_dt = x0 + pd.to_timedelta(xg_days, unit="D")

        # label + color selection
        nice_label = label_map.get(emo_raw, emo_raw) if label_map else emo_raw
        if color_map and nice_label in color_map:
            color = color_map[nice_label]
        elif color_map and emo_raw in color_map:  # also allow mapping by raw key
            color = color_map[emo_raw]
        else:
            color = fallback_colors[i]

        ribbon_color = _blend_with_white(color, alpha=0.20)
        ax.fill_between(xg_dt, y_lo, y_hi, color=ribbon_color, linewidth=0)
        ax.plot(xg_dt, y_hat, color=color, linewidth=3, label=nice_label)

    #ax.set_xlabel("Time of the comment")
    #ax.set_ylabel("Individually standardized outcome")
    #ax.set_title("Weekly trends")
    ax.set_ylim(-0.15, 0.15)
    ax.grid(True, which="major", linewidth=0.5, alpha=0.4)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)

    ax.margins(x=0)
    
    ax.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, 1.03),
        ncol=7,
        frameon=False,
        title=None,
        columnspacing=1.2,
        handletextpad=0.5,
        fontsize=10
    )
    
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(outfile, format="png", facecolor="white")
    plt.close(fig)
    print(f"Saved: {outfile}")


In [ ]:
wk = weekly_summary(timetrend)

label_map = {
    "politeness" : "Politeness", 
    "emotion_anger": "Anger", 
    "emotion_joy": "Joy",
    "emotion_fear": "Fear",
    "toxicity": "Toxicity", 
    "group_generalization_presence": "Group generalizations", 
    "hate_speech_presence": "Hate speech"
}

color_map = {
    "emotion_fear": "#4477AA", # blue
    "hate_speech_presence": "#EE6677", # red
    "emotion_joy": "#228833", # green
    "politeness" : "#CCBB44", # yellow 
    "toxicity": "#66CCEE", # cyan
    "emotion_anger": "#AA3377", # magenta   
    "group_generalization_presence": "#BBBBBB", # grey 
}
plot_timetrend_spline_mean_ci(
    wk,
    outfile="fig3d_fear.png",
    label_map=label_map,
    color_map=color_map,
    palette="tab10",
    df_spline=8,   # smoothness (6–12 is a good range)
    grid_n=300,    # curve resolution
    ci_level=0.95  # ribbon confidence level
)


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    groups = pd.read_sql("SELECT * FROM comment_group_generalizations;", conn)
    groupn = pd.read_sql("SELECT * FROM generalized_groups;", conn)

In [ ]:
dat = (
    comments
    .drop(columns=["name"], errors="ignore")  # ignore in case 'name' isn't present
    .drop_duplicates()
    .copy()
)

In [ ]:
homosexuality = ['samkynhneigðir karlmenn', 'samkynhneigð börn', 'supporters of same-sex marriage', 'gay', 'lesbía', 
                 'individuals from same-sex households', 'same-sex attracted', 'homma', 'samkynhneigð fólk', 'gays', 
                 'gaypride participants', 'homosexual individuals', 'same-sex couples', 'samkynhneigdir', 'samkynhneigður', 
                 'gay individuals', 'same-sex relationships', 'gay-pride supporters', 'lesbíur', 'lesbians and gays', 
                 'homosexual men', 'gay pride', 'samkynhneigt fólk', 'gay men', 'same-sex oriented people', 
                 'samkynhneigin fólk', 'Samkynhneigðir', 'lesbians', 'hommar', 'Gays', 'samkynhneigðu', 'samkynhneygð', 
                 'homoseksuálum', 'samkynhneigðir karlar', 'lesbian people', 'same-sex families', 'same-sex marriages', 
                 'gay pride advocates', 'hommar og lesbíur', 'gays and lesbians', 'lesbians and gay men', 'homosexuals', 
                 'gay pride participants', 'samkynhneigðu fólki', 'samkynheigð', 'same-sex marriage advocates', 'samkynhneigir', 
                 'samkynhneigðra', 'same-sex individuals', 'gay community', 'lesbian women', 'same-sex attracted individuals', 
                 'samkynhneigðir', 'homosexual people', 'same-sex oriented individuals', 'samkynja fólk', 'gay pride community', 
                 'samkynhneigur', 'homos', 'gay pride leaders', 'Homosexuals', 'karlkyns samkynhneigðir', 'samkynhneygðir', 
                 'ogift samkynhneigð pör', 'samkynhneigðar konur', 'samkynhneigðir (LGBT)', 'homosexuals and lesbians', 
                 'homosexual couples', 'Gay people', 'hommum', 'gay people', 'same-sex orientation', 'gay pride supporters', 
                 'samkynhneigðum', 'samkynhneigðir (gay people)', 'homosexual community', 'same-sex marriage supporters', 
                 'gaypride supporters', 'múslimska homma', 'samkynja pör', 'gaypride-fólks', 'same-sex attracted people', 
                 'people in same-sex relationships', 'samkynheigðir', 'gay pride individuals', 'gaypride-göng-fólk', 
                 'homosexuals and fishermen', 'Homosexuallar', 'samkynhneigða', 'Samkynhneigt fólk', 'lesbías og hommar', 
                 'Epalhommar', 'samkynhneigðir menn', 'supporters of gay pride', 'Gay community', 'gaypride-fólk', 
                 'samkynhneigð pör', 'samkynhneigð']

bisexuality = [
    "bisexuals", "bisexual people", "bisexual individuals",
    "tvíkynhneigðir sjallar",
]

intersex_gender_minorities = [
    "individuals with unknown gender", "individuals with unknown gender/sexual orientation",
    "people with mixed gender characteristics",
    "athletes with differences of sexual development",
    "individuals with intersex conditions",
    "kynvitund", "intersex individuals", "people with intersex variations", "people who use gender-neutral language",
    "people with non-binary or gender-fluid identities", "gender fluid individuals", 
]

other_queer = ['People with different sexual orientations', 'people who support LGBTQIA rights', 'people of various sexual orientations', 
               'debates on sexual orientation', 'hinseginfólk', 'those with differing sexual orientations', 'people with asexuality', 
               'people with certain sexual orientations', 'sexual minorities', 'hinsegin fólki', 'LGBT', 'LGBTQ community', 
               'people who identify as LGBTQ+', 'people with diverse sexual orientations', 'religious individuals opposed to LGBTQ+ rights',
               'Gender and sexual minorities', 'kynvillingar', 'asexual people', 'hinsegin', 'LGBTQ+ asylum seekers', 'LGBTIQ community',
               'individuals supporting LGBTQ+ rights', 'LGBT+ community', 'hinsegin drengir', 'LGBT+ individuals', 
               'people with different sexual orientations', 'individuals with non-heteronormative sexual orientation', 
               'members of the LGBTQ+ community', 'LGBTQ+ community', 'LGBT individuals', 'hinsegin samfélagið', 'LGBTIQ+', 
               'LGBTQ individuals', 'men with unsafe sexual orientation', 'LGBTQ+ communities', 'people with dangerous sexual orientations',
               'people of sexual minorities', 'LGBTQIA+ community', 'LGBTQ+ activists', 'people of different sexual orientations', 
               'LGBTQ+ individuals', 'different sexual orientations', 'Homosexuals', 'HINSEGIN samfélagi', 
               'individuals with a different sexual orientation', 'people who are LGBTQ+', 'hinsegin fólk', 
               'individuals with diverse sexual orientations', 'samkynhneigðir (LGBT)', 'LGBTQ', 'lgbt community', 
               'members of LGBTQ+ community', 'individuals identifying as LGBTQ+', 'marginalized sexual minorities', 
               'asexual individuals', 'LGBTQ+ Community', 'LGBT people', 'LGBTQ youth', 
               'people with non-heteronormative sexual orientations', 'LGBTQ+ people', 'those who identify as LGBTQ+', 
               'LGBTQ+ organizations', 'LGBTQ+ groups', 'lgbt individuals', 'individuals with certain sexual orientations', 
               'LGBTQIA+', 'people who are sexual minorities', 'marginalized sexual orientation', 'lgbtq community', 
               'sexual orientation', 'the LGBT community', 'marginalized sexual orientations', 'LGBTq community', 
               'people with differing sexual orientation', 'LGBTQ+', 'LGBT rights advocates', 'the LGBTQ+ community', 
               'people supporting LGBTQ+ rights', 'individuals with differing sexual orientations', 'people opposing LGBTQ education', 
               'non-heterosexual individuals', 'LGBTIQ', 'individuals with different sexual orientations', 'LGBTQ advocates', 
               'LGBT community', 'people against LGBTQ+ rights']


In [ ]:
trans = [
    "transgender", "trans individuals", "transgender people", "transgender individuals",
    "transgender community", "trans activists", "trans community", "trans women",
    "Trans people", "trans people", "homo/trans fólk", "transfólk", "Trans individuals",
    "people undergoing gender transition", "Trans community",
    "parents of transgender children", "individuals supporting transgender rights",
    "Transgender individuals", "people who are transgender or non-binary",
    "people who want to change gender", "gender identity",
    "people with gender dysphoria", "individuals seeking gender reassignment",
    "gender fluid individuals", "individuals with unclear gender identity",
    "gender non-conforming individuals",
    "people with non-binary or gender-fluid identities",
    "individuals without gender conformity", "gender identities",
    "people with diverse gender identities",
    "advocates for gender identity legislation",
    "people challenging traditional definitions of gender",
    "individuals seeking non-gendered names",
    "individuals with gender dysphoria",
    "individuals with diverse gender identities",
    "gender non-conformists", "gender non-conformers",
    "individuals with non-conforming gender identities",
    "people uncomfortable with their gender identity",
    "individuals with gender identities outside the binary",
    "people with differing gender identities", "gender diverse individuals",
    "gender minorities", "individuals identifying outside traditional gender norms",
    "supporters of gender diversity", "gender nonconforming individuals",
    "individuals identifying with non-binary or gender non-conforming identities",
    "gender-diverse individuals", "people identifying as a different gender",
    "people identifying with different gender identities",
    "gender non-conformity", "individuals questioning gender identity",
    "people with non-binary identities", "people who are transgender or non-binary",
    "non-binary", "non-binary individuals", "non-binary people", "people identifying as non-binary",
]


In [ ]:
physical_disability = [
    # General disability
    "people with disabilities", "disabled", "disabled individuals", "disabled persons",
    "disabled people", "the disabled", "Disabled people", "Disabled", "disability",
    "disabilities", "Disability community", "disability beneficiaries",
    "disability benefit recipients", "disability recipients", "people on disability benefits",
    "people on disability", "individuals on disability", "those receiving pensions or disability benefits",
    "those receiving financial support due to disability", "people receiving disability benefits",
    "adults on disability benefits", "people with physical or mental disabilities",
    "people with mental or physical disabilities", "children with disabilities", "disabled persons",
    "the disabled", "poor and disabled people", "individuals with disabilities", "people with mental disabilities",
    "people with physical or mental disabilities", "people with mental or physical disabilities", "people with temporary disabilities",
    "those with disabilities", "disabilities", "young disabled individuals", "persons with disabilities", "disabled community", 
    "Disabled people", "young disabled people", "disabled children", "Disabled", "disabled Icelanders", "athletes with disabilities",
    "disabled women", "people with disabilities or psychiatric disorders", "the sick and disabled", 
    "individuals with syndromes", "individuals with genetic disorders", "people with genetic disorders", 
    
    # Aldraðir og öryrkjar
    "elderly and disabled people", "elderly and disabled", "those receiving pensions or disability benefits",
    "elders and disabled individuals", "people on disability benefits", "people on disability", "elderly and disabled persons",
    "unemployed or disabled individuals", "elderly and disabled individuals", "the elderly and disabled",
    "those receiving financial support due to disability", "disability beneficiaries", "older adults and disabled individuals",
    "eldra people and disabled", "people receiving disability benefits", "adults on disability benefits", 
    "young disability beneficiaries", "disability benefit recipients", "disabled and elderly", "individuals on disability", 
    "the elderly and the disabled", "individuals with disabilities or the elderly", "disability recipients", 

    # Icelandic terms
    "fatlaðir", "fatlaða", "fatlað fólk", "fatlaða individuals", "fatlaður",
    "fatlaða einstaklinga", "fötluðum", "fólk með fötlun", "fötlunar", "fatlað fólk", 
    "fatlaða individuals", 
    
    # Mobility-related
    "hreyfihömlunar", "people using wheelchairs", "people in wheelchairs",
    "people with neurological diseases", "people needing accessibility improvements",

    # Sensory impairments
    "blind and disabled people", "blind individuals", "the blind", "blind people",
    "the blind (figurative)", "people with color blindness", "litblinda (color blind)",
    "heyrnarskertur", "heyrnarlausir", "heyrnarlausra hælisleitenda",
    "deaf individuals", "deaf asylum seekers", "the blind (literal/figurative)", 
    "heyrna- og sjónskertir", "vision impaired", 

    # Children/parents context
    "disabled children", "young disabled individuals", "young disabled people",
    "young disability beneficiaries", "disabled women", "parents of disabled children",
    "parents of children with disabilities", "children with disabilities",
    "individuals with disabilities or the elderly",

    # Chronic / terminal illnesses (kept here, though conceptually distinct)
    "people with tuberculosis", "people with cancer", "parents of sick children",
    "long-term ill children", "patients", "people with chronic fatigue syndrome", "cancer patients",
    "individuals undergoing cancer treatment", "individuals with cancer", "people affected by cancer",
    "individuals with chronic illnesses", "people with chronic headaches", "people with chronic illnesses",
    "people with chronic pain", "hjartasjúklingar", "sykursjúklingar", "individuals with diabetes",
    "people with diabetes", "those with diabetes", "people with asthma", "HIV positive individuals",
    "people living with HIV", "people living with HIV/AIDS", "people with HIV", "people with HIV/AIDS", 
    
]


intellectual_disability = [
    # Intellectual/learning disabilities
    "intellectually disabled", "mentally disabled", "people with mental disabilities",
    "people with physical or mental disabilities", "people with mental or physical disabilities",
    "people with temporary disabilities", "individuals with mental disabilities",
    "persons with disabilities", "people with intellectual disabilities",
    "students with learning disabilities", "readers with learning disabilities",
    "people with reading disabilities", "lesblindir", "greindarskertra", "greindarskert fólk",
    "vangefnir", "vangefnum", "intellectually challenged", "intellectually challenged individuals",
    "people perceived as intellectually deficient", "intellectually disadvantaged",
    "those with mental impairments", "people with mental disabilities", "people with mental limitations",
    "people with limited mental capacity", "people with different mental states or disabilities", 
    "people with low mental maturity", "mentally disabled", "people with reading disabilities",
    "individuals with mental disabilities", "people with intellectual disabilities", "vangefnir",
    "vangefna", "leshamlaða eða þroskahamlaða", "seinfær", "children with Down Syndrome",
    "individuals with Down syndrome", "people with Down syndrome", "people with down syndrome",
    "those with Down syndrome", "women with Down syndrome", "individuals with brain injuries", 
    "people with intellectual disabilities", "people with brain damage", "mongólítar", "jaðarmongólíta",

    # Dementia, Alzheimer and speech disorders
    "people with dementia", "people with speech disabilities", "people with Tourette syndrome", 
    "those suffering from dementia", "Alzheimer's patients", "people with Alzheimer", "people with Alzheimer's",
    "people with Alzheimers", "people who stutter", "individuals with cognitive impairments", 

    # ADHD & autism
    "young people with ADHD", "people with ADHD", "individuals with ADHD and related disorders",
    "individuals diagnosed with ADHD", "those with ADHD", "children with ADHD",
    "ADHD individuals", "adults with ADHD", "individuals with ADHD",
    "people with ADHD or hyperactivity", "children with ADHD or hyperactivity",
    "people with autism", "individuals with autism", "autistic individuals",
    "parents of children with autism", "Asperger's syndrome", "individuals with Asperger syndrome", 
    "people with Asperger syndrome", "children with ADHD/ADD", "people with ADHD or hyperactivity",
    "those with ADHD", "ofvirkni",

]

mental_disorders = [
    # General mental illness
    "people with mental health issues", "people with mental illness",
    "people who struggle with mental illness", "people with mental illnesses",
    "people with different mental states or disabilities",
    "mental illness", "mental health patients", "mentally ill individuals",
    "mentally ill", "Mentally ill", "the mentally ill", "mentally ill people",
    "the mentally challenged", "geðfatlaðir", "geðveikir", "geðveikt fólk",
    "individuals with mental health issues", "mental health individuals", "children with mental illness", 
    "people with weak mental resilience", "individuals with mental health problems",
    "mental health patients", "people with mental disorders", "individuals struggling with mental health",
    "those with mental health issues", "people with mental health conditions", "people suffering from mental health issues",
    "those with mental disorders", "individual with mental illness", "individuals with mental illness",
    "people struggling with mental health", "individuals with mental disabilities", "people affected by mental health issues",
    "mental illness", "people expressing mental health issues", "those with mental illness", 
    "mental illness sufferers", "those affected by mental illness", "patients with mental health issues", 
    "those with mental Health issues", "individuals suffering from mental health issues and unemployment",
    "people receiving mental health care", "people experiencing mental health issues", "mental health sufferers", 
    "people at mental health institutions", "people with mental issues", "the elderly or individuals with mental health issues",
    "people diagnosed with mental health disorders", "individuals with mental health concerns", 
    "individuals with personality disorders", "people with antisocial personality disorder", "Geðsjúka fólkið",
    "geðbilaður", "geðsjúkir", "geðsjúklingar", "geðsjúklingsins","geðsjúkur", "geðsveikja fólk", "geðtruflaðir listamenn",

    # Depression, anxiety, bipolar, etc.
    "people experiencing depression", "people with depression",
    "people suffering from depression", "depressed individuals", "depressed women",
    "people with social anxiety", "people with anxiety disorders", "people with anxiety",
    "people who attempt suicide or struggle with mental health",
    "people considering suicide", "people with suicidal tendencies",
    "individuals who committed suicide", "youth/suicidal individuals",
    "suicide victims", "suicidal individuals", "people with suicidal thoughts",
    "people who have died by suicide", "people suffering from self-harm or suicide",
    "people who attempt suicide or struggle with mental health", "individuals with bipolar disorder",
    "people with bipolar disorder", "people with schizophrenia", "schizophrenics", "geðklofinn", 
    

    # Eating & body image disorders
    "patients with eating disorders", "people with eating disorders",
    "those with eating disorders", "individuals with eating disorders",
    "people with unhealthy eating habits", "people with body image issues",
    "individuals with eating disorders", "people with eating disorders",
    "those with eating disorders", 

]

addiction = [
    "addicts", "drug users", "people with addiction issues",
    "people who drink excessively", "people with alcoholism", "alcoholics", 
    "those in addiction", "drug addicts", "people with addictions", "people in addiction",
    "people with addiction", "people with alcohol addiction", "addiction", "individuals who struggle with addiction",
    "people seeking help with addiction", "victims of drug addiction", "fíklum (addicts)", "individuals with addictions",
    "people with drug addiction issues", "people who struggle with addiction", "people with addiction problems", 
    "those who smoke or have addictions", "drug addicts and alcoholics", "individuals with addiction",
    "weak addicts", "people with mental illnesses and addictions", "those with addiction issues", "individuals facing addiction",
    "people in addiction and trafficking", "addicts and alcoholics", "people struggling with addiction", 
    "people struggling with alcohol addiction", "individuals struggling with addiction or mental health issues",
    "people with sexual addiction", "people with sex addiction", "gamblers/addicts", "sex addicts", "women with drug addiction",
    "addicts and fiends", "people involved in addiction", "drug-addicted parents", "people dealing with addiction",
    "individuals struggling with addiction",  "individuals with drug addiction", "nicotine addicts", "individuals with addiction issues",
    "those struggling with addiction", "alkohólistar (alcoholics)", "Russian alcoholics", "individuals who misuse alcohol",
    "people with alcohol dependency", "Alcoholics Anonymous members", "people with alcohol problems", "members of Alcoholics Anonymous",
    "alcohol-dependent parents", "people with alcohol-related issues", "individuals with alcohol issues", 
    "individuals with alcohol-related health issues", "people who consume drugs other than alcohol", "people who misuse alcohol",
    "people with alcohol or drug issues", "people with alcohol issues", "individuals with alcohol dependency", "active alcoholics",
    "people who struggle with alcoholism", "Alcoholics Anonymous (AA)", "Alcoholics Anonymous", 
    "people dependent on alcohol or gambling", "former alcoholics", "people struggling with alcoholism", 
    "people with genetic predispositions to alcohol issues", "people affected by alcoholism", "people with alcohol dependence",
    "people undergoing treatment for alcoholism", "people associated with alcohol and drugs", "people suffering from alcoholism",
    "individuals with alcohol dependence", "poor drug users", "rich drug users", "people who engage in illegal drug trade",
    "illegal drug users", "those engaged in drug-related activities", "those involved in drug issues", "people who use drugs",
    "users of drugs", "drug abusers", "Those who misuse drugs", "people who engage in drug use", "older drug users", 
    "victims of drug addiction", "users of recreational drugs", "users of illegal drugs", "people who use illegal drugs", 
    "people with drug problems", "people using drugs", "men who use drugs", "Drug Users", "people involved in drug use",
    "drug-users", "people involved in drug use or distribution", "people involved in drug-related issues", "those who use illegal drugs",
    "people engaging in drug use", "users of psychedelic drugs", "young drug offenders", "people who abuse drugs", 
    "people in drug use", "drug-addicted parents", "people dependent on prescription drugs", "people involved with drug abuse",
    "those who use drugs", "Drug users", "drug users in general", "those involved in drug trafficking and prostitution", 
    "people associated with alcohol and drugs", "dópista", "dópistum", "dópdílerar", "dópistar", "dóphaus", "dóphákum",
    "alkóhólistar", "alkólistar", "alkóhólismi", "fyllibyttur", "fyllibytur", "fyllibittur", "drykkjumenn", "hassistar", "hasshausar",
    "hassi", "heroin users", "cocaine users", "people with substance use disorder", 
]


In [ ]:
women = ['feminist women', 'Eldri konur', 'women in policing', 'younger women in politics', 'Þingholtsstelpur', 
         'pregnant women/ mothers', 'young women', 'women who are mothers', 'brjóstapúðakonur (breast implant women)', 
         'stay-at-home mothers', 'kvenmenn', "women advocating for women's rights", 'women in authority', 'black women', 
         'powerful women', 'women dealing with financial issues', 'smástelpur', 'women from Thailand', 'Swedish women', 
         'Athletic women', 'women and children', 'Faroese women', 'women/girls', 'British women', 'women with drug addiction', 
         'religious women', 'breast augmentation and reduction women', 'women in low-paid jobs', 'úlpukonur', 'Framsókn kelling', 
         'African women in prostitution', 'Iranian women', 'Polish women', 'women who engage with multiple partners', 
         'framsóknarkonur (Progressive Party women)', 'lower class women', 'European women', 'women seeking abortions', 
         'alþýðukonur', 'women in waste management', 'women in rural areas', 'women in sex work', 'professional women', 
         'middle-aged white women', 'kvennréttindakonur', 'young women with strollers', 'Breiðavíkurstelpur', 'young men and women', 
         'women buying utilitarian cars', 'normal women', 'hússkonur', 'feminists', 'working women', 'independent women', 
         'Samfylkingarkonur', 'uneducated women', 'Islamic women', 'women in Saudi Arabia', 'skinny women', 'Íslenskar stelpur', 
         'large black women', 'women in adult films', 'women in the sex industry', 'fiskvinnslu konur', 'homeworking women', 
         'young adult women', 'sportswomen', 'western women', 'Akureyrarkonur', 'the Iraqi women', 'women in low-wage jobs', 
         'Christian women', 'weaker women', 'Brazilian women', 'women in banking', 'privileged women', 'young women in Iceland', 
         'wealthy women', 'depressed women', 'women in the adult entertainment industry', 'women in professional roles', 
         'women in Islamic countries', 'Danish women', 'middle-aged women', 'rightist women', 'American women', 'færeyskar konur', 
         'Norse women', 'Women', 'women in adult film industry', 'Western women', 'women from rural Iceland', 'wealty women', 
         'trans women', 'stelpur', 'women over 60', 'sveitastelpur', 'einstæðar íslenskar konur', 'fermigarstelpurnar', 
         'Conservative women', 'light-haired women', 'women in finance', 'women in the sex trade', 'woman', 
         'women in adult entertainment', 'English women', 'kerlingaranginn', 'Twiggy types (thin women)', 'women from the Balkans', 
         'women in power', 'young women in politics', 'kellingar (women)', 'European Christian women', 'fat women', 
         'Eastern European women', 'Pakistani women', 'young women in care', 'Virgo women', 'women (as a group)', 'Russian women', 
         'females', 'lesbian women', 'former Muslim women', "women (specifically 'rauðsokkur')", 'women athletes', 
         'Icelandic women and children', 'Austfirskar konur', 'blúndukellingar', 'muslim women', 'tékklenskar stúlkur', 
         'women in media', 'African women', 'White women', 'Colombian women', 'women in politics', 'konur', 'overweight women', 
         'kaffihúsakerling', 'Eyjapíur (Eyjapækur)', 'Muslim women', 'eldri konur', 'women with breast surgery', 'kellingarnar', 
         'Japanese women', 'Konur', 'kreditkellingar', 'young Icelandic women', 'French women', 'women doing sit-ups', 
         'kvenfélagskonur', 'women married to doctors', 'women selling sex services', 'women from Greenland', 'kerlingar', 
         'unvaccinated women', 'northern women', 'women giving up children', 'vinstrisinnaðar konur', 'married women', 
         'young women in prostitution', 'arab women', 'Eyjapíur', 'Afganskar konur', 'unmarried young women', 'white women', 
         'elderly women', 'kellingar', 'older women', 'women in political leadership positions', 'women in their 20s', 'women over 40', 
         'young women in leadership', 'framsóknarkonur', 'Arab women', 'Icelandic businesswomen', 'ungar iðnaðar-silicon-stelpur', 
         'women teachers', 'nornirnar (the three women)', 'women in Islam', 'mothers with past in adult entertainment', 
         'women from certain countries', 'stúlkur', 'Bulgarian women', 'Icelandic women', 'gellurnar á Múlakaffi', 'jealous women', 
         'blond women', 'listakonur', 'self-identified independent women', 'thin women', 'women in high positions', 
         'women with academic qualifications', 'women from the East', 'foreign women', 'upper-class women', 
         'feminists or women activists', 'women with Down syndrome', 'right-wing women', 'women in leadership roles', 
         'younger women', 'female politicians', 'women advocating for womens rights', 'women', 'women in the industry', 
         'Jewish women', 'immigrant women', 'slender women', 'leftist women', 'Franskar konur', 'pregnant women', 
         'women in conservative societies', 'modern women', 'older women selling firewood', 'Thai women', 'snobbish women', 
         'Vietnamese women', 'athafnakonur', 'unhappy women', 'women in aviation', 'women in the airline industry', 'Irish women', 
         'kúlukellingar', 'grátkellingarnir', 'nigerískar konur', 'Celtic women', 'women and children in developing countries', 
         'refugee women', 'womens roles in politics', 'pregnant women and children', 'Guatemalan women', 'danska konur', 
         'women in sports', 'strong women', 'Lithuanian women', 'women in high-paid modeling jobs', 'light women', 'womens groups', 
         'women in leadership', 'elder women', 'Czech women dancing in Iceland', 'wealthy married women', 'women in parliament', 
         'women in certain appearances', 'women involved in pornography', 'women from the West', 'kvefaðar konur', 
         'pregnant women mothers', 'young women from Russia', 'women lacking self-care', 'women who have given birth', 
         'women in leftist parties', 'women from the Philippines', 'womens rights activists', 'young women in England', 
         'píratastúlkur', 'Nigerian women', 'migrant women', 'women with a past in pornography or sex work', 
         'women in conservative cultures', 'German women', 'sveitastelpurnar', 'hairy women', 'dellukerlingar', 
         'ljóska (blonde) women', 'single mothers or women in cohabitation', 'women in pornography', 'Norwegian women', 
         'Women of Middle Eastern descent', 'Ísfirskar stelpur konur', 'women priests', 'Italian women', 'single mothers', 
         'women in government', 'dark-haired women', 'women in tourism', 'barstelpurnar (women in Thailand)', 
         'elderly women supporting whaling', 'women in Bangladesh', 'high-paid women in Iceland', 'Þróttara kellingar', 
         'Filipino women', 'women seeking partners', 'self-governing women', 'young girls and women', 'kvenréttindakonur', 
         'women in prostitution', 'ordinary women', 'women politicians', 'young married or engaged women', 'Womens rights advocates', 
         'gáfaðar konur', 'plus-size women', 'Asian women', 'single women', 'Chinese women', 'breskar konur', 'póstburðarkonur', 
         'poor women', 'women from rural areas', 'women experiencing miscarriages', 'Afghan women', 'certain women', 'women in Iran', 
         'women involved in sex work', 'divorced women', 'öfgakellingar', 'women of color', 'disabled women', 'old women', 
         'háskólamenntaðar konur', 'nöldrara og tuðkerlinga', 'ugly women', 'Bleiku konurnar', 'fisherwomen', 'Palestinian women', 
         'Finnish women', 'Framsóknarkonur', 'women in the Middle East', 'women in medicine', 'women in Kuwait', 'stútungskerlingar', 
         'blonde women', 'women from abroad', 'innocent women', 'women (victims of false accusations)', 'akureyrskar kerlingar', 
         'Latvian women', 'cleaning women', 'women in punk group', 'vaccinated women', 'women drivers', 'famous women', 
         'red-haired women', 'flóttakonur', 'slim women', 'women selling sex', 'unmarried women', 'kvenkyns starfsmenn',]

men = ['samkynhneigðir karlmenn', 'white middle-aged men', 'fat middle-aged white men', 'lítilla karla', 'self-confident men', 
       'men from Central and South America', 'men in poetry', 'Middle-aged men', 'white men', 'Eldri menn', 'effeminate men', 
       'mörlenskir hægriöfgakarlar', 'gray-haired men', 'boys/men', 'elderly men with wealth', 'Japanese men', 'accused men', 
       'trillukarlar', 'ríku menn', 'Lithuanian men', 'certain men', 'Swiss men', 'Scandinavian men', 'hvíta miðaldra karla', 
       'láglaunakarlar', 'men in charge', 'þrítugar menn', 'men in nursing', 'men educators', 'kryppukarlar', 'middle-aged men', 
       'wealthy older men', 'foreign men', 'young men in various political parties', 'einband karlmenn', 'Asian men', 
       'young conservative men', 'male commentators', 'men in power', 'Finnish men', 'öfgahægrisinnaðir karlar', 'homosexual men', 
       'eldra menn', 'men who abuse women', 'heterosexual men', 'fátækir menn', 'ljóshærðir menn', 'unemployed men', 'the men in power', 
       'Hægriöfgakarlar', 'gagnkynhneigðir karlar', 'Jewish men', 'mörlenskir öfgahægrikarlar', 'men who use drugs', 'gay men', 
       'White men', 'alvörukarlmenn', 'óþroskaðir karlmenn', 'thin men', 'hægriöfgakarlar í Kópavoginum', 'dark-haired men', 
       'circumcised men', 'middle-aged conservative men', 'Breskar menn', 'ugly and overweight men', 'Islamic men', 'older white men', 
       'men in professional roles', 'eldri menn', 'samkynhneigðir karlar', 'menn af landsbyggðinni', 'younger men in the Framsóknarflokk', 
       'men in sports', 'Moroccan men', 'Karlar', 'yngri menn', 'powerful men', 'Öfgahægrikarlar', 'good men', 'Western men', 
       'men who purchase sex', 'men buying sex services', 'white middle-aged conservative men', 'hægriöfgakarlarnir', 'Icelandic men', 
       'elderly men', 'wealthy men', 'single men', 'white, Christian, heterosexual men', 'muslim men', 'men in tourism', 'Afghan men', 
       'younger men', 'young white men', 'hvíta karlmenn', 'macho men', 'ríkir karlar', 'hvítir karlmenn', 'men in labor leadership', 
       'very rich and powerful men', 'öfgahægrikarla', 'mörlenskir ungir karlmenn', 'múslimskir karlmenn', 'feminine men', 
       'fullorða menn', 'men in authority', 'togarakarlar', 'mörlensku öfgahægrikarlarnir', 'men buying sex', 'útrásarkarlar', 
       'karlkyns starfsmenn', 'mature adult men', 'Menn', 'ungir menn', 'karlaklúbbaklíkuna', 'menn', 'men with unsafe sexual orientation', 
       'islandskarlar', 'lítli menn', 'eldri karlmenn', 'middle-aged white men', 'kallmenn', 'bandarískir menn', 
       'Polish men', 'Arab men', 'older men with younger women', 'hvíti karlmenn', 'white heterosexual men', 'gagnkynhneigðir karlmenn', 
       'latino men', 'middle-aged single men', 'dark-haired middle-aged men', 'successful men', 'young right-wing men', 'karlmenn (men)', 
       'Men', 'rich men', 'Italian men', 'tall men', 'karlar', 'kynthystra karla', 'French men', 'wealthy and powerful men', 'married men',
       'older men', 'men in business', 'miðaldra karlar', 'hægrikarlar', 'dwarf men', 'fullorðnir karlar', 'karla', 'ríkir menn', 
       'grískir menn', 'men in the pornography industry', 'erlendir karlmenn', 'middle-aged white heterosexual men', 
       'miðaldra karlar með háskólapróf', 'men over fifty', 'gráhærðir karlmenn', 'adult men', 'Öfgahægrikarlarnir', 'rich young men', 
       'real men', 'men in choirs', 'German men', 'fat men', 'red-haired men', 'bankakarlarnir', 'men in suits', 
       'men in religious institutions', 'hvítir enskir karlmenn', 'young Icelandic men', 'strákar-karlmenn', 'large-breasted men', 
       'Ethiopian men', 'young men', 'men in politics', 'black men', 'karlmenn', 'men who visit strip clubs', 'eldri karlar', 
       'mittel-aged men', 'wealthy men in Old Europe', 'óprúttnir menn', 'Black men', 'old men', 'men who purchase sex services', 
       'young self-determined men', 'unmarried men', 'dökkir menn', 'short men', 'young men with low education and income', 
       'karlagrobbara', 'men who buy sex services', 'Norskir karlmenn', 'aumingja menn', 'verkakarlar', 'snobbish men', 'Islamic Men', 
       'jeppamenn', 'sexist men', 'kallinn (men)', 'men in finance', 'leftist men', 'men', 'hvíta menn', 'young men of color', 
       'körlumenn', 'trillukarla', 'svartrar menn', 'karmenn', 'valdamiklu menn', 'Indian men', 'útrásarhúskarlar', 'öfgahægrikarlar', 
       'Hörundsdökkir menn', 'Scottish men', 'men in positions of power', 'men in rural areas', 'Menn í Norður-Kóreu', 'erlendra menn', 
       'dimraddr karlmenn', 'local men', 'ríka karlar', 'overweight men', 'Muslim men', 'Greek men', 'Men (male flight attendants)', 
       'Young men', 'small men', 'samkynhneigðir menn', 'arrogant men', 'miðaldra hvítir hetrósexual karlmenn', 'men in a singles club', 
       'hægriöfgakarlar', 'öfgahægrikarlarnir', 'men who dislike female entertainment', 'poor men', 'men in leadership positions', 
       'öftahægrikarlar', 'stærri menn', 'men educated in economics', 'Lebanese men']

In [ ]:
black_african_descent = [
    'people with African ancestry', 'Black citizens', 'Svart fólk', 'Black people', 'negrastrákana', 'black families', 
    'Black children', 'negroes', 'blökkumenn (Black people)', 'black communities', 
    'blökkumenn (people of African descent)', 'blökkumaður', 'Sómalísvertingjar', 'black women', 'Black Americans', 
    'the black nation', 'black South Africans', 'black students', 'blökkumennirnir', 'descendants of Africans', 'Black persons', 
    'svertingja', 'black', 'African Americans', 'þeldökka verkfallsmenn', 'Black people in America', 'African American', 
    'blacks', 'Black', 'African women in prostitution', 'black teachers', 'black children', 'Black Africans', 'Blacks', 
    'svertingjar', 'African American descendants', 'young black people', 'negrastrákar', 'svertingar', 'blökkumenn', 
    'black people', 'blökkumenn (African Americans)', 'negrar', 'black candidates', 'African women', 'svarta fólkið', 
    'black Jamaicans', 'African American people', 'svarta', 'People with African ancestry', 'African-Icelandic children', 
    'American blacks', 'Blökkufólk', 'large black women', 'black people from ghettos', 'black men', 
    'single black mothers', 'Black men', 'black man', 'svörtum', 'NegrarNorðursins', 'negra', 'black individuals in the USA', 
    'svart fólk', 'blökkufólk', 'blámennblámaður', 'Black candidates', 'svartir', 'black athletes', 'Black People', 
    'young men of color', 'Blacks in America', 'svartrar menn', 'black community', 'Svertingjar', 'þeldökkra', 
    'black girls', 'blökkumann', 'Black individuals', 'black individuals', 'þeldökkum (people of color)', 
    'black people in America', 'Afro-Americans', 'svart fólk (Black people)', 'théldökkra', 'theldökki kynstofninn', 'theldökkir',
    'Hörundsdökkir menn', 'theldökkra', 'dökkir menn', 'hörundsdökkum', 'indigenous Africans',
]

white_european_descent = [
    'white middle-aged men', 'fat middle-aged white men', 'Caucasians', 'white men', 'white individuals', 'white Europeans', 
    'Romans',  'white American', 'aríar (hreinir hvítir menn)', 'hvíta miðaldra karla', 'White individuals', 
    'hvíta afbrotamenn', 'white voters', 'white capitalists', 'white community', 'hvíta millistéttin', 'White Christians', 
    'White men', 'hvíta mannsins', 'white man', 'white nations', 'hvíti maðurinn', 'White Americans', 'older white men', 'Hvítir', 
    'hvítingjarnir', 'people of the white race', 'Hvítru', 'white Australians', 'White people', 'white right-wing individuals', 
    'white race', 'white French', 'white police officers', 'spikfeitu hvítingjar', 'White males', 'wealthy white people', 
    'white middle-aged conservative men', "people considered 'redneck' or 'white trash'", 'white, Christian, heterosexual men', 
    'white Nordics', 'young white men', 'hvíta karlmenn',  'hvítir karlmenn', 'white nationalists',
    'middle-aged white men', 'hvíta fólkið', 'white farmers', 'middle-aged white women', 'hvíti karlmenn', 'white criminals', 
    'white heterosexual men', 'hvíta stofninn', 'white Westerners', 'white people of European descent', 'hvítingjar', 
    'uneducated white individuals', 'white males', 'whites', 'young white males', 'hvíti kynstofninn', 'white South Africans', 
    'White descent', 'White women', 'white girls', 'middle-aged white heterosexual men', 'white extremists', 
    'hvítir Ameríkanskar byssuglaðrar', 'white farmers in South Africa', 'hvítt fólk', 'white artists', 'white Christians', 
    'white southerners', 'Poor whites', 'Whites', 'hvítir enskir karlmenn', 'hvítir', 'white students', 
    'white trash', 'people with white skin', 'white settlers', 'white people', 'Caucasian', 'hvíta', 
    'hvíta menn', 'white Christian Russians', 'white women', 'white Icelanders', 'poor whites', 'Whites in America', 
    'white population in South Africa', 'white rural Americans', 'white middle-class', 'hvíta kynstofninn', 
    'miðaldra hvítir hetrósexual karlmenn', 'hvíta sunnanmenn', 'people with white ancestry', 'hvít kynstofn', 'white',
    'Roma people', 'sígaunarnir', 'Rómafólk', 'Róma fólk', 'gypsies', 'Sígaunar', 'sígaunar', 'Roma',
    'Catalans', 'Baskar', 'Basques', 'Slavs'
]

# Ath Rómafólk útskýrir alveg einn topp

middle_eastern_arab_descent = [
    # --- Arab ethno-cultural terms ---
    "Sunni Arabs", "Súnní-arabar", "Súnní Araba",
    "Arabs", "Arabar", "arabs", "arabar", "Arab", "Arab peoples", "Arab people", "Arabians",
    "Arab individuals", "Arab families", "Arab writers", "Arab leaders", "Arab princes",
    "Arab Christians", "Arab women", "arab women", "Arab men",
    "Arab-Muslims", "Arabs/Muslims", "Arabs (Muslims)", "Muslim Arabs", "Arabs and Muslims",
    "kristnir arabar", "Múhameðsku Arabs",
    "Arab immigrants", "immigrants from Arab descent", "people of Arab descent",
    "people of Arabic origin", "people of Arabic descent", "people of Arab origin",
    "Palestinian Arabs", "Palestínuarabar", "Palestínu-Araba", "Palestínuarabinn", "Palesrínu-Araba",
    "Israeli Arabs", "Arabs in Israel",
    "Arab population", "Arab cultures", "Arab culture", "Arab world",
    "Arab countries", "Arab states", "Arab nations", "Arab nationalities", "Arab League",
    "Arabaheimur", "Arabaheimurinn", "arabaþjóðir", "Arabaþjóðir", "Araba þjóðir", "Arabískar þjóðir",
    "Arabaríkin", "Arabaríki", "arabarríkin", "Arabíkin",
    "arabalöndin", "arabalöndunum",
    "arab-speaking nations",
    "Arab nationals",
    "Kurds", "Kúrdar", "Kúrda", "Kurdish people",
    "Persians",

    # --- Regional / Middle Eastern umbrella terms ---
    "Middle Eastern countries", "Middle Eastern nations", "Middle-Eastern countries",
    "Middle Eastern dictatorships", "Middle Eastern extremists", "Middle Eastern leaders",
    "Middle Eastern civilians", "Middle Eastern population", "Middle Eastern People",
    "Middle Eastern individuals", "Middle Eastern cultures", "Middle Eastern culture",
    "people in the Middle East", "people from the Middle East", "People from Middle East",
    "People from Middle Eastern countries", "people in Middle Eastern countries",
    "people in certain Middle Eastern countries",
    "people of the Middle East", "people from the Middle East and Africa",
    "Middle East oil producers",
    "nationalities from the Middle East",
    "Miðausturlönd", "Mið-Austurlönd", "Mið-austurlönd",
    "Miðausturlandabúar", "Mið-Austurlandi inhabitants", "Mið-Austurlendingar",
    "Middle Easterners", "middle easterners", "Middle Eastern people",
    "women in the Middle East", "women in Islamic countries",
    "Middle Eastern immigrants",
    "individuals from the Middle East",
    "Saudis and Arabs"  # mixed ethnicity + nationality, often collapsed in prejudice
]

asian_descent = [
    'Asian factory workers', 'Asíubúar', 'asians', 'people from Asia', 'asíubúar', 'Asian descent', 'People of Asian descent', 
    'Asian investors', 'Asian people', 'people with Asian heritage', 'ásíubúar', 'Asian countries', 'people of Asian descent', 
    'Asian nations', 'people of East Asian descent', 'Asian women', 'Central Asian countries', 'Austur- Asíu', 'Rohingya people', 
    'East Asian', 'Southeast Asian countries', 'Asian men', 'people from Asian descent', 'Asian nationals', 'Asian populations', 
    'austur-asíubúa', 'Asíu', 'Asian', 'Asians', 'Uyghurs', 'eastern Asians', 'Asian individuals', 'Central Asians', 'Eastern Asians', 
    'Austur-Asíubúa', 'Asian Mongols'
]

indigenous_peoples = [
    'Native Americans', 'Indigenous Americans', 'Indigenous Americans',
    'Indigenous peoples', 'indigenous Americans', 'Inuit', 'indigenous people', 'Indigenous Peoples',
    'indigenous peoples', 'indigenous populations', 'indigenous Australians', 'Indigenous people', 'indigenous workers',
    'Indigenous Australians', 'Indigenous populations', 'indigenous peoples of the Americas', 'indigenous cultures', 
    'indigenous people in Australia', 'Indigenous Alaskans', 'indigenous tribes', 'Indigenous peoples of North America',
    'indigenous', 'North American Indigenous People', 'people of indigenous descent', 'indigenous Americans', 'indigenous australians',
    'Indigenous people of the Americas', 'frumbyggjar (indigenous people)', 'indigenous criminals', 'Indjánar', 'Iroquois Indians',
    'Frumbyggjar Ameríku', 'Frumbyggjar Grænlands', 'frumbyggjar', 'frumbyggjar Norður Ameríku', 'frumbyggjar Norður-Ameríku',
    'Ástralíufrumbyggja', 'Ameríkuindíánar', 'Indíjánar', 'Indíána', 'Indíánar', 'indíána', 'indíánanna', 'indíánar', 'indíánarnir',
    
]

latino_latin_american_descent = [
    'Latin Americans', 'Latino', 'Latinos', 'Latino Americans', 'Latino countries', 'Latino culture','Latino individuals',
    'latino communities', 'latino community', 'latino men', 'latinos', 'Hispanic communities', 'Hispanics', 'Central American countries',
    'Central American ethnic groups', 'countries in Central America', 'people in Central America', 'South America', 
    'South American countries', 'South Americans', 'families from Central/South America', 'men from Central and South America',
    'people from South America', 'people in Central and South America', 'people in South America', 
]

multiethnic_general = [
    'people of color', 'People of color', 'women of color',
    'BIPOC', 'marginalized racial groups', 'mixed-race',
    'non-white individuals', 'colored races',
    'other ethnicities', 'individuals based on race',
    'developing nations', 'third world countries', 'underdeveloped countries',
    'ethnic groups', 'people of different races', 'people of different skin colors',
    'people with darker skin', 'people of color', 'People of Color', 'People of color', 'litaða',  'litað fólk',
]

In [ ]:
social_status = [
    'poor communities', 'háskólagengið fólk', 'those in authority positions',
    'plebbur (plebs)', 'poor people', 'elite individuals', 'rich people', 'those in power',
    'wealthy elites', 'almúgann', 'arðræningjar', 'unemployed', 'intellectuals',
    'powerful individuals', 'scholarly community', 'the Icelandic elite', 'workers',
    'wealthy', 'middle class', 'middle-class residents in downtown Reykjavík',
    'millistéttin (middle class)', 'lower middle class families', 'upper middle class',
    'middle-class citizens', 'middle-class individuals', 'millistéttarfólk (middle class)',
    'middle class consumers', 'lower and middle class', 'middle-class families',
    'the middle class', 'middle-income earners', 'middle-income individuals',
    'upper and middle class', 'middle-class taxpayers', 'middle-income and low-income people',
    'low and middle-income individuals', 'middle-income people', 'upper-middle class',
    'middle class feminists', 'middle-class businesses', 'middle-class residents',
    'low-income and middle-income individuals', 'people with middle incomes',
    'lower class women', 'high-paid women in Iceland', 'high-income women',
    'women in low-paid jobs', 'women in high-paid modeling jobs', 'women in low-wage jobs',
    'wealthy married women', 'wealty women', 'poor individuals', 'poor', 'the poor',
    'poor families', 'poor drug users', 'the poorly educated', 'poorly educated individuals',
    'poor children', 'poor Icelanders', 'poor laborers', 'poor students',
    'poorly educated people', 'poor immigrants', 'poor women',
    'unemployed or poor individuals', 'the poor and homeless', 'poor musicians',
    'poor workers', 'poor populations', 'the poor and elderly', 'the poor/unemployed',
    'foreign poor', 'poor drug dealers', 'poor uneducated people', 'poor farmers',
    'children from poor families', 'people with poor language skills',
    'the poor and vulnerable', 'individuals with poor language skills', 'urban poor',
    'poor patients', 'the weak and poor', 'poor buyers',
    'people who are poor or bankrupt', 'Icelandic poor', 'poor Icelandic families',
    'individuals with poor financial management', 'wealthy versus poor populations',
    'people who speak Icelandic poorly', 'poor men',
    'people with poor Icelandic language skills', 'poor citizens', 'poor households',
    'poor debtors', 'Poor people', 'poorer farmers', 'poor and vulnerable',
    'people with poor education', 'poorest individuals', 'wealthy versus poor',
    'wealthy and poor individuals', 'people with poor financial literacy',
    'wealthy vs. poor', 'local poor', 'poor whites', 'poor drivers', 'Poor whites',
    'öreiga (the poor)', 'the poor / working class', 'rich and poor', 'poor vs. rich',
    'the homeless and poor', 'poor population', 'poorer people', 'poor persons',
    'poorly educated', 'hungry poor', 'rich elites', 'rich', 'rich criminals', 'the rich',
    'rich individuals', 'rich elite', 'rich kids', 'very rich and powerful men',
    'ultrarich businesspeople', 'rich young men', 'rich upper class',
    'bankers and rich people', '10% richest', 'people who are rich', 'rich quota-holders',
    'newly rich individuals', 'richer groups', 'Icelandic rich', 'ultra-rich individuals',
    'ultrarich', 'newly rich families', 'rich men', 'rich and famous people',
    'newly rich Chinese', 'rich families', 'newly rich', 'rich Chinese', 'rich capitalists',
    'the rich elite', 'super-rich individuals', 'rich and powerful individuals',
    'rich entrepreneurs', 'rich and powerful people', 'rich business people',
    'richer individuals', 'new rich individuals', 'the elite or rich individuals',
    'rich children', 'newly rich Icelanders', 'new rich', 'richer people',
    'rich capital owners', 'the newly rich', 'rich fraudsters', 'filthy rich',
    'lower-class children', 'lower-class', 'lower-class workers', 'lower-class people',
    'lower-class individuals', 'low-class individuals', 'upper-class individuals',
    'upper-class women', 'upper-class feminists', 'upper-class', 'útrásarvíkingar',
    'íslensku útrásarvíkingarnir', 'útrásarvíkinga', 'útrásarvíkingarnir',
    'útrásarvíkíngarnir', 'f.v. útrásarvíkingar', 'útrásarvíkingur',
    'Íslensku útrásarvíkingarnir', 'útrásarVíkingarnir', 'útrásarvíkingapar',
    'útrásarvíkingunum', 'útrásarvíkingas', 'new útrásarvíkingar', 'útrásarvíkinganna',
    'útrásarvíkingum', 'kvótakóngar', 'kvótaklíka', 'afla- og kvótakóngar',
    'kvótakóngi', 'kvótakónga', 'kvótakóngarnir', 'kvótakóngana', 'kvótagreifarnir',
    'kvótabraskara', 'kvótabraskar', 'kvótavarðhundar', 'kvótabraskarar',
    'kvótabraskararnir', 'kvótagreifar', 'kvótagerðarmenn', 'kvótakall',
    'gamlar kvótasölugreifar', 'kvótapostular', 'kvótahafarnir', 'kvótaaðlinum',
    'Félópakk', 'öryrkjafýlu-offitusjúklingar', 'súperofuröryrkjar', 'bótaþegar',
    'bótahafar', 'efri millistétt', 'fólk í efri hluta tekjustigans', 'efri stétt',
    'efri-millistéttin', 'valda- og eignastéttin', 'yfirstéttin', 'valdastéttin',
    'millistéttin', 'valdstéttin á Íslandi', 'valdstéttin', 'kvennastéttir',
    'bændastéttin', 'vinnandi stéttir', 'millistéttir', 'yfirstéttir', 'valdastéttina',
    'borgarastéttina', 'láglaunastéttir', 'lágstéttir', 'forréttindastéttir',
    'Borgarastéttina', 'valdastéttir', 'hvíta millistéttin', 'lágstéttin',
    'karlastéttir', 'bændastéttina', 'lægra stéttir', 'hástéttin'
]

In [ ]:
age = [
    "children", "youths", "older women", "women over 40", "teenagers", "youth",
    "young drivers", "young offenders", "middle ages", "gray-haired men", "elderly",
    "young people", "older adults", "younger generation", "young men", "young players",
    "young protesters", "young families", "young politicians", "young women in leadership",
    "young girls", "young readers", "young women", "young Icelanders",
    "younger police officers", "young generation", "young homebuyers", "young generations",
    "younger generations", "young professionals", "young mothers",
    "young people from Ísafjörður", "young people in rural areas", "younger students",
    "younger individuals", "young voters", "young people from EU countries",
    "young journalists", "young graduates", "young individuals", "younger people",
    "young adults", "young homeowners", "young musicians", "young women with strollers",
    "the young generation", "older vs younger generations", "young Iceland",
    "younger population", "younger Icelanders", "young boys", "young adults (18-23)",
    "the younger generation", "young children", "young educated people",
    "younger healthcare professionals", "rich young men", "young doctors",
    "younger drivers", "young workers", "young couples with debts",
    "younger rock enthusiasts", "younger voters", "young people receiving vaccinations",
    "young activists", "healthy young individuals", "young girls and women",
    "young healthy people", "young and old Icelanders", "young athletes",
    "young and unruly individuals", "younger professionals", "young people from South Iceland",
    "young leaders", "younger women", "young couples", "young educated individuals",
    "young Europeans", "young women in Reykjavik", "young women in care",
    "young married or engaged women", "children and young people",
    "students young people", "young economists", "younger employees", "young woman",
    "young white males", "younger managers", "younger workers", "young women from Russia",
    "young farmers", "Young People", "young entrepreneurs", "young representatives",
    "Younger generations", "young unemployed", "young immigrants", "Young parents",
    "young male drivers", "Younger voters", "younger audience",
    "young and unstable individuals", "young students", "families with young children",
    "young political candidates", "young unemployed individuals", "young women in Iceland",
    "young motorcyclists", "young men with low education and income",
    "young members of political parties", "young cyclists", "young hosts",
    "younger adults", "young educated Icelanders", "the young", "young talented people",
    "young political participants", "elderly or younger age groups",
    "young adults living with parents", "young men from Suðurnes", "young Faroese",
    "young people in Faroe Islands", "young parents", "younger men",
    "unmarried young women", "young people in Spain", "young people in Greece",
    "young unemployed people", "young men and women", "young talented individuals",
    "young adult women", "youth or young people", "immature young adults",
    "unvaccinated young people", "young unvaccinated people", "young learners",
    "unskilled young people", "young authors", "young people living at home",
    "younger job applicants", "younger mothers", "young Norwegian workers",
    "young financial professionals", "young boys and girls", "young drug offenders",
    "young people in Europe", "young football players", "young foreign workers",
    "young radical people", "young renters", "Icelandic young players",
    "young right-wing voters", "young engineers", "young people and teachers",
    "young women in politics", "young-generation", "young males", "younger players",
    "young people in Reykjavik", "Young men", "young audience",
    "young and middle-aged people", "young fathers", "young fishermen",
    "younger partners", "younger citizens", "older men with younger women",
    "Young people", "younger families", "young Icelandic men", "young Icelandic women",
    "younger housewives", "young white men", "young people without higher education",
    "youngsters", "young travelers", "unemployed or young people", "young females",
    "older people", "older individuals", "old people", "older generations",
    "older Icelanders", "older generation", "the older generation", "older voters",
    "old men", "old politicians", "older students", "older politicians",
    "eldri borgarar (older citizens)", "older men", "old generation", "older workers",
    "older individuals without financial saving", "older drug users", "older drivers",
    "old drivers", "old workers", "old candidates", "old hippies", "older citizens",
    "old conservative men", "older members of the party", "older fishermen",
    "older employees", "older managers", "older business professionals",
    "30-60 year olds", "old women", "older audience", "20-50 year old men",
    "old party members", "old farmers", "the old people", "olders", "older population",
    "Old people", "older male politicians", "older white men", "older males",
    "18-23 year olds", "18-year-olds", "old Icelanders", "older actresses",
    "wealthy older men", "17-year-olds", "old generations", "16 - 17 year olds",
    "old aristocrats", "old age pensioners", "older mothers", "older sailors",
    "old individuals", "18-20 year olds", "70-year-olds", "the old generation",
    "retirement fund holders", "older boys", "older dentists", "older celebrities",
    "older tourists", "old retirees", "17 year olds", "older demographics",
    "older political figures", "people aged 50 and older", "certain older individuals",
    "older fathers", "middle-aged men", "middle-aged single men", "middle age communities",
    "middle-aged social workers", "middle-aged women", "middle-aged people",
    "middle-aged", "middle-aged Icelanders", "white middle-aged conservative men",
    "middle-aged white men", "white middle-aged men", "fat middle-aged white men",
    "middle-aged lawyers or business professionals", "middle-aged conservative men",
    "middle-aged politicians", "middle-aged individuals", "middle-aged white women",
    "middle-aged and elderly people", "middle-aged Christian men",
    "middle-aged white heterosexual men", "middle-aged vaccinated individuals",
    "middle-aged housewives", "middle-aged males", "middle-aged office workers",
    "middle-aged families", "dark-haired middle-aged men", "Middle-aged men",
    "middle-aged Eastern Europeans", "middle-aged privileged individuals",
    "miðaldra karlar", "miðaldra karlar með háskólapróf", "hvíta miðaldra karla",
    "miðaldra hvítir hetrósexual karlmenn", "eldra og miðaldra fólk",
    "menntaskólakrakkar", "leiksólakrakka (presumably immature people)", "krakkar",
    "landsbyggðarkrakka", "sjallakrakkar", "frjálshyggjukrakkarnir",
    "framhaldsskólakrakkar", "Framhaldsskólakrakkar", "krakkaskrattar",
    "sveitakrakkar", "unglingar (youth)", "unglingar", "mothers of teenagers",
    "teenage boys", "teenage girls", "Icelandic teenage girls", "children and teenagers",
    "uneducated teenagers", "gamlingjar", "gamla hipparnir", "gamla liðinu",
    "gamla kommana", "gamlar kvótasölugreifar", "gamla fólksins", "gamla hippa",
    "women over 40", "Elderly", "Elderly people"
]


In [ ]:
eastern_europeans = [
    'Bulgarian and Romanian people', 'Hvíta-Rússlandi', 'Bulgarian', 'Serbia', 'Czech citizens', 'Eistars', 
    'Albanian immigrants', 'Albania', 'Ungverja', 'Albanian workers', 'Czech Republic', 'Litháar', 'russian minorities', 
    'Bulgarians', 'Russian-speaking people', 'Russian nationals', 'Lithuanian men', 'Austur-evrópumenn', 'Lithuanians', 
    'Macedonians', 'the Russians', 'Romani', 'Bosniaks', 'Estonian people', 'Króatar', 'Belarusians', 'inhabitants of Kosovo', 'Slovakia', 
    'Eistland og Lettland', 'Albanian migrants', 'Rúmenarnir', 'citizens of Belarus', 'people of Eastern European descent', 'Rússar', 
    'Slóvakía', 'ethnic Russians', 'Bulgarian women', 'Ungverjaland', 'nations in Eastern Europe', 'Austur-Evrópubúa', 'Albanians', 
    'Króatía', 'people from Eastern Europe', 'people in Eastern Europe', 'Lithuania', 'Russian population', 'Moldovan workers', 
    'Litháarnir (Lithuanians)', 'Romania/Bulgaria', 'Romanians and Albanians', 'Rúmenskir innflytjendur', 'Russian-speaking separatists', 
    'Kosovar', 'eastern Europeans', 'Eastern European migrants', 'ethnic Russians in Ukraine', 'rúmenskir götulistamenn', 
    'Hungarian workers', 'newly wealthy Russians', 'Litháanar', 'Russian doctors', 'individuals from Romania', 'Albanian families', 
    'Serbians', 'Bulgarians and Romanians', 'Rúmeníu og Búlgaríu', 'Albanskir', 'Belarus', 'Latvian people', 'albanska fjölskyldan', 
    'Lithárnir', 'Serbian minority', 'Russian individuals', 'Russia', 'romanians', 'Tékki', 'Litáar', 'russian speakers', 'Eistarnir', 
    'Austur-Evrópuþjóðir', 'Serbian people', 'Russian citizens', 'russian-speaking people', 'Croats', 'Russian journalists', 
    'Russian athletes', 'Albanian', 'Albanian people', 'Eastern European mafia', 'Eastern European women', 'Eastern European immigrants', 
    'Eastern European farmers', 'Lettland', 'Eastern European nations', 'Turks, Russians, Europeans', 'families in Bulgaria', 
    'populations from Eastern European countries', 'Eastern European countries', 
    'Czech Republic, Poland, Slovenia, Slovakia and the Baltic States', 'Romanian society', 'people from Romania', 'Romanian criminals', 'Russian women', 'Albanir', 
    'eastern European nations', 'Russian mafia', 'Russian-speaking citizens', 'Eastern Europeans', 'Romania', 'Albanian refugees', 
    'albanians', 'Russians/Soviets', 'Russian voters', 'ethnic russians', 'Serbs', 'people of Eastern European nationality', 
    'bulgarians', 'Latvia', 'Hungarians', 'individuals from Eastern Europe and Asia', 'Lithuanian people', 'Eastern European workers', 
    'Slovenia', 'Russian speakers in Ukraine', 'Eistland', 'Hungarian judges', 'eastern europeans', 'Southeastern Europeans', 
    'Albaníu', 'wealthy Russians', 'Rúmeníu', 'Hvíta-Rússland', 'tékklenskar stúlkur', 'people of Russian descent', 'Russian Nationalists',
    'Latvians', 'Belarusian immigrants', 'Austur-Evrópubúar', 'Russian ordinary citizens', 'Albanans', 'Russian economy participants', 
    'Croatia', 'Albanian and Macedonian refugees', 'Bosnia', 'people with Russian heritage', 'latvia', 'Lettland and Litháen', 
    'Russian fishermen', 'Russian alcoholics', 'Lithuanian speakers', 'Macedonia', 'Albanskir innflytjendur', 'Ukrainians', 
    'Russian sympathizers', 'Estonian citizens', 'Hungarian citizens', 'Russian money holders', 'Slóvenar', 'Russian minority', 
    'Bulgarian people', 'people from old Eastern Europe', 'Prussians', 'russian public', 'Bulgarian and Romanian nationals', 
    'Bulgaria and Romania', 'Bosnians', 'Russian elite', 'Russian People', 'Slovenians', 'Poland and Romania', 
    'Russian children', 'Russian people', 'people from Albania', 'Lithuanian women', 'people of Kosovo', 'rúmenar', 
    'Russian-speaking populations', 'the Russian people', 'Serbs and Croats', 'Croatian people', 'Svartfjallaland', 'Búlgaríu og Rúmeníu', 
    'Hvít-Rússar', 'Tékkar', 'Eystrarsaltsbuar', 'Slovakians', 'Czechs', 'Albanar', 'russian-speaking individuals', 'Croatians', 
    'serbíum', 'Eastern Europe', 'Russians', 'people from Lithuania', 'Slovaks', 'Hungarian athletes', 'Czech women dancing in Iceland', 
    'residents of Estonia and Latvia', 'Russian and Ukrainian workers', 'Eastern European crime organizations', 
    'Slovakia and Czech Republic', 'Rúmenía', 'Baltic nations', 'eastern European criminals', 'people in Kosovo', 'Romanians', 
    'Macedonian immigrants', 'russians', 'Ukrainian citizens', 'Albanians in Bosnia', 'white Christian Russians', 'Estonians', 
    'Rússum (Russians)', 'Czechoslovakians', 'russian criminals', 'nationalities from Albania and Macedonia', 'Eistlandi', 
    'workers from Eastern Europe', 'Eistland og Litháen', 'Estonia', 'Rúmenana', 'Latvian women', 'citizens in Eastern Europe', 
    'Slovakian doctors', 'Hungary', 'Kosovo Albanians', 'Litháen', 'Serbar', 'Russian', 'Ungverjar', 'Estitans and Latvians', 
    'Balkans', 'Rúmenar', 'Kosovo', 'Russian minority groups', 'Ungverjalandi', 'Ethnic Russians', 'Russian society', 'Moldovans', 
    'Specific nations (Romania, Bulgaria)', 'people from Bulgaria', 'Russian-speaking residents of Ukraine', 'Croatians and Latvians', 
    'Romani people', 'Russian speakers', 'Russian-speaking nations', 'Eistar', 'people of Russian origin', 'people from Latvia', 
    'Kosovar Albanians', 'Albaner', '(specifically affected Eastern European populations)'
]

polish = ['Polish women', 'pólskar konur', 'polish gangs', 'Polish', 'Polish workers', 'Polish people', 
          'polish criminals', 'Polish farmers', 'Polish men', 'Polish pilots', 'Polish soldiers', 
          'polish people', 'Poles (Polish people)', 'Polish fishermen', 'Polish nationals', 'Polish craftsmen', 
          'Polish dentists', 'Polish Icelanders', 'polish cleaners', 'Polish immigrants', 'polish immigrants', 
          'Polish doctors', 'Polish authorities', 'Pólverjar', 'pólverjar', 'pólverja', 'Pólverja', 'Pólverjar (Poles)', 
          'Poles', 'Polish', 'Pólarar', 'workers in Poland','workers from Poland and other European countries',
          'people from Poland and former Yugoslavia','people from Poland','Poland', 'Póllands','Pólland', 
          'pólskir ökufantarnir','pólskar konur','pólska yfirstétt','pólska','Pólskar tengdadætur','poles',
          'children of educated Poles','Poles in Iceland','Poles and their work ethic','Poles', 
         
         ]

western_europeans = [
    'English people', 'British and Dutch', 'Svisslendingar', 'British entrepreneurs', 'Irish workers', 'German society', 'Belgium', 
    'Franskir borgarar', 'British thugs', 'citizens of East Germany', 'German people', 'the British and Dutch', 'Austrian society', 
    'germans', 'French children', 'France and Germany', 'people in Britain', 'Northern Irish', 'people from the UK and the Netherlands', 
    'Hollendingum', 'Luxemburg', 'Bretland, Holland og Lúxembúrg', 'British voters', 'residents of Switzerland', 'Bretar (British)', 
    'British fishermen', 'Scottish people', 'Holland', 'elderly Germans', 'the German populace', 'the English', 'N-Englands inhabitants', 
    'Belgía', 'Breta og Hollendinga', 'Liechtenstein and Switzerland', 'France and other countries in crisis', 'British bankers', 
    'Irish citizens', 'Scots', 'foreign nations (e.g., Great Britain)', 'foreigners (Dutch and British)', 'French farmers', 
    'Belgian and French taxpayers', 'specific demographic (Hollendingar)', 'the British public', 'British women', 'Frakklandi', 
    'Bretar og Hollendinga', 'Liechtensteiners', 'young generation of Germany', 'English women', 'Irish farmers', 'locals in France', 
    'German businessmen', 'Austrians', 'Hollendingar (Dutch)', 'Bretum (Britain)', 'Irish immigrants', 'Irish students', 'Ireland', 
    'Luxembourg citizens', 'German and French populations', 'hollanders', 'Dutch', 'white French', 'French tourists', 
    'Franskir mótmælendur', 'people from Britain', 'the Germans', 'Bretar (British people)', 'Scottish', 'German conservatives', 
    'Irish descendants', 'German taxpayers', 'Germany', 'ordinary British citizens', 'Belgian citizens', 'hollendinga', 
    'Irish emigrants', 'Switzerland', 'Brits and Irish', 'French pilots', 'East Germany', 'Irish and Celtic ancestors',
    'Lúxembúrg', 'Austrian citizens', 'Austrian', 'franska þjóðin', 'Þjóðverjar (Germans)', 'people in Ireland', 'Northern Germans', 
    'þýskaar', 'Dutch and British representatives', 'Englendingar', 'Eastern German athletes', 'French and German people', 'Belgian', 
    'Luxembourg', 'foreign entities (Bretar and Hollendingar)', 'German civilians', 'Eastern Germans', 'French', 
    'specific nations (France, Germany)', 'French socialists and technocrats', 'Bretar og Hollandingar', 'British taxpayers', 
    'middle class French', 'Franskar konur', 'German voters', 'West Germans', 'Switzerland residents', 'Scotland', 'the German public', 
    'frakkar', 'Svissarar', 'people from Scotland', 'British population', 'Bretum og Hollendingum', 'workers in Scotland and England', 
    'public employees in France', 'Austrian nationals', 'Austrian people', 'Eastern and Western Germany', 'þjóðverjar', 
    'English prostitutes', 'German culture', 'Bretar/British', 'Britain', 'British public', 'people from the British Isles', 
    'British citizens', 'Frakkar', 'French men', 'Brettar and Hollendingar', 'Austurríkismenn', 'French taxpayers', 
    'Bretar og Hollendingar', 'England', 'Irish women', 'England and Holland', 'french farmers', 'Belgians', 'southern Germans', 
    'French people', 'Germany and France', 'German ruling class', 'Bretar og hollendingar', 'the British working class', 
    'Belgian craftsmen', 'Germans and French', 'Britain and the Netherlands', 'belgians', 'German tourists', 'Holland and England', 
    'Frakkar (French people)', 'Hollendingar', 'Hollendinga', 'the elderly in Germany', 'French citizens', 'hollendingar', 
    'French women', 'German men', 'German drivers', 'Franska þjóðin', 'the British', 'English citizens', 'farmers in Holland', 
    'British', 'German population', 'Frakkar (French)', 'British workers', 'Hollenders', 'Irish and Dutch voters', 'people in France', 
    'Frakkland', 'people from East Germany', 'German citizens', 'Bretar og Holland', 'people in England and Scotland', 
    'Northern Ireland', 'B&H (Bretar og Hollendingar)', 'Irish people', 'British and Dutch people', 'people from France', 
    'British children', 'Great Britain', 'Irish and Scots', 'Holland and Britain', 'unemployed people in France', 'British elite', 
    'British people', 'people from Germany', 'people from Switzerland', 'Scottish descent', 'German and French people', 
    'British and Dutch citizens', 'the German people', 'French fishermen', 'German workers', 'foreign nations (Bretar, Hollendingar)', 
    'Austria', 'Belgar', 'Lúxemborg', 'French voters', 'Frakkarnir', 'East Germans', 'English middle class', 'franskir borgarar', 
    'East German communists', 'French drivers', 'English workers', 'people from Ireland and Scotland', 'Brits and French', 
    'Scottish workers', 'Scottish fishermen', 'Irish', 'the British people', 'Eastern Germany', 'French bankers', 'English', 
    'Hollanders', 'Lúxemborgarar', 'Þjóðverjar', 'people of Britain', 'French, Germans, and British', 'German bankers', 
    'British and Dutch taxpayers', 'Scottish men', 'France', 'austurríkismenn', 'young women in England', 'the French', 'Netherlands', 
    'Englendingum', 'Swiss', 'Austrian immigrants', 'Irish voters', 'British culture', 'UK and Holland', 'Bretar and Hollendingar', 
    'French society', 'German women', 'English parents', 'Germans', 'people from Britain and the Netherlands', 'Scottish nobles', 
    'elderly in Germany', 'Sviss', 'Dutch and British', 'Germany, France, Netherlands', 'people with Irish heritage'
]

southern_europeans = [
    'Italian people', 'Portuguese citizens', 'Spanish fishermen', 'Greeks', 'Spanish guides', 'Greece and Spain', 'greeks', 
    'Grikkir og Spanar', 'Spanish, Portuguese, Greeks', 'Greeks during economic hardship', 'Greek people', 'Italian women', 
    'Greek elite', 'portúgölum', 'Italian', 'Portuguese people', 'young people in Spain', 'Portugal', 'Portúgalar', 'people from Italy', 
    'Spanish households', 'Spain and Portugal', 'Portugals', 'Grikkland', 'Greek', 'Grikklandi', 'Greek citizens', 'people in Portugal', 
    'Spáni', 'Portugalska þjóð', 'grikkjum', 'Grikkar', 'Grikkir', 'Portúgalir', 'Italian winemakers', 
    'Grikkland, Spánn og Portúgal (Greece, Spain, and Portugal)', 'Grikki', 'Greek public', 'people from Portugal', 
    'Spain', 'Spanish people', 'grikkir', 'Italians', 'Italian mob', 'Spanish youth', 'Spanish investors', 'Portuguese immigrants', 
    'Portuguese', 'youth in Spain', 'workers in Spain', 'people from Spain', 'Portugals workforce', 'Italian culture', 'Spanish workers', 
    'European countries (Spain and Greece)', 'northern Italians', 'Italian mafia', 'Spánverja', 'Greek voters', 'people in Spain', 
    'Italian men', 'Spánverjar', 'Italian scientists', 'Spanish and Portuguese people', 'Portúgal', 'Greek workers', 
    'people of Greek descent', 'Italy', 'Grikklands', 'Spanish immigrants', 'Greek and Spanish people', 'Spanish parents', 
    'Portugala og Spánverja', 'residents of Malta', 'people in Greece and Spain', 'Greek society', 'Grikkir (Greeks)', 'wealthy Greeks', 
    'Italian contractors', 'Greek farmers', 'Spánn', 'Spanish citizens', 'Spanish farmers', 'Italians and Greeks', 'Grikkja', 
    'Spaniards (spánverjar)', 'Spaniards', 'Spanish society', 'Spanish job applicants', 'Spanish', 'Portuguese farmers', 'ítalar', 
    'Greek men', 'Portugalska', 'Malta', 'Grikkjum', 'people of Spanish descent', 'certain nations (Portugal, Spain, etc.)', 
    'Spanish population', 'Spanish bankers', 'Greek youth', 'southern Italians', 'Portúgala', 'people from Cyprus', 'Cyprus',
    'Kýpverjar','Kýpurbúar','Kýpur',   
]

icelanders = [
    'Indigenous Icelanders', 'Icelanders (general population)','íslendingar','Icelanders in economy', 'Icelanders in general',
    'Icelander', 'Icelandic citizens','íslendingar (Icelanders)','Icelanders on unemployment benefits', 'Iceland and Icelanders',
    'fátækir íslendingar','Icelanders with prejudices', 'eldri íslendingar', 'certain Icelanders','Icelandic society', 
    'milliardar íslendingar', 'poor Icelanders','low-income Icelanders', 'famous Icelanders', 'non-native Icelanders',
    'Icelandic people', 'Ungir íslendingar', 'common Icelanders','ordinary Icelanders','ICELANDERS', 'modern Icelanders', 
    'austfirðingar (East Icelanders)', 'Self-Determined Icelanders', 'Icelandic men','self-identified Icelanders', 'Islendingar', 
    'Icelanders without higher education','south Icelanders', 'New Icelanders', 'young educated Icelanders',
    'self-sufficient Icelanders', 'educated Icelanders',  'full Icelanders', 'western Icelanders','elderly Icelanders',
    'majority of Icelanders','Icelanders in management positions', 'Polish Icelanders','younger Icelanders', 'new Icelanders', 
    'current Icelanders', 'fishermen and Icelanders', 'local Icelanders who eat traditional foods', 'Icelandic nation', 
    'native Icelanders','Icelanders who enjoy entertainment','northern Icelanders','uneducated Icelanders', 'unemployed Icelanders',
    'young Icelanders', 'the Icelandic nation','influential Icelanders', 'less wealthy Icelanders', 'newly wealthy Icelanders', 
    'Eastern Icelanders','the common Icelander', 'other Icelanders', 'local Icelanders', 'South Icelanders','home-born Icelanders',
    'middle-aged Icelanders','some Icelanders','average Icelanders', 'Íslendingar','debt-burdened Icelanders',
    'young and old Icelanders', 'older Icelanders', 'lazy Icelanders','Icelanders in poverty','newly rich Icelanders',
    'wealthy Icelanders','debt-free Icelanders','general Icelanders','fellow Icelanders', 'working-class Icelanders',
    'eastern Icelanders', 'Ísland (Icelanders)','Icelanders employed in heavy industry', 'the majority of Icelanders', 
    'white Icelanders','Icelander general population','dark-skinned Icelanders','southern Icelanders', 
    'Íslendingar (Icelanders)', 'rural Icelanders', 'Icelander workers', 'independence-minded Icelanders', 
    'Icelanders practicing fisheries', 'north Icelanders', 'Icelanders','all Icelanders','Icelanders living on social security',
    'íslenskir hrokagikkir',
]

nordics = [
    'danish merchants', 'færeyskar konur', 'Færeyja', 'Danes',  'Swedish citizens', 'Norwegian workers', 'Finnish dairy farmers', 
    'norska þjóðfélagið', 'Finnish Nationalists', 'Danish criminals', 'Greenlandic people', 'people from Sweden', 
    'people from Faroe Islands', 'Svíar (Swedes)', 'Norwegian people', 'nómadar í norður Svíþjóð og Finnlandi', 'Norwegian women', 
    'Finnish professors', 'Norwegian descendants', 'grænlendingar', 'Norwegian healthcare workers', 'Danish teachers', 'Færeyjum', 
    'Grænland', 'Finland', 'Norwegian Nationals', 'young Norwegian workers', 'students from Norway and Sweden', 'people with Danish heritage', 
    'Finnar', 'Færeyskir', 'Faroe Islanders', 'Danir', 'Finnish consumers', 'people from Northern Norway', 
    'Finnish agricultural producers', 'Finnish men', 'færeyskur menning', 'Finnar, Svíar, Danir', 'Svíar', 'Swedish women', 
    'Finnish companies', 'Danish society', 'Danish specialists', 'Skandinavíu', 'Færeyingum', 'Færeyinga', 'Færeyingar og Norðmenn', 
    'Norway', 'Færeyingur', 'Færeyjar og Grænland', 'Danish students', 'Ísland, Færeyjar, Grænland, Danmörk og Noregur', 
    'Danish people', 'Norwegian citizens', 'Noreg', 'Swedish people', 'Danish workers', 'Swedish youth', 'Danish taxpayers',
    'Finnish culture', 'Greenland', 'Danish meteorologists', 'finnish people', 'Norwegian nurses', 'people from Norway', 
    'Færeyingar', 'greenlanders', 'Grænlendingar', 'Denmark & Scandinavia', 'Foresters and Greenlanders',
    'Danish public', 'people in Finland', 'Danish elite', 'Norwegian working class', 'Swedish', 'svíar', 'people of Danish descent', 
    'Danish merchants', 'Norwegian and Danish people', 'people in Svíþjóð', 'Swedish agricultural producers', 
    'Danish speakers', 'Northeastern Icelanders', 'residents of West Greenland', 'swedish youth', 'Danish farmers', 'Danish traders', 
    'Danish', 'Finnish people', 'Norwegian elite', 'Norway, Sweden, Finland', 'Svíaríki', 'young people in Faroe Islands', 
    'Finnland', 'Norwegian fishermen', 'Denmark', 'old Icelanders', 'Swedish doctors', 'Faroe Islands', 'Swedes', 
    'norsk fólk', 'individuals in Denmark', 'Swedish teachers', 'Grænlandi', 'people living in Greenland','Danish environmentalists', 
    'Iceland and Norway', 'people from Denmark and Sweden', 'Nordic people', 'those in Denmark', 'people in Denmark and Sweden', 
    'Danir (Danish people)', 'Færeyjingar', 'Færeyskt fólk', 'Grænlands', 'Danmörk', 'Danish hunters', 'Noregs', 'Swedish entrepreneurs', 
    'workers in Denmark', 'Danmörku', 'finnish businessmen', 'Norðmenn', 'Sannir Finnar', 'Færeyjar', 'Norwegian farmers', 
    'Noregur og Færeyjar', 'Finnarnir', 'Ísland og Noregur', 'færeyringarnir', 'Norwegian', 'Finnish drinkers', 'Finnish citizens', 
    'Norwegian skiers', 'norrænna menn', 'Greenlanders', 'Norska veðurfræðingar',  'Danish consumers', 'people from Denmark', 
    'people who speak Danish', 'northern Norwegians', 'Danish family', 'people from the Faroe Islands', 'finland', 
    'people in Norway', 'Danish meat producers', 'Faroese (færeyjingur)', 'women from Greenland', 'Norwegian employers', 
    'norskir skattgreiðendur', 'Noregi', 'population of Sweden', 'Venjulegir Norðmenn', 'Finnish women', 'Færeyingjar', 
    'Danish fanatics', 'people in Denmark', 'Danish citizens', 'Swedes (svíar)', 'Finnish diplomats', 'Danish pensioners', 'Svíþjóð', 
    'Danish small fishermen', 'Norskir karlmenn', 'norskir ríkisborgarar', 'people in Sweden', 'Finnish landowners', 
    'inhabitants of Greenland', 'Færeyingar (Faroese)', 'Danir (Danes)', 'Svíþjóð og Finnland', 'Noregur', 'Finnish Sann Finnar', 
    'Færeyingarnir', 'Finnish and Swedish people', 'Svíar, Danir og Finnar', 'Finnish and Swedish', 'Danish feminists', 'norskir', 
    'Finnish workers', 'Finnish dairy producers', 'Scandinavians', 'Swedish farmers', 'færeyingar', 'ljóshærða Svía', 'Finnish farmers', 
    'Norse people', 'Norwegians', 'Finnish dancers', 'Frumbyggjar Grænlands', 'Danes and Norwegians', 'Danish employers', 
     'Sweden', 'Svíar, Finna og Dana', 'Danish women', 'denmark', 'norwegian children', 'Finnish', 'færeyskur', 
    'residents of Greenland', 'Swedish society', 'fólk á Norðurlöndunum','Norðurlöndin','Norðurlönd','workers in Nordic countries',
    'people in the Nordic countries', 'people in Nordic countries','people from the Nordic countries','nordic countries',
    'North Nordic countries','Nordic countries'
]

east_asians = [
    'Japanese culture', 'Japanir', 'North Korean citizens', 'Kínamenn', 'Mongólakyn', 'Mongolian nations', 'the Chinese', 
    'Japanese pilots', 'Filipino people', 'South Koreans', 'Chinese diplomats', 'Japanese men', 'Chinese domestic workers', 
    'Chinese people', 'Filipinos', 'Japan', 'Vietnamese civilians', 'Innri Mongólia', 'Kahnar Mongóla', 'Suður-Kóreumenn (South Koreans)', 
    'barstelpurnar (women in Thailand)', 'Mongóla kynsins', 'women from Thailand', 'Vietnam', 'citizens of China', 'Chinese population', 
    'kínverskur', 'Japanese people', 'people in Kína', 'people from North Korea', 'Filipino women', 'people from Vietnam and China', 
    'Mongolíubúar', 'A-Tímorbúa', 'Singapúr', 'Norður Víetnam', 'millistéttin í Kína', 'Chinese technology employees', 'workers in China', 
    'Kínverja', 'Mongólar', 'people from Singapore', 'Cambodian people', 'Chinese culture', 'Kínverskir verkamenn', 'North Korea', 
    'Taiwan', 'Chinese healers', 'Japanese community', 'íbúar Singapora', 'Mongólarnir', 'Myanmar', 'newly rich Chinese', 
    'kínverska millistéttin', 'people of Japan', 'kínakomma', 'Kínverska millistétt', 'Vietnam veterans', 'Filipino nationality', 
    'Chinese women', 'Vietnamese people', 'Vietnamese immigrants', 'Vietnamese', 'kínverska þjóðin', 'Víetnam', 'Kínverjum', 
    'Kínverjinn', 'China as a whole', 'Filippseyjar', 'fátækir í Kína', 'Chinese citizens', 'Kínverjar', 'Laos', 'people of North Korea', 
    'Vietnamese women', 'Chinese researchers', 'Kínverjar og Japanir', 'Kínverskir ferðamenn', 'Suður Víetnam', 'Chinese miners', 
    'Kínverjar (Chinese)', 'people of Chinese descent', 'Ríki eins og Kína', 'Chinese students', 'Indónesar', 'Filipino workers', 
    'Kambódía', 'Filippseyingar', 'Mongolos', 'Singapore', 'South Korea', 'kínakommar', 'Chinese migrant workers', 'Chinese immigrants', 
    'Tælendingar', 'kínversk millistétt', 'samfélagið í Kína', 'Burmese people', 'Chinese', 'Kínverskir neytendur', 'Kines (Chinese)', 
    'Malaysians', 'Japanese women', 'politicians in Korea and Taiwan', 'people from northeastern Thailand', 'Chinese community', 
    'people in Búrma and China', 'Mongóla', 'Kínverjarnir', 'people in China', 'Japanese society', 'people from China', 
    'Kínversku ferðamenn', 'Kínverska þjóðin', 'people in North Korea', 'rich Chinese', 'Chinese entrepreneurs', 'poor Chinese', 
    'Chinese families', 'Japanese', 'thailenska konan (Thai woman)', 'Chinese businessmen', 'Asian Mongols', 'chinese', 
    'People in China', 'Indonesians', 'Filipino nursing staff', 'Kína', 'North Koreans', 'other ethnic groups in Myanmar', 
    'Chinese coal miners', 'Chinese travelers', 'Innri-Mongólía', 'China', 'Víetnamar', 'Mongólum', 'Suður Kórea', 'Norður Kórea', 
    'Kínar', 'Chinese individuals', 'kínverjar', 'Kóreanskir og Japanskir kaupsýslumenn', 'Chinese workers', 
    'Chinese engineers', 'Taiwanese', 'Chinese tourists', 'Mongolians', 'people in Japan', 'kínverskir fjárfestar', 
    'people with Mongolian ancestry', 'Han-Kínverjar', 'Burmese', 'women from the Philippines', 'people from Thailand', 
    'Chinese investors', 'Vietnamese individuals', 'Japani', 'families from Thailand', 'the people in China', 'Norður Kóreu', 
    'Mongols', 'Chinese (Kínverjar)', 'Chinese laborers', 'people from Malaysia', 'mongólsku kyni', 'tævönungar', 'Chinese society', 
    'Burmans'  
]

south_central_asians = [
    "women in Bangladesh","Bangladeshis","Bangladesh","residents of India","poor populations in India","poor farmers in India",
    "people in India","people from India","Indverjar (Indians)","Indians","Indian people","Indian men","Indian individuals",
    "Indian healthcare staff","Indian environmentalists","Indian Workers","Indian","India","Indlandi","Indland","indverjarnir",
    "indverjar","Indverska menning","Indverjum","Indverjar (Indians)","Indverjar","Indverjana","Indverja","residents of Maldives",
    "Maldivian people","Nepali","people in Pakistan","people from Pakistan","Pakistön","Pakistanis","Pakistani youth",
    "Pakistani women","Pakistani people","Pakistani immigrants","Pakistani doctors","Pakistani","Pakistan","people in Sri Lanka",
    "Sri Lankans","Sri Lanka","Turkmens","Turkmenistan","Turkmen people","Túrkmenar","Uzbeks","Uzbekistanis","Uzbekistan",
    "young exiled Tibetans","Tibetans","Tibetan people","Tibet"
]

# Suggested extra category to keep things tidy
middle_easterners = [
    "people in Afghanistan","people fleeing Afghanistan","afghans","Sudanese and Afghan","Afghans","Afghanistan","Afghan youth",
    "Afghan women","Afghan tribes","Afghan refugees","Afghan politicians","Afghan people","Afghan men","Afghan inhabitants",
    "Afghan farmers","Afghan families","Afghan drug producers","Afghan citizens","Afghan","Afganistan","Afgani", 'Armenia',
    'Armenians','Armenian people','Georgía','Georgians','Georgíska þjóðin','Georgíumenn', "people from Azerbaijan","Azerbaijanis",
    "Azerbaijan","Azerbaidjan","Azerbadjan","the regime of Bahrain","people in Egypt","people from Egypt","Egyptians","Egyptian voters",
    "Egyptian people","Egyptian families","Egyptar","Egypt","women in Iran","people from Iran","citizens of Iran","Iraqi and Iranian people",
    "Iranians/Persians","Iranians","Iranian women","Iranian society","Iranian people","Iranian individuals","Iranian hackers",
    "Iranian citizens","Iranian athletes","Iranian","Iran","Iranians","the Iraqi women","the Iraqi people","residents of Iraq",
    "people of Iraq","people in Iraq","Iraqis","Iraqi refugees","Iraqi people","Iraqi immigrants","Iraqi civilians","Iraqi children",
    "Iraqi","Iraq residents","Iraq","Irakians","the people of Israel","the Israelites","people of Israel","people in Israel",
    "people from Israel","nation of Israel","israelíta","israelis","israeli","acting Israelis","Palestinians and Israelis",
    "Israelsþjóð","Israelsmenn","Israels","Israelites/Palestinians","Israelites","Israelite society","Israelis and Palestinians",
    "Israelis and Arabs","Israelis & Palestinians","Israelis","Israelians","Israeli state", "Israeli society","Israeli scientists",
    "Israeli people","Israeli individuals","Israeli civilians","Israeli citizens","Israeli children","Israeli and Palestinian people",
    "Israeli academics","Israeli Workers","Israeli State","Israeli People","Israeli Palestinians","Israeli Arabs","Israeli","Israelar",
    "Israel.","Israel","ISRAELI","Israelis","Jordanians","Jordanian people","women in Kuwait","citizens of Kuwait","Kuwaitis","Kuwait",
    "Lebanese men","Lebanese descendants of the Phoenicians","Lebanese","Lybia","people in Palestine","Palestine","palestínumenn",
    "palestínska þjóðin","Palestínuvillimenn","Palestínuríkið","Palestínumönnum","Palestínumenn","Palestínufólk","Palestínuarabinn",
    "Palestínuarabar","Palestínu-Araba","Palestínu","people from Qatar","Qataris","Qatari","Qatar workers","Qatar laborers","Qatar",
    "Katarar", "Katar","Soudi-Arabía","women in Saudi Arabia","people in Saudi Arabia","people from Saudi Arabia",
    "Saudi arabians", "Saudi Arabíu","Saudi Arabía","Saudi Arabs","Saudi Arabien","Saudi Arabians","Saudi Arabian citizens",
    "Saudi Arabian","Saudi Arabia","Sádarnir","Sádar","Sádana","Sudanese","Sudan","South Sudanese","residents of Syria",
    "rebels in Syria","people in Syria","citizens of Libya and Syria","Syrians","Syrian refugees","Syrian rebels","Syrian people",
    "Syrian civilians","Syrian citizens","Syria","Assyrians","Sýrlenskir flóttamenn","Sýrlendingar","Sýrlendinga","Sýrlands",
    "Sýrland (Sýrlenska þjóðin)","Sýrland","rural people in Northern Turkey","people from Turkey","Turkey","turkish people",
    "small Turkish people","Turkish youth","Turkish workers","Turkish villagers","Turkish people","Turkish immigrants",
    "Turkish citizens","Turkish","Tyrklandsbúa","Tyrklands","Tyrkland","Sameinuðu Arabísku furstadæmin","people in Yemen",
    "people from Yemen","civilians in Yemen","Yemenis","Yemeni people","Yemeni architects","Yemen","Jemen",
]

africans = [
    "Ivory Coast","Gambian people","Gambíumenn","people from Ghana","Ghanaians","people in Liberia","Liberians","somali refugees",
    "people in Somalia","Sómali","Somalis (sómali)","Somalis","Somaliers","Somalians","Somalia","Somali pirates","Somali people",
    "Somali immigrants","Somali","Malinese villagers","people in Nigeria accused of witchcraft","people in Nigeria","people in Niger",
    "people from Nigeria","nigerískar konur","nigerians","individuals with degrees from Nigeria","Nigeríumenn","Nigerians",
    "Nigerian women","Nigerian scammers","Nigerian citizens","Nigeria","Nígeríu","Nígería","Kamerún","people from Congo","Congolese",
    "Congo","Eritreans","Ethiopians","Ethiopian people","Ethiopian men","Ethiopia","Kenyan tribes","Kenyan immigrants",
    "Kenyan citizens","Rwandans","people in Somalia","Somalians","Somalia","Sómalísvertingjar","Sómalskir sjóræningjar",
    "Sómali","Sómalar","Somalis (sómali)","people in Uganda","Ugandan citizens","common people in Angola","Angola","Angólamenn",
    "Mozambique","people in Namibia","Namibíumenn","Namibians","Namibian citizens","white population in South Africa",
    "white farmers in South Africa","white South Africans","people in South Africa","black South Africans","South Africans",
    "South Africa","fólk í Mósambík","people in Zimbabwe","Zimbabwians","Zimbabwei","Zimbabweans","Zimbabwe",    
]

north_americans = [
    "young Americans","uneducated Americans","the Americans","rich Americans","poor Americans","minority Americans",
    "middle-aged Americans","low-income Americans","americans","West Americans","US Americans","Scandinavian Americans",
    "North Americans","Native Americans","Kanar (Americans)","Bandaríkjamenn (Americans)","Bandaríkin (Americans)",
    "Ameríkananum (Americans)","Americans and Canadians","Americans"," Americans", "people from the U.S.A.","U.S.A.",
    "voters in the USA","unmarried people in the USA","the USA","states in the USA","southerners USA",
    "residents of the USA","population of the USA","people in the USA","people in USA","people from the USA",
    "people from the Southern states of the USA","people from the South USA","people from the Northern states USA",
    "people from USA","ordinary people in USA","northern and southern regions of the USA","general public in the USA",
    "fat people in the USA","children in the USA","young people in the US","voters of the US electoral system",
    "the US elite","the US","people from southern US states","ríkjum/borgum Bandaríkjanna","people in BNA",
    "kotungar í BNA","bandaríkjanna","bandaríkjamenn","Spænskumælandi Bandaríkjamenn","Bandaríkjanna","Bandaríkjanir",
    "Bandaríkjan","Bandaríkjamönnum","Bandaríkjamenn (Americans)","Bandaríkjamenn","Bandaríkjamanna","Bandaríkjadmen",
    "Bandaríkja borgarar","kanadamennir","Kanadamenn","Kanadabúar","Kanada","people in Canada","people from Canada",
    "Canada","Canadians","Canadian regions","Canadian people","Canadian fishermen","Canadian","people in Mexico",
    "people from Mexico","Spanish speakers from Mexico","Mexico","Mexicans","Mexicanos","Mexican workers","Mexican population",
    "Mexican people","Mexican men","Mexican immigrants","Mexican drug cartels","Mexíkó","Mexíkanar","Mexícó",
    "people from El Salvador","citizens of El Salvador","El Salvador","Guatemalans","Guatemalan women","Guatemala",
    "Honduras","Hondurans","Hondúras","Nicaraguan freedom fighters","Panamanians","Panama","Bahamian people","the Cuban people",
    "people in neighboring countries to Cuba","people from Cuba","citizens of Cuba","Cubans","Cuban residents","Cuban refugees",
    "Cuban people","Cuban exiles","Cuban elders","Cuban doctors","Cuba","kúbanar","Kúbversku þjóðinni","Kúbverjar",
    "Kúbumenn","Kúbu","Kúba","people of Haiti","people in Haiti","Haitians","Haiti citizens","people from Jamaica",
    "Jamaicans","Jamaican citizens","workers in Trinidad and Tobago","Puerto Ricans",
]

south_americans = [
    "people from Argentina and Brazil","Argentina","teams in Argentina","people from Argentina and Brazil","Argentinians",
    "Argentines","Argentínska þjóðin","Argentína","children in Brazil","brazilians","Brazilians","Brazilian women",
    "Brazilian population","Brazilian people","Brazilian culture","Brazilian citizens","Brazilian","Brazil","Brasilíumenn",
    "Brasilía","workers from Chile","Chileans","Chilean people","Chilean conservatives","Chile","people in Ecuador and Colombia",
    "Colombians (kólumbíumenn)","Colombians","Colombian women","Colombian farmers","Kólumbíufólk","Ecuadorians","Ecuador",
    "Ekvador","Peruvians","Surinamese","Uruguayans","íbúar Venesúela","residents of Venezuela","refugees from Venezuela",
    "people in Venezuela","people from Venezuela","fátæka fólkið í Venesúela","Venezúela","Venezuelans","Venezuelan refugees",
    "Venezuelan people","Venezuela","Venesúela","Venesuela",
    
]

oceanians = [
    "Australians","Australian people","Australian aboriginals","Australian","Australia","Aboriginal Australians",
    "New Zealanders","New Zealand tourists","New Zealand cooks","Nýsjálendingar","Fijians","Papúar á Nýju-Gíneu",
    "Papúar","Papuan","Samoa","Tuvalu citizens",
]

foreigners = [
    'kvótaflóttamenn', 'aðrir útlendingar', 'foreigners viewing games', 'foreign spouses', 'foreign workers', 'foreign landlords', 
    'people from foreign cultures', 'húsnæðislausir hælisleitendur', 'parents of foreign origin', 'foreign speakers', 
    'people in Iceland', 'foreign invaders', 'foreign men', 'foreign tourists', 'flóttamenn (refugees)', 'foreign stakeholders', 
    'flóttamenn', 'Hælisleitendur', 'útland þjóð', 'foreign countries', 'immigrants or foreigners', 'Foreign Nations', 
    'ill-educated foreigners', 'immigrants and foreigners', 'utlendingar', 'útlendingar', 'Foreign bondholders', 
    'people of non-Icelandic descent', 'foreigners opposing EU membership', 'foreign-born individuals', 'ólöglegir innflytjendur', 
    'foreigners lodging applications for asylum', 'dökkir útlendingar', 'foreign women', 'Foreign immigrants', 'developed countries', 
    'those supporting foreign interests', 'flóttafólk', 'foreign salesmen', 'flóttabörn', 'útlanderinnflytjenda', 
    'foreigners seeking asylum', 'people with foreign surnames', 'innflytjendum', 'útlendingar (foreigners)', 'hælisleitendur', 
    'foreigner workers', 'foreign thieves', 'immigrants', 'indigenous Icelanders', 'útlendinga', 'fámenn eyríki', 
    'foreign children', 'people living in Greenland', 'foreign asylum seekers', 'útlendingum', 'foreign nations', 
    'innflytjendur', 'foreign crime gangs', 'migrants', 'Ólöglegir innflytjendur', 'refugees and immigrants', 'illegal immigrants', 
    'útlendar ljósmæður', 'foreign customs', 'wealthy foreigners', 'flóttamenn og hælisleitendur', 'útlendar flokkana', 'Flóttamenn', 
    'foreigners or critics from outside Iceland', 'people who sing in foreign languages', 'nýbúar', 'foreign people', 'landflótta fólk', 
    'foreign fishermen', 'Foreign workers', 'foreign sellers', 'people from foreign countries', 'foreign nurses', 
    'people from foreign descent', 'foreign individuals', 'those who adopt foreign customs', 'Foreigners', 'foreign families', 
    'ESB útlendingar', 'people of foreign descent', 'foreign fishers', 'foreign residents', 'people of foreign origin', 
    'foreign immigrants', 'flóttamanna', 'foreigners in Iceland', 'flóttamönnum', 'foreigners', 'foreigneers', 'foreign labor', 
    'foreign couples with children', 'uneducated foreigners', 'ólöglega innflytjendur', 'foreign mothers', 'túrista hælisleitendur', 
    'foreigners seeking refuge', 'foreigner', 'ólaglegur útlendingar (illegal immigrants)', 'foreign nationals', 
    'Foreign workersforeign children', 'foreign filmmakers', 'flóttakonur', 'small island nations', 'foreign criminals', 
    'being ruled by foreigners', 'foreign prisoners', 'former colonies', 'útland', 'foreigners from Muslim countries', 
    'foreign citizens'
]

In [ ]:
christianity = [
    'Christianity',
    'Christians',
    'Christians/Catholics',
    'clergy',
    'people of the church',
    'Catholics',
    'church authority figures',
    'church leaders',
    'followers of Jesus Christ',
    'critics of Christianity',
    'Mormons',
    'believers in Christianity',
    'those who believe in biblical accounts',
    'preachers',
    'the church',
    'members of the national church',
    'kirkjan',
    'Christian fundamentalists',
    'those who follow Biblical orders',
    'state church members',
    'Kristnir menn',
    'Wstboro Baptists',
    'Kristnir',
    'members of the church',
    'Christian priests',
    'Catholic Church',
    'kaþólska kirkjan',
    'believers in Jesus',
    'Christian believers',
    'christians',
    'heilagsandahoppur',
    'Christian leaders',
    'followers of the Bible',
    'Christian religious groups',
    'religions including Jéhovahs Witnesses and Catholics',
    'creationists',
    'evangelical Christians',
    'evangelicals',
    'extremist Christians',
    'kristin trú',
    'kristið fólk',
    'kristnar',
    'kristni',
    'kristnir',
    'mormons',
    'ofurkrissar',
    'traditional Christians',
    'white Christians',
    'White Christians',
    'Western Christians',
    'Christian Zionists',
    'Christian asylum seekers',
    'Christian community',
    'Christian extremists',
    'Christian nations',
    'Christian people',
    'Eastern Orthodox Christians',
    'Greek Orthodox',
    'Protestants',
    'Rússneska rétttrúnaðarkirkjan',
    'Sjöunda dags Aðventistar',
    'Waldensians',
    'catholics',
    'churchgoers',
    'followers of faith',  # (Christian-leaning phrasing in context)
    'strong Christians',
    'Hvítasunnusöfnuðurinn',
    'Adventists',
    'Jehovah\'s Witnesses',
    'religious individuals (specifically the Catholic Church)',  # generic phrasing
    'religion (Protestantism)', 
]

judaism = [
    'Jews',
    'Jewish people',
    'gyðingar',
    'Gyðingar',
    'Jewish religion',
    'Israeli Jews',
    'Jewish',
    'Jewish community',
    'Jewish extremists',
    'people adhering to Jewish faith',
    'jews',
    "American Jews",
    "Zionist Jews",
    "general Jews",
    "Ashkenazi Jews",
    "Jewish",
    "jews",
    "Jewish religion",
    "Austrian Jews",
    "Israeli Jews",
    "jew",
    "Sephardic Jews",
    "Jewish descendents",
    "Jerusalem Jews",
    "people adhering to Jewish faith",
    "Torah Jews",
    "Jewish People",
    "Jews during Soviet era",
    "Ukrainian Jews",
    "Jewish culture",
    "Jewish community",
    "Jewish Culture", 
    "Jewish children",
    "Jewish scholars",
    "Jewish media representatives",
    "Jewish individuals",
    "Orthodox Jews",
    "Ethiopian Jews",
    "Jewish population",
    "Jewish families",
    "true Jews",
    "non-Jewish people",
    "jewish people", 
    "Jews",
    "people who dislike Jews",
    "Jews/Soros",
    "Estonian Jews",
    "Jews in Iceland",
    "gyðingar (Jews)",
    "Haredi Jews",
    "South American Jews",
    "Jewish settlers",
    "German Jews",
    "Jewish men",
    "Jewish women",
    "Haredim Jews",
    "Jewish critics of Israel",
    "Karaite Jews",
    "Jewish society",
    "Hellenized Jews",
    "Hellenistic Jews",
    "orthodox Jews",
    "European Jews",
    "Isreali Jews"
    "Sérstaklega gyðingar",
    "gyðingurinn",
    "Gyðinga",
    "gyðingum",
    "Gyðingar",
    "Gyðingdómur",
    "gervigyðingar",
    "strangtrúaðra gyðinga",
    "júðabörn",
    "Júðar"
]

non_abrahamic_religions = [
    'Other religions', 'nepalese hindus', 'Búddismi', 'Zoroastrians', 'heiðnir norrænir menn', 'hindúatrú', 'old religions', 
    'Asatruarmenn', 'heiðnir menn', 'non-Christian religious groups', 'Deists', 'deists', 'Shintóistar', 'followers of Dalai Lama', 
    'ásatrúarfólk', 'Buddhists', 'Hinduism', 'Búddistar', 'Asatru', 'Tibetan Buddhists', 'heiðnir', 'ásatrúarmenn', 'Hindúar (Hindus)', 
    'Buddhism', 'modern religions', 'people practicing certain religious rituals', 'Jainism', 'Heathens', 'Hindu', 'Hindúismi', 'asatru', 
    'Asatru practitioners', 'Buddisman', 'primitivist religions', 'Búddatrúar', 'Hindu Rohingya', 'Bhuddists', 'traditional religions', 
    'buddhists', 'other non-Christian religions', 'followers of non-Abrahamic religions', 'Buddhists and Hindus', 'hindus', 'heathens', 
    'Hindus', 'Confucianists', 'heiðni', 'Hinduisma', 'people of Eastern religions', 'Vodou religion', 'ásatrú', 'ásatrúar', 
    'heathen Icelanders', 'followers of primitive religions', 'shamanists', 'Hindúar', 'other religions', 
    'followers of Zoroastrianism'
]

atheists = [
    'trúlaus', 'people without religious affiliation', 'the agnostic/atheist', 'those who are atheists or have differing beliefs', 
    'trúlausir (atheists)', 'trúleysingjar', 'Atheist individuals', 'atheists/agnostics', 'trúleysi', 'öfgatrúleysingar', 'the atheists', 
    'Atheism', 'atheists', 'öfgatrúleysingjar', 'people who identify with atheism', 'atheists or non-believers', 'trúlause', 'trúlausu', 
    'critics of religious beliefs', 'trúleysigjar', 'trúleysisbloggari', 'supporters of new atheism', 'Atheists', 'irreligious people', 
    'hardline atheists', 'assertive atheists', 'trúleysingar', 'trúlausir', 'trúleysiströllin', 'extreme atheists', 'Trúleysingjar', 
    'agnostics', 'agnostics/atheists', 'Trúlausir', 'trúleysingjatalibanar', 'Agnostics/Atheists', 'new atheists', 'trúlaust fólk', 
    'trúleysingjar (atheists)', 'trúlausa', 'secular individuals', 'trúlausir menn', 'young atheists', 'trúleysingja', 
    'talsmenn trúleysisins', 'scientists/atheists', 'trúlausir (trúleysingjar)', 'critics of religion', 
    'people who reject religious morals', 'so-called atheists', 'those who believe in non-religious perspectives', 
    'hinir trúlausu', 'people with anti-religious sentiments', 'trúleysingjar/trúlaust fólk'
]

general_religion = [
    'öfgamenn',
    'children with religious upbringing',
    'people with different religious beliefs',
    'believers',
    'religions',
    'religious groups',
    'religious leaders',
    'religious individuals',
    'religious people',
    'theists', 'theistic believers',
    'professional biblical scholars',
    'religious institutions',
    'religious figures',
    'faith',
    'Gnostics',
    'religious believers',
    'people who believe in faith-based practices',
    'the religious',
    'religious beliefs',
    'people with strong faith',
    'people with faith',
    'Abrahamic religions',
    'people who believe in religion',
    'people questioning religious beliefs',
    'fundamentalists',
    'people with religious beliefs',
    'those who hold religious beliefs',
    'people who believe in God',
    'people of faith',
    'religious communities',
    'monotheists',
    'religion',
    'religious minorities',
    'people with strong religious beliefs',
    'other religious groups',
    'religious organizations',
    'religious parents', 'children of religious parents',
    'believers in different interpretations of religion',
    'people with differing religious views',
    'individuals with differing religious interpretations',
    'certain religious groups',
    'those with religious beliefs',
    'the religious or spiritual community',
    'Religious leaders',
    'people believing in religious prophecies',
    'Religion',
    'people who follow Abrahamic religions',
    'proponents of various religious texts',
    'religious fundamentalists',
    'people who practice religion',
    'specific religions',
    'people who believe in different religious interpretations',
    'people with religious opinions',
    'those who follow religious sentiment',
    'religious extremists',
    'religious authorities',
    'people of different religions',
    'religious practitioners',
    'religious groups (Christians vs Muslims)',
    'believers in different religions',
    'religious followers',
    'foreign religions',
    'all religions',
    'specific religious groups',
    'religious officials',
    'those raised in a religious environment',
    'those with religious upbringing',
    'religious schools',
    'people who follow religious texts',
    'religious community',
    'believers in religious myths',
    'people who misunderstand religion',
    'those who commit violence in the name of religion',
    'people who rely on religious moral frameworks',
    'religious conservatives',
    'believers in religion',
    'followers of religion',
    'believers of different religions',
    'believers of certain religions',
    'speakers of religion',
    'moralists/religious adherents',
    'members of religious groups',
    'religious belief',
    'followers of organized religion',
    'religious scholars',
    'people following religious laws',
    'those who follow religious teachings',
    'believers in organized religion',
    'religious authors',
    'people following religious beliefs',
    'people of other religions',
    'people with religious affiliations',
    'individuals practicing religion',
    'people following false religions',
    'organized religion',
    'extreme religious groups',
    'those who follow structured methodologies in religion',
    'gullible individuals donating to religious figures',
    'people who donate to religious organizations',
    'religious zealots',
    'foreign religious groups',
    'Religious individuals',
    'those of different religions',
    'followers of various religions',
    'followers of other religions',
    'people carrying religious texts',
    'religion advocates',
    'religious officiants',
    'people who follow religious beliefs',
    'other religious denominations',
    'people with varying expertise in religious matters',
    'extreme religious individuals',
    'members of certain religious groups',
    'Religious groups',
    'minor religious groups',
    'religious experts',
    'minoritized religious groups',
    'members of religious institutions',
    'believers in other religions',
    'people who are religious',
    'specific religious followers',
    'religious majority',
    'followers of different religions',
    'Religious People',
    'organized religions',
    'people who interpret religious texts',
    'religious faith',
    'followers of specific religious beliefs',
    'people of various religions and non-religious people',
    'people who interpret religious texts literally',
    'those who are religious',
    'those with different religious backgrounds',
    'believers of religious faiths',
    'people of certain religions',
    'certain religions',
    'religious movements',
    'followers of certain religious beliefs',
    'conservative religious individuals',
    'self-righteous religious individuals',
    'believers of other religions',
    'North American religious groups',
    'the religious community',
    'believers in religious texts',
    'those in religious institutions',
    'religious bloggers',
    'different religious groups',
    'religious persons',
    'people involved with religious institutions',
    'followers of religions',
    'people who believe in God or religious texts',
    'the deeply religious',
    'people involved in religion',
    'believers in supernatural or religious ideologies',
    'people involved in religious political organizations',
    'religious fanatics',
    'beliefs of religious individuals',
    'people who follow religious organizations',
    'minority religions',
    'members of religious organizations',
    'different religions',
    'fundamentalist religious groups',
    'individuals holding differing religious views',
    'people who read religious texts',
    'religious educators',
    'religious interpreters',
    'people who believe in different religions',
    'individuals who believe in non-atheistic religions',
    'religious group',
    'people based on religion',
    'different religious beliefs',
    'followers of certain religions',
    'others (related to religion)',
    'religious professionals',
    'children from different religious backgrounds',
    'People subscribing to the religion',
    'people who practice foreign religions',
    'other religious organizations',
    'children raised in religious environments',
    'believers in religious faith',
    'those who believe in religion',
    'people who misuse religion',
    'believers of a religion',
    'certain religious individuals',
    'people interpreting religious texts',
    'people who misuse religious texts',
    'religious academics',
    'religious individuals opposed to LGBTQ+ rights',
    'people using religion',
    'people who believe in religious authority',
    'religious institutions and their followers',
    'literalist religious groups',
    'people associated with religious activities',
    'those who hold prejudiced views based on religious beliefs',
    'extremist religious people',
    'believers of various religions',
    'people who hold opposing views on religion',
    'people from religious communities',
    'general religious followers',
    'people with different religions',
    'people with certain religious beliefs',
    'religious politicians',
    'people who express religious beliefs',
    'individuals with strong religious beliefs',
    'people who support religions',
    'particular religions',
    'people who follow religion',
    'church/religious leaders',
    'people who support state-funded religious institutions',
    'religious leaders/rainbow priests',
    'individuals practicing literal interpretations of religion',
    'believers in literal interpretation of religious texts',
    'interpreters of religious texts',
    'religous women',
    'people with religions',
    'follower of religions',
    'followers of a specific religion',
    'people with religious belief',
    'people who use religious texts to justify slavery and violence',
    'religious individuals who overlook the morality of their texts',
    'people with certain religious beliefs',
    'people who follow religious rules',
    'certain groups of religious individuals',
    'followers of certain religious traditions',
    'children of different religions',
    'religious apologetics',
    'believers in organized religions',
    'members of organized religion',
    'supporters of organized religions',
    'followers of organized religions',
    'people who believe in religions',
    'different religious practitioners',
    'believers (religious people)',
    'followers of religious practices',
    'religious characters',
    'believers of various religious creation myths',
    'those who interpret religious texts',
    'believers in different religious concepts',
    'Religious fanatics',
    'European religious people',
    'people of different races and religions',
    'fundamentalist religions',
    'followers of dominant religions',
    'people with religious differences',
    'people in religious communities',
    'people who compare religious texts',
    'believers in religious doctrines',
    'people involved in religious organizations',
    'those who interpret religion',
    'religiously marginalized groups',
    'individuals involved in religious discussions',
    'self-defined religious groups',
    'people in religious groups',
    'small religious communities',
    'individuals who hold different religious views',
    'people who hold extreme religious beliefs',
    'unaffiliated religious groups',
    'speaking subjects of subjects like languages or religion',
    'those who follow religion',
    'men in religious institutions',
    'religious granters',
    'religious denominations',
    'People who believe in religion',
    'state and organized religions',
    'institutional religion',
    'strict religions',
    'Religious institutions'
]

In [ ]:
muslim = [
'Muslims','íslam', 'múslímar','íslamista', 'íslamistum', 'íslamistar', 'islam', 'Islam',
'Islamskir fræðimenn', 'Islamists', 'Islamic extremists', 'women in Islam',
'followers of Islam', 'islamists', 'Islamic countries', 'radical Islamists',
'Islamic culture', 'islamist groups', 'followers of islam', 'Islamic terrorists',
'european Islamists', '“Islam”', 'individuals opposing violent interpretations of Islam',
'islamsku norsku ríkisborgara', 'individuals associated with Islam', 'Islam adherents',
'people from Islamic backgrounds', 'people of the Islamic faith', 'Islamist Extremists',
'Islamic State', 'Islamic Extremists', 'Islamist extremists', 'islamista',
'Islamic nations', 'Islamic individuals', 'people who leave Islam',
'believers of Islam', 'Islam followers', 'people discussing Islam', 'Islamic community',
'Islamic leaders', 'Islamic groups', 'Radical Islamists', 'Islamska samtök',
'Islamistarnir', 'Islamic Groups', 'Islamic immigrants', 'people who support Islam',
'Islamic women', 'Islamic teachers', 'those sympathetic to Islam',
'Íslamistar (Islamists)', 'Different Islamic sects', 'Islamic scholars',
'believers in Islam', 'Islamic followers', 'Islamism',
'people criticizing Islamic jurisprudence', 'people from Islamic countries',
'Christian and Islamic teachings', 'people adopting Islam', 'supporters of Islam',
'Islamic Men', 'Islamistar', 'Islamic jihadists', 'Islamic Jihad', 'Islamic world',
'Followers of Islam', 'ISLAM', 'Islam believers', 'Sunni Islam', 'people of Islam',
'Islamic states', 'Islam supporters', 'extremist Islamists', 'Islamist communities',
'Islams', 'people who travel to Islamic countries', 'Islamic people',
'people who buy books about Islam', 'islamic individuals',
'people with Islamic names', 'women in Islamic countries',
'nationality of Islamic countries', 'Islamic organizations', 'Islamic men',
'those who misinterpret Islam', 'proponents of Islam', 'múslimar', 'múslimun',
'Múslimar', 'Múslima', 'öfgamúslimar', 'Múslimum', 'múslima',
'múslimskir karlmenn', 'múslimum', 'múslimatrúar', 'múslimir', 'múslimska homma',
'múslimska', 'múslimskra hryðjuverkasveita', 'múslimskar konur', 'Súnní múslimar',
'Bræðralag Múslima', 'múslimaríkin', 'Súnní Múslimar', 'öfgamúslima',
'Úrslit Múslima', 'múslimakonur', 'múslima-ríki', 'múslimaríkjum', 'músliminn',
'Sunni múslimar', 'Öfgamúslimar', 'Múslimir', 'Múslima/ Múslimar',
'múslimsku landanna', 'múslimi', 'múslimarnir', 'muslims', 'Muslim communities',
'Muslim groups', 'Muslim Brotherhood', 'Muslim women', 'Muslim countries',
'Muslim men', 'migrant Muslims', 'immigrants from Muslim countries', 'muslim',
'Muslim individuals', 'Muslim community', 'Muslim refugees', 'Muslim extremists',
'former Muslim women', 'extremist Muslims', 'Arabs (Muslims)', 'Ahmadiyya Muslims',
'Muslims and Jews', 'muslim community', 'muslim countries', 'specific Muslim countries',
'Muslim-majority countries', 'Muslim migrants', 'margarísar',
'Muslims and immigrants', 'Muslim states', 'Muslim', 'Muslims and Christians',
'Muslim people', 'Muhammadan Muslims', 'Shia Muslims', 'muslim men',
'Muslim immigrants', 'Bosnian Muslims', 'Sunni Muslims', 'Muslims in general',
'muslim populations', 'muslim immigrants', 'Muslim population', 'Muslim nations',
'muslimar', 'Christians and Muslims', 'Mu Muslims', 'foreigners from Muslim countries',
'Christian and Muslim communities', 'muslima', 'Rohingya Muslims', 'young Muslims',
'Muslims in Russia', 'Muslim populations', 'fathers in Muslim culture',
'Danish Muslims', 'Muslims and people from the Middle East',
'Muslims in Western China', 'muslim women', 'people from Muslim backgrounds',
'Muslima', 'extreme muslims', 'Turkish Muslims', 'Arabs/Muslims',
'Súnní og Shíamuslima', 'Muslim clerics', 'muslim clerics', 'Muslimar',
'refugees from Muslim countries', 'MUSLIM', 'Syrians and Muslim countries',
'Shia and Sunni Muslims', 'Arab-Muslims', 'General Muslim Population',
'm Muslims', 'foreign Muslims', 'muslimaríkjum', 'Shiite Muslims',
'certain nationalities (Muslims)', 'Arabs and Muslims', 'muslim people',
'Muslim Arabs', 'Pakistani Muslims', 'Iranian Muslims', 'Múhameðska villimenn',
'Múhameðstrúarmenn', 'Múhameðsfólk', 'múhameðstrúarmenn',
'Múhameðs fylgismenn', 'Múhameðstrúar', 'Múhameðskra', 'múhameðskir',
'Múhameðskur', 'múhameðstrú', 'Múhameðska villimennzkunni', 'Múhameðski menning',
'Múhameðstrúarmanna', 'Múhameðsmenn', 'Múhameðskir flóttamenn', 'Múhameð',
'Múhameðskra villimanna', 'Múhameðsku', 'múhameðskur lýður', 'Múhameðstrúar fólk',
'Múhameðska', 'Múhameðskir', 'Múhameðska skrælingja', 'Múhameðsku landanna',
'Múhameðstrú', 'múhameðstrúarmönnum', 'Múhameðsku Arabs',
'Múhameðskum villimanna', 'Múhameðstrúar hyskið', 'Múhammeds fylgjendur',
'Múhameðstrúar einstaklingar', 'Múhameðska fólk', 'Múhameðskur fólks', 'Muslima', 
'Múhameðska skrælingja', 'Múhameðskra', 'Múhameðstrú', 'Sunnis', 'Sunnítar', 'Shia', 'Súnní Arabar', 
'Súnní og Shíamuslima', 'Wahhabis', 'muslim', 'muslims', 'múslíar', 'people of Middle Eastern religions', 
'proponents of Islam' ]

In [ ]:
# 0) Imports + presence threshold
import pandas as pd
presence_threshold = 3

# 1) Define helpers (run these once, before using them)
def ctr(df: pd.DataFrame, tag: str) -> pd.DataFrame:
    out = (
        df.loc[:, ["comment_id", "generalized"]]
          .groupby("generalized", as_index=False)
          .size()
          .rename(columns={"size": "n"})
          .sort_values("n", ascending=False)
    )
    out["step"] = tag
    return out

def build_gen(groups: pd.DataFrame, groupn: pd.DataFrame, comments: pd.DataFrame) -> pd.DataFrame:
    
    # Ensure comment_id exists
    if "comment_id" not in comments.columns:
        if "id" in comments.columns:
            comments = comments.rename(columns={"id": "comment_id"})
        else:
            raise KeyError("`comments` must contain `comment_id` or `id`.")
    
    g = groups.copy()
    
    # Ensure groups has comment_id
    if "comment_id" not in g.columns:
        for cand in ["comment", "commentID", "comment_id_fk", "id_comment"]:
            if cand in g.columns:
                g = g.rename(columns={cand: "comment_id"})
                break
        if "comment_id" not in g.columns:
            raise KeyError("`groups` must have a comment id column.")
    
    # Ensure generalized label exists
    if "generalized" not in g.columns:
        key_in_groupn = next((c for c in ["id","generalized_id","group_id"] if c in groupn.columns), None)
        label_in_groupn = next((c for c in ["generalized","name","label"] if c in groupn.columns), None)
        key_in_groups  = next((c for c in ["generalized_id","generalized_group_id","group_id","gid","id_group"] if c in g.columns), None)
        if not (key_in_groupn and label_in_groupn and key_in_groups):
            raise KeyError("Could not infer join keys to fetch `generalized` label.")
        g = g.merge(
            groupn[[key_in_groupn, label_in_groupn]].rename(columns={label_in_groupn: "generalized"}),
            left_on=key_in_groups,
            right_on=key_in_groupn,
            how="left"
        )
    
    # -------- NEW: bring in ALL required columns from comments --------
    required_fields = [
        "comment_id",
        "group_generalization_presence",
        "dt",                     # REQUIRED for timetrend
        "hate_speech_presence",   # REQUIRED for time trend input
        "toxicity",               # REQUIRED for overlay
    ]
    
    optional_fields = [
        "language", "date", "source",
        "sexism_presence", "abuse_presence",
        "offensive_presence", "violence_presence",
        "hate_any",              # if already created
    ]
    
    cols_to_join = [c for c in required_fields + optional_fields if c in comments.columns]
    if "group_generalization_presence" not in cols_to_join:
        raise KeyError("comments must include `group_generalization_presence`.")

    g = g.merge(comments[cols_to_join], on="comment_id", how="left")
    
    # required structure
    needed = ["comment_id", "generalized", "group_generalization_presence"]
    miss = [c for c in needed if c not in g.columns]
    if miss:
        raise KeyError(f"`gen` missing required columns: {miss}")
    
    return g.dropna(subset=["generalized"]).reset_index(drop=True)


# 3) Build `gen`
gen = build_gen(groups, groupn, comments)

# Ensure comments has a real datetime column named dt
comments["dt"] = pd.to_datetime(comments["dt"], errors="coerce")

# Merge dt into gen (this is the missing step)
if "dt" not in gen.columns:
    gen = gen.merge(
        comments[["comment_id", "dt"]],
        on="comment_id",
        how="left"
    )

# 4) Now run your counting/threshold/join steps
s0 = ctr(gen, "all generalizations")

gen_thr = gen[gen["group_generalization_presence"] >= presence_threshold]
s1 = ctr(gen_thr, "presence ≥ 3")



# re-run the cell that builds `comments` with run_pipeline() first
comments_hate = comments[["comment_id", "hate_speech_presence"]].copy()
comments_hate["hate_any"] = comments_hate["hate_speech_presence"] > 0
comments_hate = comments_hate[["comment_id", "hate_any"]]


comments_hate = comments[["comment_id", "hate_speech_presence"]].copy()
comments_hate["hate_any"] = comments_hate["hate_speech_presence"] > 0
comments_hate = comments_hate[["comment_id", "hate_any"]]

g2 = gen_thr.merge(comments_hate, on="comment_id", how="inner")
s2 = ctr(g2, "after join to hate labels")

s3 = ctr(g2[g2["hate_any"].notna()], "after drop NA hate")

gen_for_plot = g2
s_final = ctr(gen_for_plot, "final plot data")

comparison = (
    pd.concat([s0, s1, s2, s3, s_final], ignore_index=True)
      .sort_values(["step", "n"], ascending=[True, False])
)

In [ ]:
# --- Sanity checks for required columns ---
for df_name, df, need in [
    ("groups", groups, {"comment_id", "group_id"}),
    ("groupn", groupn, {"id", "name"}),
    ("dat", dat, {"comment_id"}),
]:
    missing = need - set(df.columns)
    if missing:
        raise KeyError(f"{df_name} is missing columns: {sorted(missing)}")

def ids_for_names(names_list):
    """Return groupn IDs whose name is exactly in names_list (case-sensitive, like R’s %in%)."""
    if not names_list:
        return pd.Index([], dtype=groupn["id"].dtype)
    return groupn.loc[groupn["name"].isin(names_list), "id"]

def comment_ids_for_group_ids(gids):
    """Return unique comment_id values from groups where group_id ∈ gids."""
    if gids.empty:
        return pd.Index([], dtype=groups["comment_id"].dtype)
    return groups.loc[groups["group_id"].isin(gids), "comment_id"].dropna().unique()

bucket_specs_all = [
    ("Addiction disorders", addiction),
    ("Africa (nationalites)", africans),
    ("Asians (ethnicities)", asian_descent),
    ("Atheism", atheists),
    ("Bisexuality", bisexuality),
    ("Black/African (ethnicities)", black_african_descent),
    ("Christianity", christianity),
    ("East Asia (nationalities)", east_asians),
    ("Eastern Europe (nationalities)", eastern_europeans+polish),
    ("Foreigners, asylum seekers, refugees (general)", foreigners),
    ("Religious people (general terms)", general_religion),
    ("Homosexuality", homosexuality),
    ("Icelanders", icelanders),
    ("Indigenous peoples (ethnicities)", indigenous_peoples),
    ("Intersex/Gender minorities", intersex_gender_minorities),
    ("Judaism", judaism),
    ("Latino/Latin Americans (ethnicities)", latino_latin_american_descent),
    ("Men", men),
    ("Intellectual/Developmental disabilities", intellectual_disability),
    ("Mental illnesses/disorders", mental_disorders),
    ("Middle Eastern/Arab (ethnicities)", middle_eastern_arab_descent),
    ("Middle East (nationalities)",middle_easterners),
    ("Multiethnic/General (ethnicities)", multiethnic_general),
    ("Islam", muslim),
    ("Non-Abrahamic religions", non_abrahamic_religions),
    ("Nordic countries (nationalities)", nordics),
    ("North America (nationalities)", north_americans),
    ("Oceania (nationalities)", oceanians),
    ("Queer (general/other)", other_queer),
    ("Physical disabilities and general disability terms", physical_disability),
    ("South America (nationalities)",south_americans),
    ("South and Central Asia (nationalities)", south_central_asians),
    ("Southern Europe (nationalities)", southern_europeans),
    ("Transgender identities", trans),
    ("Western Europe (nationalities)", western_europeans),
    ("White/European (ethnicities)", white_european_descent), 
    ("Women", women)
]

bucket_specs = [
    ("Disability", addiction+intellectual_disability+mental_disorders+physical_disability),
    ("Nationality", africans+east_asians+eastern_europeans+polish+middle_easterners+nordics+icelanders+north_americans+oceanians+south_americans+south_central_asians+southern_europeans+western_europeans+foreigners),
    ("Ethnicity", asian_descent+black_african_descent+indigenous_peoples+latino_latin_american_descent+middle_eastern_arab_descent+multiethnic_general+white_european_descent),
    ("Religion", atheists+christianity+general_religion+judaism+muslim+non_abrahamic_religions),
    ("LGBTQIA+", bisexuality+homosexuality+intersex_gender_minorities+other_queer+trans),
    ("Gender", men+women),
]


In [ ]:
# ---- Build grouped buckets ----
frames = []
for label, vocab in bucket_specs:
    gids = ids_for_names(vocab)
    cids = comment_ids_for_group_ids(gids)
    if len(cids):
        frames.append(pd.DataFrame({"comment_id": cids, "generalized": label}))

gengroup = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["comment_id","generalized"])
gengroup["comment_id"] = pd.to_numeric(gengroup["comment_id"], errors="coerce")

# Use a NEW name so we don't collide with any existing gen from bucket_specs_all
gen_grouped = (
    dat.merge(gengroup, on="comment_id", how="inner")  # inner ensures only matched buckets
)

# ===== Denominator (use the grouped version!) =====
gen_base = gen_grouped.loc[
    gen_grouped["group_generalization_presence"] > 2,
    ["comment_id", "generalized"]
]

n_by_group = (
    gen_base.groupby("generalized", dropna=False)
            .size()
            .rename("n_total").reset_index()
)

# later, in plotting code, reference "generalized_group" instead of "generalized"
#cats = hate_plot["generalized"].astype(str).tolist()


In [ ]:
def build_gen_from_specs(bucket_specs, comments, presence_threshold=2):
    """
    Build a gen-like table for a given bucket_specs list.

    bucket_specs: list of (label, vocab_list) where vocab_list are groupn names
    comments:     comments DataFrame (must have comment_id, group_generalization_presence, dt, etc.)
    """
    frames = []
    for label, vocab in bucket_specs:
        gids = ids_for_names(vocab)
        cids = comment_ids_for_group_ids(gids)
        if len(cids):
            frames.append(pd.DataFrame({"comment_id": cids, "generalized": label}))

    gengroup = (
        pd.concat(frames, ignore_index=True)
        if frames else pd.DataFrame(columns=["comment_id", "generalized"])
    )
    gengroup["comment_id"] = pd.to_numeric(gengroup["comment_id"], errors="coerce")
    # FIX (dedup copy): one row per (comment, bucket)
    gengroup = gengroup.drop_duplicates(["comment_id", "generalized"])

    # join to comments so we carry dt, hate_speech_presence, toxicity, etc.
    gen = comments.merge(gengroup, on="comment_id", how="inner")

    # apply the presence threshold (comments is one row per comment, so gen is
    # one row per (comment, bucket))
    gen = gen[gen["group_generalization_presence"] > presence_threshold]

    return gen


In [ ]:
# -----------------------------
# 1) Grouped buckets (bucket_specs) → base plots
# -----------------------------
gen_grouped = build_gen_from_specs(bucket_specs, comments)

gen_base = gen_grouped.loc[:, ["comment_id", "generalized", "group_generalization_presence"]]


n_by_group_grouped = (
    gen_base.groupby("generalized", dropna=False)
            .size()
            .rename("n_total").reset_index()
)


In [ ]:
gen_allgroups = build_gen_from_specs(bucket_specs_all, comments)

gen_all_base = gen_allgroups.loc[:, ["comment_id", "generalized", "group_generalization_presence"]]

n_by_group_all = (
    gen_all_base.groupby("generalized", dropna=False)
                .size()
                .rename("n_total").reset_index()
)


In [ ]:
# =========================
# Group Generalization — HATE (levels 3+4, constant denominator) — GROUPED (BASE)
# =========================

def attach_n_labels(df, n_col="n_total", label_col="n_label"):
    df[label_col] = df[n_col].apply(lambda x: f"n={x:,.0f}")
    return df

if "gen_grouped" not in globals():
    raise NameError("gen_grouped is not defined. Build it from bucket_specs (grouped) first.")

gen_base = gen_grouped.loc[
    gen_grouped["group_generalization_presence"] > 2,
    ["comment_id", "generalized"]
]

n_by_group = (
    gen_base.groupby("generalized", dropna=False)
            .size().rename("n_total").reset_index()
)

hate_df = gen_grouped.loc[
    gen_grouped["group_generalization_presence"] > 2,
    ["comment_id", "generalized", "hate_speech_presence"]
].copy()

hate_num = pd.to_numeric(hate_df["hate_speech_presence"], errors="coerce").fillna(0)
hate_df["hate_level"] = hate_num.apply(lambda x: 4 if x == 4 else (3 if x == 3 else 0))

hate_counts = (
    hate_df.groupby(["generalized", "hate_level"]).size()
           .unstack("hate_level", fill_value=0)
)
for col in [0, 3, 4]:
    if col not in hate_counts.columns:
        hate_counts[col] = 0
hate_counts = hate_counts.rename(
    columns={0: "not_hate_count", 3: "hate3_count", 4: "hate4_count"}
).reset_index()

hate_plot = (
    n_by_group.merge(hate_counts, on="generalized", how="left")
              .fillna({"not_hate_count": 0, "hate3_count": 0, "hate4_count": 0})
)
hate_plot["prop_hate4"]     = hate_plot["hate4_count"] / hate_plot["n_total"]
hate_plot["prop_hate3"]     = hate_plot["hate3_count"] / hate_plot["n_total"]
hate_plot["prop_not_hate"]  = 1.0 - hate_plot["prop_hate4"] - hate_plot["prop_hate3"]
hate_plot["prop_hate_total"] = hate_plot["prop_hate4"] + hate_plot["prop_hate3"]

hate_plot = hate_plot.sort_values("prop_hate_total", ascending=False).reset_index(drop=True)
hate_plot = attach_n_labels(hate_plot)

expected  = set(n_by_group["generalized"].astype(str))
found     = set(hate_plot["generalized"].astype(str))
unexpected = found - expected
if unexpected:
    raise ValueError(f"Unexpected categories in hate_plot: {sorted(unexpected)}")

# =========================
# Plot
# =========================
from matplotlib.ticker import PercentFormatter
import matplotlib.patheffects as pe
import numpy as np
import matplotlib.pyplot as plt

cats = hate_plot["generalized"].astype(str).tolist()
y    = np.arange(len(cats))

COLOR_HATE4    = "#8B1A1A"   # dark red  — level 4
COLOR_HATE3    = "#D73027"   # lighter red — level 3
COLOR_NOT_HATE = "#4575B4"   # blue
COLOR_HATE_TEXT = "#8C1B13"

FS_BASE   = 16
FS_TICK   = 14
FS_CAT    = 20
FS_LEGEND = 18

fig, ax = plt.subplots(figsize=(14, max(7, 0.48 * len(cats))), dpi=300)

left = np.zeros(len(cats))
ax.barh(y, hate_plot["prop_hate4"].values, left=left, height=0.8,
        color=COLOR_HATE4, label="Hate speech (level 4)")
left += hate_plot["prop_hate4"].values

ax.barh(y, hate_plot["prop_hate3"].values, left=left, height=0.8,
        color=COLOR_HATE3, label="Hate speech (level 3)")
left += hate_plot["prop_hate3"].values

ax.barh(y, hate_plot["prop_not_hate"].values, left=left, height=0.8,
        color=COLOR_NOT_HATE, label="Not hate speech")

ax.set_yticks([])
ax.invert_yaxis()
ax.set_xticks([])
ax.tick_params(axis="both", labelsize=FS_TICK)
for s in ax.spines.values():
    s.set_visible(False)

ax.legend(
    loc="upper left", bbox_to_anchor=(0, 1.10), ncol=3,
    frameon=False, fontsize=FS_LEGEND, handlelength=1.8, columnspacing=1.6,
)

FS_SMALL     = FS_BASE - 4   # individual level % labels
FS_COMPOSITE = FS_BASE + 1   # combined % label

# individual level-4 %: right-aligned just before segment boundary
for yi, p4 in enumerate(hate_plot["prop_hate4"].values):
    if p4 < 0.06:
        continue
    ax.text(p4 - 0.01, yi + 0.32, f"{p4:.0%}", va="bottom", ha="right",
            fontsize=FS_SMALL, color="white", zorder=3)

# individual level-3 %: left-aligned just after segment boundary
for yi, (p4, p3) in enumerate(zip(hate_plot["prop_hate4"].values,
                                   hate_plot["prop_hate3"].values)):
    if p3 < 0.06:
        continue
    ax.text(p4 + 0.01, yi + 0.32, f"{p3:.0%}", va="bottom", ha="left",
            fontsize=FS_SMALL, color="white", zorder=3)

# composite %: always inside the colored section, right-aligned just before the CI bar
for yi, ptot in enumerate(hate_plot["prop_hate_total"].values):
    ax.text(ptot - 0.025, yi, f"{ptot:.0%}", va="center", ha="right",
            fontsize=FS_COMPOSITE, color="white", zorder=3)

# 95% CI: vertical I-beam (vertical stem spanning bar height, horizontal caps for CI range)
for yi, (ptot, n) in enumerate(zip(hate_plot["prop_hate_total"].values,
                                    hate_plot["n_total"].values)):
    ci = 1.96 * np.sqrt(ptot * (1 - ptot) / n)
    half_h = 0.38
    ax.plot([ptot, ptot], [yi - half_h, yi + half_h],
            color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi - half_h, yi - half_h],
            color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi + half_h, yi + half_h],
            color="black", lw=1.5, zorder=5)

# category names + n labels
pad_left, pad_right = 0.008, 0.008
for yi, name, nlab in zip(y, cats, hate_plot["n_label"].tolist()):
    ax.text(pad_left, yi - 0.32, name, va="top", ha="left", fontsize=FS_CAT, color="white")
    ax.text(1 - pad_right, yi, nlab, va="center", ha="right", fontsize=FS_BASE - 2, color="white")

ax.set_xlim(0, 1)
fig.tight_layout()

plt.savefig("group_generalization_hate_base.png", dpi=300, format="png", bbox_inches="tight")
print("group_generalization_hate_base.png created")
plt.close(fig)


In [ ]:
# =========================
# Group Generalization — HATE (levels 3 & 4, constant denominator) — ALL GROUPS
# =========================

from matplotlib.ticker import PercentFormatter
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def attach_n_labels(df, n_col="n_total", label_col="n_label"):
    df[label_col] = df[n_col].apply(lambda x: f"n={x:,.0f}")
    return df

if "gen_allgroups" not in globals():
    raise NameError("gen_allgroups is not defined. Build it from bucket_specs_all first.")

base = gen_allgroups.loc[
    gen_allgroups["group_generalization_presence"] > 2,
    ["comment_id", "generalized"]
]
n_by_group_all = (
    base.groupby("generalized", dropna=False)
        .size().rename("n_total").reset_index()
)

# ---- pull hate labels, split levels 3 / 4 ----
hate_df = gen_allgroups.loc[
    gen_allgroups["group_generalization_presence"] > 2,
    ["comment_id", "generalized", "hate_speech_presence"]
].copy()

hate_num = pd.to_numeric(hate_df["hate_speech_presence"], errors="coerce").fillna(0)
hate_df["hate_level"] = hate_num.apply(lambda x: 4 if x == 4 else (3 if x == 3 else 0))

hate_counts = (
    hate_df.groupby(["generalized", "hate_level"]).size()
           .unstack("hate_level", fill_value=0)
           .reset_index()
)
for col in [0, 3, 4]:
    if col not in hate_counts.columns:
        hate_counts[col] = 0
hate_counts = hate_counts.rename(columns={0: "not_hate_count", 3: "hate3_count", 4: "hate4_count"})

hate_plot = (
    n_by_group_all.merge(hate_counts, on="generalized", how="left")
                  .fillna({"not_hate_count": 0, "hate3_count": 0, "hate4_count": 0})
)
hate_plot["prop_hate4"]     = hate_plot["hate4_count"] / hate_plot["n_total"]
hate_plot["prop_hate3"]     = hate_plot["hate3_count"] / hate_plot["n_total"]
hate_plot["prop_hate_total"] = hate_plot["prop_hate4"] + hate_plot["prop_hate3"]
hate_plot["prop_not_hate"]  = 1.0 - hate_plot["prop_hate_total"]

hate_plot = hate_plot.sort_values("prop_hate_total", ascending=False).reset_index(drop=True)
hate_plot = attach_n_labels(hate_plot)

# =========================
# Plot
# =========================

COLOR_HATE4    = "#8B1A1A"
COLOR_HATE3    = "#D73027"
COLOR_NOT_HATE = "#4575B4"

FS_BASE = 14
FS_CAT  = 14
N_X     = 1.02   # x position for n-labels, outside bars
half_h  = 0.38
pad     = 0.01   # gap from segment boundary for % labels

cats  = hate_plot["generalized"].astype(str).tolist()
p4s   = hate_plot["prop_hate4"].values
p3s   = hate_plot["prop_hate3"].values
ptots = hate_plot["prop_hate_total"].values
ns    = hate_plot["n_total"].values

fig, ax = plt.subplots(figsize=(16, max(7, 0.48 * len(cats))), dpi=300)

for yi, (p4, p3, ptot, n, nlab) in enumerate(
        zip(p4s, p3s, ptots, ns, hate_plot["n_label"].tolist())):

    ax.barh(yi, p4,       left=0,    height=0.8, color=COLOR_HATE4)
    ax.barh(yi, p3,       left=p4,   height=0.8, color=COLOR_HATE3)
    ax.barh(yi, 1 - ptot, left=ptot, height=0.8, color=COLOR_NOT_HATE)

    # level-4 %: right-aligned at dark-red/light-red boundary
    if p4 >= 0.04:
        ax.text(max(p4 - pad, 0.01), yi, f"{p4:.0%}",
                va="center", ha="right", fontsize=FS_BASE - 3, color="white", zorder=3)

    # composite %: right-aligned at total-hate/not-hate boundary (inside light red)
    if p3 >= 0.03:
        ax.text(ptot - pad, yi, f"{ptot:.0%}",
                va="center", ha="right", fontsize=FS_BASE - 3, color="white", zorder=3)

    # n-label: outside bars to the right
    ax.text(N_X, yi, nlab, va="center", ha="left", fontsize=FS_BASE - 2, zorder=3)

    # vertical I-beam CI bar at composite boundary
    ci = 1.96 * np.sqrt(ptot * (1 - ptot) / n) if n > 0 else 0
    ax.plot([ptot, ptot], [yi - half_h, yi + half_h], color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi - half_h, yi - half_h], color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi + half_h, yi + half_h], color="black", lw=1.5, zorder=5)

# category labels outside bars on the left
ax.set_yticks(np.arange(len(cats)))
ax.set_yticklabels(cats, fontsize=FS_CAT)
ax.tick_params(axis="y", length=0, pad=10)
ax.invert_yaxis()

ax.set_xticks([])
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xlim(0, 1.45)

leg_handles = [
    Patch(color=COLOR_HATE4,    label="Hate speech (level 4)"),
    Patch(color=COLOR_HATE3,    label="Hate speech (level 3)"),
    Patch(color=COLOR_NOT_HATE, label="Not hate speech"),
]
ax.legend(handles=leg_handles, loc="lower center",
          bbox_to_anchor=(0.5, -0.03), frameon=False, fontsize=FS_BASE, ncol=3)

plt.savefig("group_generalization_hate_all.png", dpi=300, format="png", bbox_inches="tight")
print("group_generalization_hate_all.png created")
plt.close(fig)


In [ ]:
# =========================
# Group Generalization — TOXICITY (levels 3+4, constant denominator) — BASE (grouped)
# =========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

if "gen_grouped" not in globals():
    raise NameError("gen_grouped is not defined. Build it from bucket_specs (grouped) first.")

cat_col = "generalized"

base = gen_grouped.loc[
    gen_grouped["group_generalization_presence"] > 2,
    ["comment_id", cat_col]
]
n_by_group = (
    base.groupby(cat_col, dropna=False)
        .size().rename("n_total").reset_index()
)

toxic_df = gen_grouped.loc[
    gen_grouped["group_generalization_presence"] > 2,
    ["comment_id", cat_col, "toxicity"]
].copy()

toxic_num = pd.to_numeric(toxic_df["toxicity"], errors="coerce").fillna(0)
toxic_df["toxic_level"] = toxic_num.apply(lambda x: 4 if x == 4 else (3 if x == 3 else 0))

toxic_counts = (
    toxic_df.groupby([cat_col, "toxic_level"]).size()
            .unstack("toxic_level", fill_value=0)
)
for col in [0, 3, 4]:
    if col not in toxic_counts.columns:
        toxic_counts[col] = 0
toxic_counts = toxic_counts.rename(
    columns={0: "not_toxic_count", 3: "toxic3_count", 4: "toxic4_count"}
).reset_index()

toxic_plot = (
    n_by_group.merge(toxic_counts, on=cat_col, how="left")
              .fillna({"not_toxic_count": 0, "toxic3_count": 0, "toxic4_count": 0})
)
toxic_plot["prop_toxic4"]     = toxic_plot["toxic4_count"] / toxic_plot["n_total"]
toxic_plot["prop_toxic3"]     = toxic_plot["toxic3_count"] / toxic_plot["n_total"]
toxic_plot["prop_not_toxic"]  = 1.0 - toxic_plot["prop_toxic4"] - toxic_plot["prop_toxic3"]
toxic_plot["prop_toxic_total"] = toxic_plot["prop_toxic4"] + toxic_plot["prop_toxic3"]

toxic_plot = toxic_plot.sort_values("prop_toxic_total", ascending=False).reset_index(drop=True)
toxic_plot = attach_n_labels(toxic_plot)

expected  = set(n_by_group[cat_col].astype(str))
found     = set(toxic_plot[cat_col].astype(str))
unexpected = found - expected
if unexpected:
    raise ValueError(f"Unexpected categories in toxic_plot: {sorted(unexpected)}")

# =========================
# Plot
# =========================
COLOR_TOXIC4     = "#D95F02"   # dark orange — level 4
COLOR_TOXIC3     = "#FFA500"   # lighter orange — level 3
COLOR_NOT_TOXIC  = "#4575B4"   # blue
COLOR_TOXIC_TEXT = "#7F3B08"   # darker orange for boundary label

FS_BASE   = 16
FS_TICK   = 14
FS_CAT    = 20
FS_LEGEND = 18
BAR_H     = 0.8

cats = toxic_plot[cat_col].astype(str).tolist()
y    = np.arange(len(cats))

fig, ax = plt.subplots(figsize=(14, max(7, 0.48 * len(cats))), dpi=300)

left = np.zeros(len(cats))
ax.barh(y, toxic_plot["prop_toxic4"].values, left=left, height=BAR_H,
        color=COLOR_TOXIC4, label="Toxic (level 4)")
left += toxic_plot["prop_toxic4"].values

ax.barh(y, toxic_plot["prop_toxic3"].values, left=left, height=BAR_H,
        color=COLOR_TOXIC3, label="Toxic (level 3)")
left += toxic_plot["prop_toxic3"].values

ax.barh(y, toxic_plot["prop_not_toxic"].values, left=left, height=BAR_H,
        color=COLOR_NOT_TOXIC, label="Not toxic")

ax.set_yticks([])
ax.invert_yaxis()
ax.set_xticks([])
ax.tick_params(axis="both", labelsize=FS_TICK)
for s in ax.spines.values():
    s.set_visible(False)

ax.legend(
    loc="upper left", bbox_to_anchor=(0, 1.10), ncol=3,
    frameon=False, fontsize=FS_LEGEND, handlelength=1.8, columnspacing=1.6,
)

FS_SMALL     = FS_BASE - 4   # individual level % labels
FS_COMPOSITE = FS_BASE + 1   # combined % label

# individual level-4 %: right-aligned just before segment boundary
for yi, p4 in enumerate(toxic_plot["prop_toxic4"].values):
    if p4 < 0.06:
        continue
    ax.text(p4 - 0.01, yi + 0.32, f"{p4:.0%}", va="bottom", ha="right",
            fontsize=FS_SMALL, color="white", zorder=3)

# individual level-3 %: left-aligned just after segment boundary
for yi, (p4, p3) in enumerate(zip(toxic_plot["prop_toxic4"].values,
                                    toxic_plot["prop_toxic3"].values)):
    if p3 < 0.06:
        continue
    ax.text(p4 + 0.01, yi + 0.32, f"{p3:.0%}", va="bottom", ha="left",
            fontsize=FS_SMALL, color="white", zorder=3)

# composite %: always inside the colored section, right-aligned just before the CI bar
for yi, ptot in enumerate(toxic_plot["prop_toxic_total"].values):
    ax.text(ptot - 0.025, yi, f"{ptot:.0%}", va="center", ha="right",
            fontsize=FS_COMPOSITE, color="white", zorder=3)

# 95% CI: vertical I-beam (vertical stem spanning bar height, horizontal caps for CI range)
for yi, (ptot, n) in enumerate(zip(toxic_plot["prop_toxic_total"].values,
                                    toxic_plot["n_total"].values)):
    ci = 1.96 * np.sqrt(ptot * (1 - ptot) / n)
    half_h = 0.38
    ax.plot([ptot, ptot], [yi - half_h, yi + half_h],
            color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi - half_h, yi - half_h],
            color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi + half_h, yi + half_h],
            color="black", lw=1.5, zorder=5)

# category names + n labels
pad_left, pad_right = 0.008, 0.008
for yi, (name, ptot, nlab) in enumerate(zip(cats, toxic_plot["prop_toxic_total"].values,
                                              toxic_plot["n_label"].tolist())):
    ax.text(pad_left, yi - 0.32, name, va="top", ha="left", fontsize=FS_CAT, color="white")
    ax.text(1.0 - pad_right, yi, nlab, va="center", ha="right", fontsize=FS_BASE - 2, color="white", clip_on=True)

ax.set_xlim(0, 1.0)
fig.tight_layout()

save_name = "group_generalization_toxic_base.png"
plt.savefig(save_name, dpi=300, format="png", bbox_inches="tight")
print(f"{save_name} created")
plt.close(fig)


In [ ]:
# =========================
# Group Generalization — TOXICITY (levels 3 & 4, constant denominator) — ALL GROUPS
# =========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def attach_n_labels(df, n_col="n_total", label_col="n_label"):
    df[label_col] = df[n_col].apply(lambda x: f"n={x:,.0f}")
    return df

if "gen_allgroups" not in globals():
    raise NameError("gen_allgroups is not defined. Build it from bucket_specs_all first.")

cat_col = "generalized"

base = gen_allgroups.loc[
    gen_allgroups["group_generalization_presence"] > 2,
    ["comment_id", cat_col]
]
n_by_group_all = (
    base.groupby(cat_col, dropna=False)
        .size().rename("n_total").reset_index()
)

# ---- pull toxicity, split levels 3 / 4 ----
toxic_df = gen_allgroups.loc[
    gen_allgroups["group_generalization_presence"] > 2,
    ["comment_id", cat_col, "toxicity"]
].copy()

toxic_num = pd.to_numeric(toxic_df["toxicity"], errors="coerce").fillna(0)
toxic_df["toxic_level"] = toxic_num.apply(lambda x: 4 if x == 4 else (3 if x == 3 else 0))

toxic_counts = (
    toxic_df.groupby([cat_col, "toxic_level"]).size()
            .unstack("toxic_level", fill_value=0)
            .reset_index()
)
for col in [0, 3, 4]:
    if col not in toxic_counts.columns:
        toxic_counts[col] = 0
toxic_counts = toxic_counts.rename(columns={0: "not_toxic_count", 3: "toxic3_count", 4: "toxic4_count"})

toxic_plot = (
    n_by_group_all.merge(toxic_counts, on=cat_col, how="left")
                  .fillna({"not_toxic_count": 0, "toxic3_count": 0, "toxic4_count": 0})
)
toxic_plot["prop_toxic4"]     = toxic_plot["toxic4_count"] / toxic_plot["n_total"]
toxic_plot["prop_toxic3"]     = toxic_plot["toxic3_count"] / toxic_plot["n_total"]
toxic_plot["prop_toxic_total"] = toxic_plot["prop_toxic4"] + toxic_plot["prop_toxic3"]
toxic_plot["prop_not_toxic"]  = 1.0 - toxic_plot["prop_toxic_total"]

toxic_plot = toxic_plot.sort_values("prop_toxic_total", ascending=False).reset_index(drop=True)
toxic_plot = attach_n_labels(toxic_plot)

# =========================
# Plot
# =========================

COLOR_TOXIC4    = "#D95F02"
COLOR_TOXIC3    = "#FFA500"
COLOR_NOT_TOXIC = "#4575B4"

FS_BASE = 14
FS_CAT  = 14
N_X     = 1.02
half_h  = 0.38
pad     = 0.01

cats  = toxic_plot[cat_col].astype(str).tolist()
p4s   = toxic_plot["prop_toxic4"].values
p3s   = toxic_plot["prop_toxic3"].values
ptots = toxic_plot["prop_toxic_total"].values
ns    = toxic_plot["n_total"].values

fig, ax = plt.subplots(figsize=(16, max(7, 0.48 * len(cats))), dpi=300)

for yi, (p4, p3, ptot, n, nlab) in enumerate(
        zip(p4s, p3s, ptots, ns, toxic_plot["n_label"].tolist())):

    ax.barh(yi, p4,       left=0,    height=0.8, color=COLOR_TOXIC4)
    ax.barh(yi, p3,       left=p4,   height=0.8, color=COLOR_TOXIC3)
    ax.barh(yi, 1 - ptot, left=ptot, height=0.8, color=COLOR_NOT_TOXIC)

    # level-4 %: right-aligned at dark-orange/light-orange boundary
    if p4 >= 0.04:
        ax.text(max(p4 - pad, 0.01), yi, f"{p4:.0%}",
                va="center", ha="right", fontsize=FS_BASE - 3, color="white", zorder=3)

    # composite %: right-aligned at total-toxic/not-toxic boundary
    if p3 >= 0.03:
        ax.text(ptot - pad, yi, f"{ptot:.0%}",
                va="center", ha="right", fontsize=FS_BASE - 3, color="white", zorder=3)

    # n-label: outside bars to the right
    ax.text(N_X, yi, nlab, va="center", ha="left", fontsize=FS_BASE - 2, zorder=3)

    # vertical I-beam CI bar at composite boundary
    ci = 1.96 * np.sqrt(ptot * (1 - ptot) / n) if n > 0 else 0
    ax.plot([ptot, ptot], [yi - half_h, yi + half_h], color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi - half_h, yi - half_h], color="black", lw=1.5, zorder=5)
    ax.plot([ptot - ci, ptot + ci], [yi + half_h, yi + half_h], color="black", lw=1.5, zorder=5)

ax.set_yticks(np.arange(len(cats)))
ax.set_yticklabels(cats, fontsize=FS_CAT)
ax.tick_params(axis="y", length=0, pad=10)
ax.invert_yaxis()

ax.set_xticks([])
for s in ax.spines.values():
    s.set_visible(False)
ax.set_xlim(0, 1.45)

leg_handles = [
    Patch(color=COLOR_TOXIC4,    label="Toxicity (level 4)"),
    Patch(color=COLOR_TOXIC3,    label="Toxicity (level 3)"),
    Patch(color=COLOR_NOT_TOXIC, label="Not toxic"),
]
ax.legend(handles=leg_handles, loc="lower center",
          bbox_to_anchor=(0.5, -0.03), frameon=False, fontsize=FS_BASE, ncol=3)

save_name = "group_generalization_toxic_all.png"
plt.savefig(save_name, dpi=300, format="png", bbox_inches="tight")
print(f"{save_name} created")
plt.close(fig)


In [ ]:
cutoff = pd.Timestamp("2008-01-01")

# Ensure dt is datetime
comments["dt"] = pd.to_datetime(comments["dt"], errors="coerce")
gen_allgroups["dt"] = pd.to_datetime(gen_allgroups["dt"], errors="coerce")

# Monthly totals of all comments
ncomms = (
    comments.loc[comments["dt"] > cutoff, ["dt"]]
            .assign(month=lambda d: d["dt"].dt.to_period("M").dt.to_timestamp())
            .groupby("month", as_index=False)
            .size()
            .rename(columns={"size": "N"})
)

# Monthly counts per generalized group (from gen_allgroups)
timetrend = (
    gen_allgroups.loc[
        (gen_allgroups["dt"] > cutoff) &
        (gen_allgroups["group_generalization_presence"] > 2),
        ["dt", "generalized", "hate_speech_presence"]
    ]
    .assign(month=lambda d: d["dt"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "generalized"], as_index=False)
    .size()
    .rename(columns={"size": "i"})
    .merge(ncomms, on="month", how="left")
    .assign(hlutfall=lambda d: d["i"] / d["N"])
)
# timetrend columns: month, generalized, i (count in that month/group), N (all comments that month), hlutfall (i/N)


In [ ]:
# =========================
# Time trend — ALL GROUPS UNIVERSE
# Uses: gen_allgroups (built from bucket_specs_all)
# =========================

import pandas as pd

if "gen_allgroups" not in globals():
    raise NameError("gen_allgroups is not defined. Build it from bucket_specs_all first.")

cutoff = pd.Timestamp("2008-01-01")

# Ensure dt is datetime
comments["dt"] = pd.to_datetime(comments["dt"], errors="coerce")
gen_allgroups["dt"] = pd.to_datetime(gen_allgroups["dt"], errors="coerce")

# Monthly totals of all comments (denominator N)
ncomms = (
    comments.loc[comments["dt"] > cutoff, ["dt"]]
            .assign(month=lambda d: d["dt"].dt.to_period("M").dt.to_timestamp())
            .groupby("month", as_index=False)
            .size()
            .rename(columns={"size": "N"})
)

# Monthly counts per generalized group (from ALL-universe gen_allgroups)
timetrend = (
    gen_allgroups.loc[
        (gen_allgroups["dt"] > cutoff) &
        (gen_allgroups["group_generalization_presence"] > 2),
        ["dt", "generalized", "hate_speech_presence"]
    ]
    .assign(month=lambda d: d["dt"].dt.to_period("M").dt.to_timestamp())
    .groupby(["month", "generalized"], as_index=False)
    .size()
    .rename(columns={"size": "i"})
    .merge(ncomms, on="month", how="left")
    .assign(hlutfall=lambda d: d["i"] / d["N"])
)


# timetrend columns:
#   month:       month timestamp
#   generalized: group label (all-universe)
#   i:           count of comments in that month/group
#   N:           all comments that month
#   hlutfall:    i / N


In [ ]:
# =========================
# Proportion time trends + compact numbered side event panels
# (no horizontal leaders; bold/colored vlines; N events per column)
# + FAINT TOXICITY + HATE SPEECH TIMELINE OVERLAYS
# Saves: proportion_of_comments_per_generalized.png
# =========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.dates import YearLocator, MonthLocator, DateFormatter
from matplotlib.ticker import PercentFormatter
from matplotlib.colors import to_rgba
import textwrap, math, re
from pathlib import Path

# ---- Matplotlib look ----
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.facecolor":     "white",
    "figure.facecolor":   "white",
    "axes.grid":          False,
    "grid.color":         "#e0e0e0",
    "grid.linewidth":     0.6,
    "grid.alpha":         1.0,       # don't use alpha in EPS — bake it in
    "axes.axisbelow":     True,
})

# ---------- EVENTS ----------
EVENTS_BY_GROUP = {
    "Women": [
        {"date": "2008-08-11", "label": "Reykjavík chief of police withdraws his earlier support for closing a local strip club. Debate includes hostile remarks about feminists.", "vline": True},
        {"date": "2009-11-24", "label": "Minister of Social Affairs, Árni Páll Árnason, wants gender quotas in corporate boards. Debate vilifies feminists and blames them for social ills.", "vline": True},
        {"date": "2009-03-11", "label": "Parliamentary bill on criminalizing the purchase of sex work in Iceland. Debate on whether to penalize sex workers or purchasers.", "vline": True},
        {"date": "2024-04-23", "label": "Criticism of female presidential candidates (particularly then-Prime Minister, Katrín Jakobsdóttir), links feminism to social decay.", "vline": True},        
        {"date": "2017-12-09", "label": "#MeToo movement reaches Iceland. Debate on sexual harassment, gendered power imbalances, and the political and social fallout.", "vline": True},
        {"date": "2018-10-12", "label": "University of Reykjavík fires lecturer Kristinn Sigurjónsson over sexist Facebook posts. Leaking of women's responses sparks anti-feminist backlash.", "vline": True},
        {"date": "2023-03-15", "label": "Katrín Jakobsdóttir, Prime Minister, and Þórdís KR Gylfadóttir, Minister of Foreign Affairs, travel to Ukraine amid the ongoing war. Criticism of female politicians and feminism causing national decline.", "vline": True},
        {"date": "2010-05-19", "label": "Mary Glasspool becomes first openly lesbian bishop in the Anglican Communion. Debate on women's authority in Christian churches.", "vline": True},
        {"date": "2021-09-04", "label": "Several members of the Icelandic national football team accused of sexual assault. Debate on the legitimacy of these claims, and accusations of feminists exaggerating victimhood.", "vline": True},
        {"date": "2011-08-08", "label": "Pop singer and queer activist Páll Óskar claims 'the white Christian man [...] with a Bible in one hand a gun in the other' is the only group not to face discrimination. Debate on gendered prejudice.", "vline": True},   
    ],
    "Men": [
        {"date": "2008-07-18", "label": "Swedish authorities announce new legislation on sex work, criminalizing purchasers and not sex workers. Polarized debate on men who buy sexual services.", "vline": True},
        {"date": "2011-08-08", "label": "Páll Óskar’s remarks on the white Christian man. Debate includes sarcastic remarks on the alleged dangers of men, and warnings against using recent terrorism in Norway to justify anti-male sentiment.", "vline": True},
        {"date": "2018-10-12", "label": "University of Reykjavík fires Kristinn Sigurjónsson. Discussion centers on men being broadly presumed guilty and the consequences for their careers.", "vline": True},
        {"date": "2019-10-31", "label": "The Social Democratic Alliance implements gender quotas for electoral candidate lists. Debate on whether quotas hurt gender equality by limiting men's candidacy opportunities.", "vline": True},
        {"date": "2008-02-19", "label": "A news report about Hugh Hefner and his three girlfriends appearing together on the cover of Playboy magazine. Debate on gendered sexual double standards.", "vline": True},
        {"date": "2020-06-08", "label": "Comedian Pétur Jóhann Sigfússon recorded making a racist and sexist joke. Debate on identity politics and perceived targeting of (white) men by media and activist movements.", "vline": True},
        {"date": "2017-12-08", "label": "Women in politics share stories of sexual harassment and abuse in an online group. Debate on the credibility of these accounts with sarcastic generalizations portraying men as a threat to women.", "vline": True},
        {"date": "2009-10-08", "label": "A blog post notes that women author very few of the most popular blogs. Rebuttals against feminism and gender representation.", "vline": True},
    ],

    "LGBTQIA+": [
        {"date": "2012-02-15", "label": "Primary-school teacher Snorri Óskarsson blogs that homosexuality is a sin. Debate on hate speech, teachers’ public religious views, and whether schools should fire staff to protect pupils.", "vline": True},
        {"date": "2023-04-05", "label": "A mass shooting at a Nashville elementary school reportedly carried out by a trans man. Debate treats the shooter's identity as central, framing trans people as a violent threat.", "vline": True},
        {"date": "2010-09-12", "label": "Faroese MP and religious leader Jenis av Rana publicly refused to sit at a dinner with Jóhanna Sigurðardóttir, openly lesbian Prime Minister of Iceland. Discussion includes condemnation as well as homophobia.", "vline": True},
        {"date": "2009-02-12", "label": "Jóhanna Sigurðardóttir becomes the first openly queer prime minister in the world. Discussion includes slurs and hostile generalizations, as well as supportive posts.", "vline": True},
        {"date": "2009-12-06", "label": "Pop singer Friðrik Ómar reports that he isn't allowed to perform at the Pentecostal church because of his homosexuality. Polarized debate, e.g., questions on why he would want to perform in a 'church that hates [gay men]'.", "vline": True},
        {"date": "2008-07-11", "label": "Survey reveals that 77% of Christian priests in Iceland are willing to officiate same-sex weddings. Polarized discussion centers on whether homosexuality is sinful or morally acceptable.", "vline": True},
        {"date": "2024-04-02", "label": "Announcement of Baldur Þórhallsson's presidential candidacy with his husband Felix Bergsson prominently featured. Homophobic discussion with severely transphobic tendencies (Felix Bergsson is an advocate for trans rights).", "vline": True},
        {"date": "2012-07-14", "label": "Snorri Óskarsson fired from his teaching post over his homophobic blog posts. Polarized discussion centers around freedom of speech vs. hate speech.", "vline": True},
        {"date": "2013-08-09", "label": "Controversy over a planned visit by U.S. evangelist Franklin Graham (who opposes same-sex marriage). Discussion centers around freedom of speech vs. hate speech.", "vline": True},
        {"date": "2019-07-08", "label": "Iceland passes law on gender autonomy and intersex rights. Severely transphobic discussion follows, equating trans people with paedophiles, mentally ill people, or 'men who invade women's places'.", "vline": True},
        ],

    "Foreigners, asylum seekers, refugees (general)": [
        {"date": "2008-05-15", "label": "Akranes town council collapses over disagreement about accepting 30 refugees from Iraq. Polarized debate includes both refugee support and overt racism.", "vline": True},
        {"date": "2012-05-11", "label": "Teenagers from Algeria convicted for using forged travel documents. Debate centers on what qualifies as racism, and alleged 'importation of criminals'.", "vline": True},
        {"date": "2020-09-08", "label": "Deportation of an Egyptian refugee family from Iceland sparks protests. Polarized discussion centers on claims that Middle Eastern refugees don't assimilate to Icelandic culture.", "vline": True},
        {"date": "2014-06-20", "label": "MP Sveinbjörg Birna Sveinbjörnsdóttir calls for a vote on land allocation for a mosque in Reykjavík. Polarized debate on freedom of religion vs. supposed dangers of (Muslim) immigrants.", "vline": True},
        {"date": "2011-05-11", "label": "News report on newly arrived asylum seekers accused of shoplifting. Hostile discussion about welfare abuse and cultural incompatibility.", "vline": True},
        {"date": "2016-09-18", "label": "Anti-immigrant incidents in Germany and political fallout. Debate on Iceland's capacity to accept refugees, features racism towards Middle Eastern people.", "vline": True},
        {"date": "2016-01-21", "label": "Mass sexual assaults and robberies in Cologne on New Year's Eve, with perpetrators reported to include asylum-seekers of Arab and North African origin. Racist debate follows.", "vline": True},
        {"date": "2017-09-27", "label": "UNICEF encourages Icelandic parliament to change recently passed immigration laws to protect refugee children. Racist comments equate refugee children to the luggage of 'criminals who invade the country'.", "vline": True},
        {"date": "2019-03-19", "label": "Protests by refugees calling for no more deportations and fair processing of applications lead to confrontations with the police. Discussion depicts migrants as invaders, criminals, or culturally incompatible.", "vline": True},
        {"date": "2018-06-25", "label": "Political crisis in Germany over asylum policy, dispute between Angela Merkel and Horst Seehofer on refusing entry to assylum seekers. Hostile discussion on Angela Merkel's policy, with misogynistic undertones.", "vline": True},
    ],
    "Middle East (nationalities)": [
        {"date": "2014-07-19", "label": "UN Secretary-General Ban Ki-moon calls for ceasefire on Gaza. Polarized discussion centers on the nature of war vs. genocide.", "vline": True},
        {"date": "2009-01-07", "label": "Hamas attack on Israel. Polarized discussion on the war and its consequences on human rights.", "vline": True},
        {"date": "2017-12-08", "label": "Donald Trump announces recognition of Jerusalem as the capital of Israel. Discussion involves fear of renewed violence.", "vline": True},
        {"date": "2015-09-18", "label": "Reykjavík city council agrees to boycott Israeli products during the Israeli occupation of Palestinian territories. Debate pits accusations of antisemtitism against criticism of IDF human rights violations.", "vline": True},
        {"date": "2010-06-02", "label": "Israeli military raid on the Marvi Marmara aid flotilla, during which multiple people were killed or wounded. Debate on who is to blame, each side calling the other supporters of terrorism.", "vline": True},
        {"date": "2023-12-19", "label": "Protests and a petition to withdraw Iceland from Eurovision if Israel competes. Debate on the nature of war vs. genocide, includes both overt antisemitism and Islamophobia.", "vline": True},
        {"date": "2024-05-06", "label": "A concert held in protest of Iceland's participation in Eurovision. Highly polarized discussion includes Islamophobia and accusations of antisemitism amid ongoing debate over whther Israeli actions constitute genocide.", "vline": True},
        {"date": "2018-05-18", "label": "Protests at the Gaza border coincide with the relocation of the US embassy to Jerusalem, at least 60 Palestinians killed by IDF and thousands injured. Polarized debate on who is responsible, Hamas or Israel.", "vline": True},
        {"date": "2012-11-18", "label": "Escalation of violence between Israel and Hamas. Polarized debate includes Islamophobia and accusations of antisemitism along with criticism of Israeli military actions.", "vline": True},
        {"date": "2008-01-20", "label": "Reports on power cuts in Gaza causing a humanitarian emergency. Debate centers on who is to blame, Hamas or Israel.", "vline": True},
    ], 
    "Eastern Europe (nationalities)": [
        {"date": "2022-04-10", "label": "Photos reveal mass graves and bodies on the streets of Bucha, Ukraine. Debate on Russia's stated justification of fighting neo-Nazis in Ukraine.", "vline": True},
        {"date": "2014-08-29", "label": "Reports of Russian-supplied military equipment and trained personnel reaching separatists in Ukraine. Debate includes claims of Ukraine being run by neo-Nazis and support for Russian intervention.", "vline": True},
        {"date": "2008-02-28", "label": "Kosovo declares independence. Debate on whether Iceland should recognize it, xenophobic generalizations about Albanians and Muslims.", "vline": True},
        {"date": "2015-08-30", "label": "Intensified fighting along the eastern Ukraine ceasefire line. Polarized debate on NATO's response, and Iceland's responsibility to take in refugees.", "vline": True},
        {"date": "2022-10-10", "label": "Sabotage of the Nord Stream gas pipelines. Debate on who is to blame, generalizations about Russians and the Western countries.", "vline": True},
        {"date": "2018-03-13", "label": "The nerve agent attack on Sergei and Yulia Skripal in Salisbury leading to British accusations against Russia and diplomatic expulsions. Debate over motives and consequences.", "vline": True},
        {"date": "2023-06-24", "label": "The Nova Kakhovka dam collapse and subsequent decision to close the Icelandic embassy in Moscow. Hostile generalizations on Russians and NATO countries.", "vline": True},
        {"date": "2017-07-25", "label": "Report about pro-Russian separatists in Donetsk and Luhansk, and claims that they are selling factories to Russia. Accusations of neo-Nazi influences in Ukraine.", "vline": True},
        {"date": "2024-03-09", "label": "Discussion on the Russia-Ukraine war triggered by the two-year anniversary of the Russian invasion. Debate is mostly descriptive of events rather than group-specific.", "vline": True},
        {"date": "2015-02-09", "label": "Russian ongoing takeover of the Crimean Peninsula. Discussion shows concern over escalation of the conflict.", "vline": True},
    ], 
    "Icelanders": [
        {"date": "2011-05-07", "label": "Recurring arguments on whether Iceland should join the EU. Debate centers on sovereignty and national identity.", "vline": True},
        {"date": "2008-10-26", "label": "The collapse of Iceland's banks and UK's invocation of anti-terrorism legislation against Icelandic assets. Debate calls for accountability.", "vline": True},
        {"date": "2010-06-24", "label": "Currency-swap agreement signed between the Central Bank of Iceland and the People’s Bank of China. Debate includes claims about Chinese involvement in Iceland, framed as a threat to sovereignty.", "vline": True},
        {"date": "2009-07-25", "label": "Iceland formally applies for EU membership (application was eventually withdrawn). Hostile, nationalistic discourse, claims that joining the EU would destroy Iceland's sovereignty.", "vline": True},
        {"date": "2012-02-03", "label": "Croatia EU-accession referendum. Heightened debate on Iceland's EU membership application.", "vline": True},
        {"date": "2013-11-03", "label": "No event detected. Debate over Icelandic national identity and whether Icelanders display nationalist, exclusionary or a sense of cultural superiority towards immigrants.", "vline": True},
        {"date": "2012-07-07", "label": "EU Fisheries Commissioner María Damanaki visits Iceland for accession talks. Debate on EU influence over Icelandic fishing rights and sovereignty.", "vline": True},
        {"date": "2008-03-04", "label": "MP Ögmundur Jónasson calls for Iceland to immediately sever diplomatic relations with Israel. Debate includes claims that Icelanders are antisemitic.", "vline": True},
        {"date": "2020-02-24", "label": "The first COVID-19 case in Iceland confirmed. Debate centers on related travel advisories and concerns about the lack of quarantine measures for travellers.", "vline": True},
        {"date": "2017-09-08", "label": "News report on the Icelandic government doubling its funding for refugee reception for the following year. Debate featuring xenophobic rhetoric.", "vline": True},
    ],
    "Nationalities (other)": [
        {"date": "2011-05-20", "label": "Discussion of the eurozone sovereign debt crisis and institutional responses.", "vline": True},
        {"date": "2020-02-14", "label": "Financial Times story on the Karakax list, reporting on detention criteria and the scale of Uyghur detention in Xinjiang. Discussion criticizes China and other communist regimes.", "vline": True},
        {"date": "2014-08-30", "label": "Faroese ship Næraberg had engine trouble and was initially denied assistance in Iceland. Polarized debate over whether Iceland's initial refusal of assistance was justified.", "vline": True},
        {"date": "2018-08-22", "label": "Discussion of ongoing conflict between Turkey and the US, framed as economic war sparked by US trade measures by debaters, alongside accusations of Turkey practicing 'hostage politics'.", "vline": True},
        {"date": "2015-09-11", "label": "Angela Merkel's open-door policy on refugee admissions to Germany. Debate includes hostile generalizations about Muslims and refugees.", "vline": True},
        {"date": "2017-06-22", "label": "Ongoing refugee crisis and subsequent integration challenges in Nordic countries. Islamophobic refrain of refugees as a security risk and calls for stricter immigration policies.", "vline": True},
        {"date": "2012-02-05", "label": "The Greek parliament approves controversial austerity measures tied to a €130 billion EU bailout. Discussion criticizes the EU.", "vline": True},
        {"date": "2008-06-27", "label": "A Jordanian group advocates for boycotting Danish products due to the Danish cartoons of the Prophet Muhammad. Islamophobic discussion criticizes the boycott.", "vline": True},
        {"date": "2013-11-03", "label": "No event detected. Discussion on immigration, cultural integration, and perceived threats from 'Islamization'. Multiple generalizations about Germans, Swedes, and Norwegians.", "vline": True},
        {"date": "2015-02-15", "label": "Copenhagen terrorist attacks kill two and wound five at a cultural centre and synagogue. Debate on whether the attacker was a Muslim or a Dane (framing the identities as mutually exclusive).", "vline": True},
    ],

    "Ethnicities": [
        {"date": "2020-06-07", "label": "Ongoing protests following the murder of George Floyd. Polarized discussion on the Black Lives Matter movement, debating systematic racism vs. identity politics.", "vline": True},
        {"date": "2013-10-24", "label": "News reports on a white child found living with a Roma family in Greece, allegations of child abduction. Debate criticizes racist media and public reactions towards Roma people.", "vline": True},
        {"date": "2014-08-02", "label": "Escalation of hostilities between Israel and Hamas in Gaza. Debate includes anti-Arab hostility and accusations of antisemitism.", "vline": True},
        {"date": "2008-05-25", "label": "A proposal to resettle a group of Iraqi refugees in Akranes, and public opposition by MP Magnús Þór Hafsteinsson. Debate includes hostile generalizations about Arabs.", "vline": True},
        {"date": "2009-01-07", "label": "Israeli military assault on Gaza. Debate includes hostile generalizations about 'the Arab countries' alleged plan to destroy Israel.", "vline": True},
        {"date": "2016-12-28", "label": "Multiple reports of violent crimes allegedly committed by migrants in Germany. Debate includes hostile generalizations about migrants and Arabs.", "vline": True},
        {"date": "2018-07-22", "label": "Arguments about Nelson Mandela's legacy and South Africa's treatment of white people, alongside controversy over a local band performing in blackface. Discussion centers on the nature of racism.", "vline": True},
        {"date": "2015-09-11", "label": "A spike in refugee arrivals in Iceland. Many posts express hostile generalizations about Arabs and argue against accepting refugees.", "vline": True},
        {"date": "2011-08-08", "label": "The Norway terrorist attacks by Anders Behring Breivik. Discussion includes hostile generalizations about Arabs and white Christians.", "vline": True},
        {"date": "2012-07-28", "label": "Greek athlete Voula Papachristou barred from participating in the London Olympics due to racist comments on Twitter. Debate centers on free speech.", "vline": True},
    ],

    "Islam": [
        {"date": "2014-06-07", "label": "MP Sveinbjörg Birna Sveinbjörnsdóttir makes it a campaign issue that the city's land allocation for a mosque will be revoked. Xenophobic debate follows.", "vline": True},
        {"date": "2015-01-13", "label": "Charlie Hebdo shooting. Discussion centers on the alleged dangers of Islam, immigration, and opposition to mosque construction in Reykjavík.", "vline": True},
        {"date": "2015-09-27", "label": "Surge of refugees into Europe (and reactions to Angela Merkel's open‑door stance) and Reykjavík city‑council proposal of boycotting Israeli products. Islamophobic debate, accusations of antisemitism.", "vline": True},
        {"date": "2017-06-11", "label": "Reports of Islamist attacks in the UK (London Bridge attack, Manchester attack, and others). Debate frames Muslims as a security threat and culturally incompatible with Western society.", "vline": True},
        {"date": "2016-07-31", "label": "Reports of Islamist attacks in Nice, France, where a truck was driven into crowds celebrating Bastille Day. Hostile generalizations about Muslims, immigrants, and lack of cultural assimilation.", "vline": True},
        {"date": "2008-06-08", "label": "Akranes town council passes a motion to receive refugees to the town, leading to internal debates and the council's collapse. Hostile generalizations about Muslims.", "vline": True},
        {"date": "2013-07-21", "label": "A plot allocated to The Icelandic Muslim Association for mosque construction in Reykjavík. Heated debate and xenophobic criticism.", "vline": True},
        {"date": "2018-06-25", "label": "No event detected. A wave of hostile discourse about Muslims, portraying Islam as inherently violent and incompatible with Western society. Objections to headscarves and mosques.", "vline": True},
        {"date": "2012-09-21", "label": "Publication of Muhammad cartoons and release of the film Innocence of Muslims spark protests across Muslim countries, Western embassy closures, and violence, including the death of US ambassador in Libya. Anti-Muslim rhetoric.", "vline": True},
        {"date": "2020-09-23", "label": "Reports on an Egyptian family being deported from Iceland. Hostile generalizations about Muslim immigrants and asylum-seekers.", "vline": True},
    ],
    "Judaism": [
        {"date": "2014-07-19", "label": "Israeli airstrikes on Gaza and reciprocal rocket attacks by Hamas. Debate includes comparisons to the Holocaust, as well as Islamophobic rhetoric and accusations of antisemitism.", "vline": True},
        {"date": "2011-12-27", "label": "Sparked by Haaretz article about Jews in Iceland and the follow-up Icelandic media coverage, debate centers on antisemitism in Iceland and whether criticism of Israel constitutes anti‑Jewish hostility.", "vline": True},
        {"date": "2015-10-23", "label": "Debate on whether Israel is committing a genocide against Palestinians and whether framing Israeli actions as genocide constitutes antisemitism.", "vline": True},
        {"date": "2012-11-27", "label": "Outbreak of hostilities between Israel and Hamas. Polarized debate on alleged media bias and pro‑ or anti‑Jewish sentiment in Icelandic politics and society.", "vline": True},
        {"date": "2010-06-02", "label": "Israeli forces attack Gaza-bound aid flotilla, resulting in multiple deaths and widespread international outcry. Accusations of antisemitism in Icelandic politics and society.", "vline": True},
        {"date": "2023-12-16", "label": "Protest at a University of Iceland event, calling for political action and a trade boycott of Israel. Debate includes condemnation of the Israeli genocide and accusations of antisemitism.", "vline": True},
        {"date": "2017-12-16", "label": "Donald Trump recognizes Jerusalem as Israel's capital and announces plans to move US embassy there. Polarized debate includes clearly antisemitic and Islamophobic remarks.", "vline": True},
        {"date": "2008-12-28", "label": "A major Israeli military operation in Gaza with large numbers of Palestinian civilian casualties and international condemnation. Debate includes accusations of war crimes and media bias.", "vline": True},
        {"date": "2018-05-20", "label": "Mass protests at the Gaza–Israel border in which dozens of Palestinians were killed. Polarized discussion on who is to blame, IDF or Hamas.", "vline": True},
        {"date": "2008-03-04", "label": "MP Ögmundur Jónasson calls for immediate sever of diplomatic relations with Israel. Polarized debate includes both Islamophobia and antisemitism.", "vline": True},
    ], 
    "Christianity": [
        {"date": "2008-09-16", "label": "No event detected. Heated debate between Christian and atheist bloggers on creationism vs. Darwinian evolution, each accusing the other of ignorance.", "vline": True},
        {"date": "2014-06-24", "label": "Sveinbjörg Birna Sveinbjörnsdóttir's comments on land allocation for a mosque. Islamophobia and comparison between Christian scripture and Islamic practices.", "vline": True},
        {"date": "2012-02-24", "label": "Primary-school teacher Snorri Óskarsson's dismissal following his homophobic blogs. Debate on freedom of religion vs. freedom of speech.", "vline": True},
        {"date": "2017-06-25", "label": "No event detected. Hostile debate frames Muslims as inherently violent terrorists. Responses frame Christianity as equally violent.", "vline": True},
        {"date": "2009-05-19", "label": "Media coverage of the fossil Ida, nicknamed the 'missing link', and related claims about human evolution. Broad hostile generalizations about Christians.", "vline": True},
        {"date": "2015-10-19", "label": "No event detected. Criticism and hostile generalizations about the perceived hypocrisy of Christian institutions and believers, historical violence and misogyny in the church.", "vline": True},
        {"date": "2010-09-16", "label": "Faroese MP and Christian preacher Jenis av Rana refuses to sit with Icelandic PM Jóhanna Sigurðardóttir due to her homosexuality. Debate includes claims that this is not what Christianity is about.", "vline": True},
        {"date": "2015-01-04", "label": "Charlie Hebdo attack. Hostile generalizations about Muslims, claims that left-wing political parties in Iceland favor minorities and thereby undermine the national church.", "vline": True},
        {"date": "2011-07-24", "label": "The terrorist attacks in Norway by Anders Behring Breivik. Debate centers on whether Breivik should be considered Christian and whether such violence is linked to his Christianity.", "vline": True},
        {"date": "2008-01-01", "label": "Recent media coverage about Icelandic primary-school law removing references to 'Christian ethics'. Criticism of Christianity in historical and contemporary context.", "vline": True},
    ],      
    "Religion (other)" : [
        {"date": "2010-07-18", "label": "Heated debate between atheists and Christians on the platform, citing a recently published article by Dr. John R. Mabry in Philosophy Now. Centers on whether atheism is in itself a religion.", "vline": True},
        {"date": "2011-08-31", "label": "Debate on whether religion or atheism is more responsible for violence and moral failure. Participants cite recent news of violence (including Breivik) to argue for or against religion as the root cause of violence.", "vline": True},
        {"date": "2009-05-19", "label": "Dalai Lama visits Iceland. Debate contrasts defense of Christian faith and the claim that morality comes from God, with assertions that morality is independent of belief.", "vline": True},
        {"date": "2015-09-28", "label": "No event detected. A cluster of comments featuring generalizations about Muslims as well as religious people in general.", "vline": True},
        {"date": "2010-02-19", "label": "No event detected. Debate over religion vs. atheism, specifically whether the atheist group Vantrú constitutes a religious organization and whether morality can be grounded without god.", "vline": True},
        {"date": "2008-05-30", "label": "No event detected. A poll posted on a blog about which group is more hostile, Christians or members of Vantrú.", "vline": True},
        {"date": "2014-12-24", "label": "Primary school trips to religious institutions banned in Iceland. Cluster of comments debating perceived attacks on Christian culture and accusations that politicians seek to marginalize the national church.", "vline": True},
        {"date": "2013-09-04", "label": "No event detected. Debate over whether atheism is hostile to religion and whether widespread atheism leads to social harm.", "vline": True},
    ],
    "Religious people (general terms)": [
        {"date": "2010-08-05", "label": "No event detected. Cluster of comments about atheism vs. religion, how critical thinking relates to faith, and complaints about censorship on the platform.", "vline": True},
        {"date": "2008-09-16", "label": "No event detected. Debate over religion vs. science with generalizations about religious groups. Claims that belief is irrational and even dangerous.", "vline": True},
        {"date": "2009-05-18", "label": "Four MPs do not attend the mass traditionally held at the reopening of the Icelandic parliament after summer recess. Debate about church-state rituals.", "vline": True},
        {"date": "2012-02-15", "label": "Primary school teacher Snorri Óskarsson's blogs on homosexuality as a sin. Debate on whether the blogs constitute hate speech or freedom of religion.", "vline": True},
        {"date": "2012-07-31", "label": "The public dismissal of Snorri Óskarsson. Debate over public expressions of religious belief and their limits.", "vline": True},
        {"date": "2010-02-19", "label": "No event detected. Heated argument between religious commenters and atheists.", "vline": True},
        {"date": "2011-08-31", "label": "No event detected. Debate linking religion to violence and social harm, includes references to Breivik.", "vline": True},
        {"date": "2014-06-21", "label": "MP Sveinbjörg Birna Sveinbjörnsdóttir's comments on land allocation for a mosque. Debate mixes Islamophobia and broad generalizations on religious people.", "vline": True},
        {"date": "2008-01-01", "label": "A heated debate about priests' visits to preschools and primary schools and whether they constitute proselytizing.", "vline": True},
        {"date": "2018-01-04", "label": "Publication of the 2018 government budget proposal, showing 6.5 billion ISK allocated to religious institutions. Debate on the privileged status of the national church.", "vline": True}
    ], 
    
    "Disability": [
        {"date": "2009-09-09", "label": "A blog post mocks drunken men by portraying their postures as yoga poses. Hostile generalizations on substance use and mental illness, debate on whether alcoholism is a disease or a choice.", "vline": True},
        {"date": "2009-03-26", "label": "News on a large local cannabis bust sparked debate over legalization versus strict punishment, alongside hostile generalizations about drug users and an explicitly ableist remark.", "vline": True},
        {"date": "2008-07-04", "label": "Protests against a rehab facility being placed in a Reykjavík neighborhood. Debate over child safety vs. the need for treatment, broader discussion of addiction stigma and causes.", "vline": True},
        {"date": "2008-01-12", "label": "A photo of a televangelist parking in a disabled space sparked debate on disrespect toward disabled people and alleged exploitation by religious fundraisers. Discussions of 'welfare cheats' and media portrayals of addiction.", "vline": True}
    ],

}

def _events_for(group_name: str):
    if group_name in EVENTS_BY_GROUP:
        return EVENTS_BY_GROUP[group_name]
    low = {k.strip().lower(): v for k, v in EVENTS_BY_GROUP.items()}
    return low.get(group_name.strip().lower(), [])

def _events_with_peak_ranks(group_name: str, shown_peaks: pd.DataFrame):
    """
    Strict 1-to-1 mapping: each shown peak rank gets one row.
    Month shown in the table is always the detected peak month.
    Event text is attached by exact month match; optional nearest fallback is controlled by EVENT_MATCH_EXACT_ONLY.
    """
    if shown_peaks is None or shown_peaks.empty:
        return []

    evs_raw = _events_for(group_name)
    events = []
    for e in evs_raw:
        d = pd.to_datetime(e.get("date"), errors="coerce")
        if pd.isna(d):
            continue
        events.append({"date": d, "label": str(e.get("label", "")).strip()})

    rank_col = "display_rank" if "display_rank" in shown_peaks.columns else "abs_rank"
    p = shown_peaks[["month", rank_col]].copy()
    p["month"] = pd.to_datetime(p["month"], errors="coerce")
    p[rank_col] = pd.to_numeric(p[rank_col], errors="coerce")
    p = p.dropna(subset=["month", rank_col]).drop_duplicates(subset=[rank_col]).sort_values(rank_col)

    rows = []
    used_event_idx = set()

    for _, pr in p.iterrows():
        rank = int(pr[rank_col])
        peak_month = pr["month"]
        peak_month_key = peak_month.to_period("M")

        best_idx = None
        best_delta = None

        # Prefer exact peak-month match first.
        for i in range(len(events)):
            if i in used_event_idx:
                continue
            if pd.to_datetime(events[i]["date"]).to_period("M") == peak_month_key:
                best_idx = i
                best_delta = 0.0
                break

        # Optional fallback: nearest remaining event.
        if best_idx is None and not EVENT_MATCH_EXACT_ONLY:
            for i, ev in enumerate(events):
                if i in used_event_idx:
                    continue
                delta = abs((ev["date"] - peak_month).days)
                if best_delta is None or delta < best_delta:
                    best_delta = delta
                    best_idx = i

        if best_idx is None:
            rows.append({
                "peak_rank": rank,
                "peak_month": peak_month,
                "label": "No mapped event for this peak month",
                "delta_days": np.nan,
            })
        else:
            used_event_idx.add(best_idx)
            ev = events[best_idx]
            rows.append({
                "peak_rank": rank,
                "peak_month": peak_month,
                "label": ev["label"],
                "delta_days": float(best_delta) if best_delta is not None else np.nan,
            })
    return rows

DEN_COL = "N"   # monthly total-comments denominator for original hlutfall

# ---------- Sidebar + vline styling ----------
SIDEBAR = dict(
    width_rel = 0.82,
    pad_rel   = 0.035,
    inner_pad = 0.0035,
    base_fs   = 10.5,
    wrap_chars = 30,
    label_linespacing = 1.35,
    events_per_col = 4,
    max_cols = 18,
    col_gap_ax = 0.30,
    row_gap_ax = 0.10,
    vline_ls = (0, (4, 2)),
    vline_lw = 1.6,
    vline_alpha = 0.75,
    show_badges = True,
    badge_y_ax = 0.965,
    badge_text_fs = 8.5,
    badge_pad = 0.18,
    panel_badge_fs = 8.5,
    panel_badge_pad = 0.18,
    panel_badge_dx = 0.015,
)

PALETTE = ["#6B8FFF","#F39C6B","#6BC39B","#D27AC5","#C2C96A",
           "#5BB3C5","#ED807D","#8FA3AD","#B39CD0","#77C1A1"]
RAW_COLOR = "#2455A4"
RAW_LW = 2.1

# Faint toxicity overlay style (blue-ish)
TOX_COLOR = "#2CA89A"
TOX_LW    = 1.7
TOX_ALPHA  = 0.55

# Faint hate-speech overlay style (red)
HATE_COLOR = "#C00000"
HATE_LW    = 1.7
HATE_ALPHA = 0.55


# Peak methodology (k-neighbor local maxima)
PEAK_NEIGHBOR_K = 4
PEAK_MIN_COMMENTS = 5
PEAK_RANK_MODE = "hybrid"  # "hybrid", "count", or "percentage"
PEAK_RANK_COUNT_WEIGHT = 0.75  # only used in hybrid mode

PEAK_MARKER_EDGE = "#8B1A1A"
PEAK_MARKER_FACE = "white"
PEAK_MARKER_SIZE = 26
PEAK_RANK_FONTSIZE = 12
PEAK_RANK_DY = 6
PEAK_SHOW_TOP_K = 10  # set an integer to show only the top-k peaks; use None for all
PEAK_REQUIRE_WINDOW_EVIDENCE = True
PEAK_WINDOW_DAYS = 15
PEAK_WINDOW_MIN_COMMENTS = PEAK_MIN_COMMENTS
PEAK_RENUMBER_SHOWN = True
EVENT_MATCH_EXACT_ONLY = True  # set False to allow nearest-date fallback when no same-month event exists
EVENT_TABLE_MAX_ROWS = None
EVENT_TABLE_WRAP_CHARS = 85
EVENT_TABLE_FONT = 12
EVENT_TABLE_HEADER_FONT = 12
EVENT_TABLE_DESC_COLUMNS = 1
EVENT_TABLE_COLUMN_GAP = 0.06
EVENT_TABLE_SHOW_HEADERS = True

def detect_local_peaks(sub_df, k_neighbors=PEAK_NEIGHBOR_K, min_comments=PEAK_MIN_COMMENTS):
    """
    Detect peak candidates using a strict local-maximum rule over a k-neighbor window.

    A time point t is retained as a peak candidate if:
      - y[t] is >= each available neighbor within k points to the left,
      - y[t] is >= each available neighbor within k points to the right, and
      - group monthly comment count >= min_comments.

    Candidate ranking is configurable:
      - "count": rank by estimated group monthly comment count.
      - "percentage": rank by monthly share (hlutfall).
      - "hybrid": weighted blend of normalized count and normalized percentage.
    """
    if sub_df.empty:
        return pd.DataFrame(columns=["month", "hlutfall", "group_month_comments", "peak_score", "is_peak", "abs_rank"])

    ser = pd.to_numeric(sub_df["hlutfall"], errors="coerce").fillna(0.0).astype(float)
    k = int(max(1, k_neighbors))

    candidate_mask = pd.Series(True, index=ser.index, dtype=bool)
    for i in range(1, k + 1):
        left = ser.shift(i)
        right = ser.shift(-i)
        # Edge-aware: only enforce comparisons where that neighbor exists.
        candidate_mask &= ((ser >= left) | left.isna())
        candidate_mask &= ((ser >= right) | right.isna())

    # Use group-level monthly comment counts for thresholding.
    # If explicit counts are not provided, estimate as hlutfall * monthly denominator N.
    if "group_month_comments" in sub_df.columns:
        nvals = pd.to_numeric(sub_df["group_month_comments"], errors="coerce").fillna(0.0)
    elif DEN_COL in sub_df.columns:
        nvals = pd.to_numeric(sub_df[DEN_COL], errors="coerce").fillna(0.0) * ser
    else:
        nvals = pd.Series(0.0, index=sub_df.index)
    nvals = nvals.clip(lower=0.0)
    candidate_mask &= nvals >= float(min_comments)

    candidate_mask = candidate_mask.fillna(False)

    out = sub_df[["month"]].copy()
    out["hlutfall"] = ser.values
    out["group_month_comments"] = nvals.values
    out["is_peak"] = candidate_mask.values

    # Compute ranking score.
    mode = str(globals().get("PEAK_RANK_MODE", "hybrid")).lower().strip()
    w_count = float(globals().get("PEAK_RANK_COUNT_WEIGHT", 0.70))
    w_count = min(max(w_count, 0.0), 1.0)

    pct = ser.abs().astype(float)
    cnt = nvals.astype(float)

    pct_norm = pct / pct.max() if float(pct.max()) > 0 else pd.Series(0.0, index=pct.index)
    cnt_norm = cnt / cnt.max() if float(cnt.max()) > 0 else pd.Series(0.0, index=cnt.index)

    if mode == "count":
        score = cnt
    elif mode == "percentage":
        score = pct
    else:
        score = w_count * cnt_norm + (1.0 - w_count) * pct_norm

    out["peak_score"] = pd.to_numeric(score, errors="coerce")

    out["abs_rank"] = np.nan
    candidate_order = out.index[out["is_peak"]].tolist()
    candidate_order = sorted(candidate_order, key=lambda idx: float(out.loc[idx, "peak_score"]), reverse=True)
    for rank, idx in enumerate(candidate_order, start=1):
        out.loc[idx, "abs_rank"] = rank

    return out

def _blend_with_white(color, alpha):
    """Simulate transparency by blending color with white background."""
    import matplotlib.colors as mcolors
    r, g, b, _ = mcolors.to_rgba(color)
    return (1 - alpha*(1-r), 1 - alpha*(1-g), 1 - alpha*(1-b), 1.0)

def _wrap(text, width):
    parts = []
    for seg in str(text).split("\n"):
        parts.append(textwrap.fill(seg, width=width, break_long_words=True))
    return "\n".join(parts)

def add_event_table(ax, group_name: str, shown_peaks: pd.DataFrame):
    """Right-side table: strict 1-to-1 mapping; full descriptions in multiple columns."""
    rows = _events_with_peak_ranks(group_name, shown_peaks)
    rows = [r for r in rows if pd.notna(r.get("peak_rank"))]
    if not rows:
        return

    rows = rows[:EVENT_TABLE_MAX_ROWS]
    n_cols = max(1, int(EVENT_TABLE_DESC_COLUMNS))
    rows_per_col = int(np.ceil(len(rows) / n_cols))

    col_gap = float(EVENT_TABLE_COLUMN_GAP)
    usable_w = 0.98 - col_gap * (n_cols - 1)
    col_width = usable_w / n_cols

    for ci in range(n_cols):
        col_rows = rows[ci * rows_per_col:(ci + 1) * rows_per_col]
        if not col_rows:
            continue

        x0 = 0.02 + ci * (col_width + col_gap)
        x_peak = x0
        x_date = x0 + 0.12
        x_event = x0 + 0.3
        y = 0.99

        if EVENT_TABLE_SHOW_HEADERS:
            ax.text(x_peak, y, "Peak", transform=ax.transAxes,
                    ha="left", va="top", fontsize=EVENT_TABLE_HEADER_FONT, fontweight="bold", color="0.2")
            ax.text(x_date, y, "Month", transform=ax.transAxes,
                    ha="left", va="top", fontsize=EVENT_TABLE_HEADER_FONT, fontweight="bold", color="0.2")
            ax.text(x_event, y, "Event", transform=ax.transAxes,
                    ha="left", va="top", fontsize=EVENT_TABLE_HEADER_FONT, fontweight="bold", color="0.2")
            y -= 0.060

        wrap_chars = EVENT_TABLE_WRAP_CHARS if n_cols == 1 else max(20, int(EVENT_TABLE_WRAP_CHARS * 0.52))

        for r in col_rows:
            rank_txt = f"{int(r['peak_rank'])}"
            date_txt = pd.to_datetime(r["peak_month"]).strftime("%Y-%m")
            wrapped_event = textwrap.fill(
                str(r["label"]),
                width=wrap_chars,
                break_long_words=False,
            )

            ax.text(x_peak, y, rank_txt, transform=ax.transAxes,
                    ha="left", va="top", fontsize=EVENT_TABLE_FONT, color="0.1")
            ax.text(x_date, y, date_txt, transform=ax.transAxes,
                    ha="left", va="top", fontsize=EVENT_TABLE_FONT, color="0.1")
            ax.text(x_event, y, wrapped_event, transform=ax.transAxes,
                    ha="left", va="top", fontsize=EVENT_TABLE_FONT, color="0.1", linespacing=1.30)

            line_count = wrapped_event.count("\n") + 1
            # Slightly tighter row spacing than before.
            y -= (0.032 * line_count + 0.014)
            if y < 0.05:
                break


# ---------- Data prep ----------
BASE_GROUPS_KEEP = ["Women", "Men", "Foreigners, asylum seekers, refugees (general)", "Middle East (nationalities)", "Eastern Europe (nationalities)", "Icelanders", "Islam", "Judaism", "Christianity"]
COMBINED_GROUPS = {"LGBTQIA+": [
        "Transgender identities", "Homosexuality", "Bisexuality",
        "Intersex/Gender minorities", "Queer (general/other)"],    
    
    "Disability": 
         ["Intellectual/Developmental disabilities", 
         "Physical disabilities and general disability terms",
         "Mental illnesses/disorders", "Addiction disorders"],

     "Ethnicities": 
         ["Asians (ethnicities)", "Black/African (ethnicities)",
         "Indigenous peoples (ethnicities)", "Latino/Latin Americans (ethnicities)",
         "Middle Eastern/Arab (ethnicities)", "White/European (ethnicities)", 
         "Multiethnic/General (ethnicities)"],
                   
     "Nationalities (other)": 
         ["Africa (nationalities)", "East Asia (nationalities)", 
          "South and Central Asia (nationalities)", "North America (nationalities)", 
          "Oceania (nationalities)", "South America (nationalities)", 
          "Southern Europe (nationalities)", "Western Europe (nationalities)", 
          "Nordic countries (nationalities)"],
                   
     "Religion (other)" : 
         ["Non-Abrahamic religions", "Atheism", "Religious people (general terms)"]

    }


timetrend["month"] = pd.to_datetime(timetrend["month"], errors="coerce")
groups_keep = list(BASE_GROUPS_KEEP)
tt = timetrend.copy()

# combined groups (main generalization series)
if DEN_COL in tt.columns and COMBINED_GROUPS:
    monthly_den = (
        tt[["month", DEN_COL]]
        .dropna(subset=["month", DEN_COL])
        .drop_duplicates(subset=["month"])
        .set_index("month")
    )
    combined_rows = []
    for new_name, src_list in COMBINED_GROUPS.items():
        src = tt[tt["generalized"].isin(src_list)][["month","hlutfall"]].dropna()
        if src.empty:
            continue
        tmp = src.merge(monthly_den.reset_index(), on="month", how="left").dropna(subset=[DEN_COL])
        tmp["num"] = tmp["hlutfall"] * tmp[DEN_COL]
        agg = (tmp.groupby("month", as_index=False)["num"].sum()
                 .merge(monthly_den.reset_index(), on="month", how="left"))
        agg["hlutfall"] = (agg["num"]/agg[DEN_COL]).replace([np.inf,-np.inf], np.nan).fillna(0.0)
        agg["generalized"] = new_name
        combined_rows.append(agg[["month","generalized","hlutfall"]])
        if new_name not in groups_keep:
            groups_keep.append(new_name)
    combined_df = pd.concat(combined_rows, ignore_index=True) if combined_rows else \
                  pd.DataFrame(columns=["month","generalized","hlutfall"])
else:
    combined_df = pd.DataFrame(columns=["month","generalized","hlutfall"])

desired_order = [
    "Foreigners, asylum seekers, refugees (general)",
    "Islam",
    "LGBTQIA+",
    "Disability",
    "Women",
    "Judaism",
    "Christianity", 
    "Non-Abrahamic religions", 
    "Religious people (general terms)", 
    "Atheism",
    "Transgender identities", 
    "Homosexuality", 
    "Queer (general/other)",
    "Men",
    "Africa (nationalities)", "East Asia (nationalities)", 
    "South and Central Asia (nationalities)", "North America (nationalities)", 
    "Middle East (nationalities)", "Oceania (nationalities)", 
    "South America (nationalities)", 
    "Eastern Europe (nationalities)", "Southern Europe (nationalities)", 
    "Western Europe (nationalities)", "Nordic countries (nationalities)", 
    "Icelanders",
    "Intellectual/Developmental disabilities", 
    "Physical disabilities and general disability terms",
    "Mental illnesses/disorders", 
    "Addiction disorders",
    "Ethnicities",
    "Nationalities (other)",
    "Religion (other)",
    
]
groups_keep = [g for g in desired_order if g in groups_keep]

plot_df = pd.concat(
    [tt[tt["generalized"].isin(BASE_GROUPS_KEEP)][["month","generalized","hlutfall"]],
     combined_df],
    ignore_index=True
)

# drop bad months, but keep everything else for densification
plot_df = plot_df.dropna(subset=["month"])

# ---------- NEW: make plot_df dense over time for all groups ----------
# full monthly range over the whole timetrend period
all_months = pd.date_range(
    start=tt["month"].min(),
    end=tt["month"].max(),
    freq="MS",  # monthly start; swap to "W-MON" if you go weekly later
)

# full grid: every (month, group) combination
full_idx = pd.MultiIndex.from_product(
    [all_months, groups_keep],
    names=["month", "generalized"],
)

# reindex to full grid, missing hlutfall -> 0
plot_df = (
    plot_df
    .set_index(["month", "generalized"])
    .reindex(full_idx)
    .reset_index()
)

plot_df["hlutfall"] = plot_df["hlutfall"].fillna(0.0)
plot_df["generalized"] = pd.Categorical(plot_df["generalized"], categories=groups_keep, ordered=True)
plot_df = plot_df.sort_values(["generalized", "month"])


# ---------- Toxicity share over ALL comments (same denominator N) ----------
tox_plot_df = None
req_cols_tox = {"dt","generalized","toxicity","group_generalization_presence"}
if 'gen_allgroups' in globals() and req_cols_tox.issubset(set(gen_allgroups.columns)) and DEN_COL in tt.columns:
    monthly_den = (
        tt[["month", DEN_COL]]
        .dropna(subset=["month", DEN_COL])
        .drop_duplicates(subset=["month"])
    )

    g2 = gen_allgroups.copy()
    g2["dt"] = pd.to_datetime(g2["dt"], errors="coerce")
    g2 = g2.dropna(subset=["dt", "generalized"])
    g2 = g2[g2["group_generalization_presence"].isin([3,4])]

    tox_mask = pd.to_numeric(g2["toxicity"], errors="coerce") > 2
    tox = (g2.loc[tox_mask, ["dt","generalized"]]
             .assign(month=lambda d: d["dt"].dt.to_period("M").dt.to_timestamp())
             .groupby(["month","generalized"], as_index=False)
             .size()
             .rename(columns={"size":"tox_cnt"}))

    pairs = plot_df[["month", "generalized"]].drop_duplicates()
    tox_cnts = tox[["month","generalized","tox_cnt"]]
    tox_pairs = (pairs
        .merge(tox_cnts, on=["month","generalized"], how="left")
        .merge(monthly_den, on="month", how="left")
    )
    tox_pairs["tox_cnt"] = tox_pairs["tox_cnt"].fillna(0)
    tox_pairs["tox_share"] = (tox_pairs["tox_cnt"] / tox_pairs[DEN_COL]).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    tox_plot_df = tox_pairs[["month","generalized","tox_share"]].copy()

    if COMBINED_GROUPS:
        md = monthly_den.rename(columns={DEN_COL: "N_den"})
        tox_combined_rows = []
        for new_name, src_list in COMBINED_GROUPS.items():
            src_tox = tox[tox["generalized"].isin(src_list)].copy()
            if src_tox.empty:
                continue
            agg = (src_tox.groupby("month", as_index=False)["tox_cnt"].sum()
                       .merge(md, on="month", how="left")
                       .fillna({"tox_cnt":0}))
            agg["tox_share"] = (agg["tox_cnt"]/agg["N_den"]).replace([np.inf,-np.inf], np.nan).fillna(0.0)
            agg["generalized"] = new_name
            tox_combined_rows.append(agg[["month","generalized","tox_share"]])
        if tox_combined_rows:
            tox_combined_df = pd.concat(tox_combined_rows, ignore_index=True)
            tox_plot_df = pd.concat([tox_plot_df, tox_combined_df], ignore_index=True)

    tox_plot_df = tox_plot_df[tox_plot_df["generalized"].isin(groups_keep)].dropna(subset=["month"])
    tox_plot_df["generalized"] = pd.Categorical(tox_plot_df["generalized"], categories=groups_keep, ordered=True)
    tox_plot_df = tox_plot_df.sort_values(["generalized","month"])
else:
    print("[tox overlay] Skipped: missing toxicity columns or denominator.")

# ---------- Hate-speech share over ALL comments (same denominator N) ----------
hate_plot_df = None
req_cols_hate = {"dt","generalized","hate_speech_presence","group_generalization_presence"}
if 'gen_allgroups' in globals() and req_cols_hate.issubset(set(gen_allgroups.columns)) and DEN_COL in tt.columns:
    monthly_den_h = (
        tt[["month", DEN_COL]]
        .dropna(subset=["month", DEN_COL])
        .drop_duplicates(subset=["month"])
    )

    h2 = gen_allgroups.copy()
    h2["dt"] = pd.to_datetime(h2["dt"], errors="coerce")
    h2 = h2.dropna(subset=["dt", "generalized"])
    h2 = h2[h2["group_generalization_presence"].isin([3,4])]

    hate_num = pd.to_numeric(h2["hate_speech_presence"], errors="coerce").fillna(0)
    hate_mask = hate_num > 2
    hate = (h2.loc[hate_mask, ["dt","generalized"]]
              .assign(month=lambda d: d["dt"].dt.to_period("M").dt.to_timestamp())
              .groupby(["month","generalized"], as_index=False)
              .size()
              .rename(columns={"size":"hate_cnt"}))

    pairs_h = plot_df[["month", "generalized"]].drop_duplicates()
    hate_cnts = hate[["month","generalized","hate_cnt"]]
    hate_pairs = (pairs_h
        .merge(hate_cnts, on=["month","generalized"], how="left")
        .merge(monthly_den_h, on="month", how="left")
    )
    hate_pairs["hate_cnt"] = hate_pairs["hate_cnt"].fillna(0)
    hate_pairs["hate_share"] = (hate_pairs["hate_cnt"] / hate_pairs[DEN_COL]).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    hate_plot_df = hate_pairs[["month","generalized","hate_share"]].copy()

    if COMBINED_GROUPS:
        mdh = monthly_den_h.rename(columns={DEN_COL: "N_den"})
        hate_combined_rows = []
        for new_name, src_list in COMBINED_GROUPS.items():
            src_h = hate[hate["generalized"].isin(src_list)].copy()
            if src_h.empty:
                continue
            agg = (src_h.groupby("month", as_index=False)["hate_cnt"].sum()
                       .merge(mdh, on="month", how="left")
                       .fillna({"hate_cnt":0}))
            agg["hate_share"] = (agg["hate_cnt"]/agg["N_den"]).replace([np.inf,-np.inf], np.nan).fillna(0.0)
            agg["generalized"] = new_name
            hate_combined_rows.append(agg[["month","generalized","hate_share"]])
        if hate_combined_rows:
            hate_combined_df = pd.concat(hate_combined_rows, ignore_index=True)
            hate_plot_df = pd.concat([hate_plot_df, hate_combined_df], ignore_index=True)

    hate_plot_df = hate_plot_df[hate_plot_df["generalized"].isin(groups_keep)].dropna(subset=["month"])
    hate_plot_df["generalized"] = pd.Categorical(hate_plot_df["generalized"], categories=groups_keep, ordered=True)
    hate_plot_df = hate_plot_df.sort_values(["generalized","month"])
else:
    print("[hate overlay] Skipped: missing hate columns or denominator.")

def _build_window_evidence_context():
    """Precompute daily stats used to verify peak evidence in a +/-window around each peak day."""
    needed = {"gen_allgroups", "comments", "groups_keep", "COMBINED_GROUPS"}
    if any(name not in globals() for name in needed):
        return None

    comments_daily = comments.copy()
    comments_daily["dt"] = pd.to_datetime(comments_daily["dt"], errors="coerce")
    comments_daily = comments_daily.dropna(subset=["dt"])
    comments_daily["day"] = comments_daily["dt"].dt.floor("D")

    if "comment_id" in comments_daily.columns:
        daily_N = comments_daily.groupby("day", as_index=False)["comment_id"].nunique().rename(columns={"comment_id": "N_day"})
    else:
        daily_N = comments_daily.groupby("day", as_index=False).size().rename(columns={"size": "N_day"})

    g_daily = gen_allgroups.copy()
    g_daily["dt"] = pd.to_datetime(g_daily["dt"], errors="coerce")
    g_daily = g_daily.dropna(subset=["dt", "generalized"])
    if "group_generalization_presence" in g_daily.columns:
        g_daily = g_daily[g_daily["group_generalization_presence"].isin([3, 4])]
    g_daily["day"] = g_daily["dt"].dt.floor("D")
    g_daily["month"] = g_daily["dt"].dt.to_period("M").dt.to_timestamp()

    if "comment_id" in g_daily.columns:
        g_daily = g_daily.drop_duplicates(subset=["comment_id", "generalized", "day"])

    if "comment_id" in g_daily.columns:
        daily_group = (
            g_daily.groupby(["generalized", "month", "day"], as_index=False)["comment_id"]
            .nunique()
            .rename(columns={"comment_id": "group_day_cnt"})
        )
    else:
        daily_group = (
            g_daily.groupby(["generalized", "month", "day"], as_index=False)
            .size()
            .rename(columns={"size": "group_day_cnt"})
        )

    daily_group = daily_group.merge(daily_N, on="day", how="left")
    daily_group["day_share"] = (
        pd.to_numeric(daily_group["group_day_cnt"], errors="coerce")
        / pd.to_numeric(daily_group["N_day"], errors="coerce")
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    base_comments = g_daily[[c for c in ["comment_id", "generalized", "dt"] if c in g_daily.columns]].drop_duplicates().copy()
    group_to_source = {
        g: (COMBINED_GROUPS[g] if g in COMBINED_GROUPS else [g])
        for g in groups_keep
    }
    return {
        "daily_group": daily_group,
        "base_comments": base_comments,
        "group_to_source": group_to_source,
    }


def _window_counts_for_peaks(group_name: str, peak_months: pd.Series, ctx: dict, window_days: int):
    """Return {peak_month_ts: n_comments_in_window} using same method as export cell."""
    if ctx is None or len(peak_months) == 0:
        return {}

    daily_group = ctx["daily_group"]
    base_comments = ctx["base_comments"]
    source_labels = ctx["group_to_source"].get(group_name, [group_name])

    out = {}
    months = pd.to_datetime(peak_months, errors="coerce").dropna().drop_duplicates()
    for m in months:
        cand = daily_group[(daily_group["generalized"].isin(source_labels)) & (daily_group["month"] == m)].copy()

        if cand.empty:
            peak_day = m + pd.offsets.Day(14)
        else:
            cand = cand.sort_values(["day_share", "group_day_cnt", "day"], ascending=[False, False, True])
            peak_day = cand.iloc[0]["day"]

        lo = peak_day - pd.Timedelta(days=window_days)
        hi = peak_day + pd.Timedelta(days=window_days)
        n_in_window = int(len(base_comments[
            (base_comments["generalized"].isin(source_labels)) &
            (base_comments["dt"] >= lo) &
            (base_comments["dt"] <= hi)
        ]))
        out[pd.to_datetime(m)] = n_in_window

    return out


window_evidence_ctx = _build_window_evidence_context() if PEAK_REQUIRE_WINDOW_EVIDENCE else None
if PEAK_REQUIRE_WINDOW_EVIDENCE and window_evidence_ctx is None:
    print("[peak evidence] Missing inputs for +/-window evidence filter; plotting without window-evidence filtering.")

# ---------- Plot ----------
groups = groups_keep
FIGURE_OUTDIR = Path("proportion_time_trends_by_group")
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

def _group_slug(name: str) -> str:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(name)).strip("_").lower()
    return slug or "group"

monthly_n_lookup = (
    tt[["month", DEN_COL]]
    .dropna(subset=["month", DEN_COL])
    .drop_duplicates(subset=["month"])
)

saved_paths = []

for g in groups:
    fig, ax = plt.subplots(
        #nrows=1,
        #ncols=2,
        figsize=(14.0, 6.5),
        dpi=300,
        #gridspec_kw={"width_ratios": [3.0, 2.0], "wspace": 0.04},
    )

    #ax, ax_tbl = axes

    sub = (
        plot_df.loc[plot_df["generalized"] == g, ["month", "hlutfall"]]
        .sort_values("month")
        .merge(monthly_n_lookup, on="month", how="left")
    )
    sub[DEN_COL] = pd.to_numeric(sub[DEN_COL], errors="coerce").fillna(0.0)
    peak_df = detect_local_peaks(sub)


    # faint toxicity overlay
    if tox_plot_df is not None:
        tox_sub = tox_plot_df.loc[tox_plot_df["generalized"] == g, ["month","tox_share"]].dropna().sort_values("month")
        if len(tox_sub):
            ax.plot(
                tox_sub["month"], tox_sub["tox_share"],
                color=_blend_with_white(TOX_COLOR, TOX_ALPHA), lw=TOX_LW,
                zorder=1, solid_joinstyle="round", linestyle=":"
            )

    # faint hate-speech overlay (red)
    if hate_plot_df is not None:
        hate_sub = hate_plot_df.loc[hate_plot_df["generalized"] == g, ["month","hate_share"]].dropna().sort_values("month")
        if len(hate_sub):
            ax.plot(
                hate_sub["month"], hate_sub["hate_share"],
                color=_blend_with_white(HATE_COLOR, HATE_ALPHA), lw=HATE_LW,
                zorder=1.1, solid_joinstyle="round", linestyle=(0, (4, 2))
            )
        # Main generalization line
        ax.plot(sub["month"], sub["hlutfall"], lw=RAW_LW, color=RAW_COLOR,
                alpha=1.0, antialiased=True, solid_joinstyle="round", zorder=2)

    # Mark detected peaks + show rank labels (1=largest absolute peak)
    peaks = peak_df.loc[peak_df["is_peak"]].copy()
    shown_peaks = peaks.dropna(subset=["abs_rank"]).copy()
    if len(peaks):
        if PEAK_SHOW_TOP_K is not None:
            shown_peaks = shown_peaks[shown_peaks["abs_rank"] <= int(PEAK_SHOW_TOP_K)]

        if PEAK_REQUIRE_WINDOW_EVIDENCE and window_evidence_ctx is not None and len(shown_peaks):
            win_counts = _window_counts_for_peaks(
                group_name=g,
                peak_months=shown_peaks["month"],
                ctx=window_evidence_ctx,
                window_days=int(PEAK_WINDOW_DAYS),
            )
            shown_peaks = shown_peaks.copy()
            shown_peaks["n_comments_in_window"] = shown_peaks["month"].map(lambda m: win_counts.get(pd.to_datetime(m), 0)).fillna(0).astype(int)
            shown_peaks = shown_peaks[shown_peaks["n_comments_in_window"] >= int(PEAK_WINDOW_MIN_COMMENTS)]

        if len(shown_peaks) and PEAK_RENUMBER_SHOWN:
            shown_peaks = shown_peaks.sort_values(["month", "abs_rank"]).copy()
            shown_peaks["display_rank"] = np.arange(1, len(shown_peaks) + 1)
        elif len(shown_peaks):
            shown_peaks = shown_peaks.copy()
            shown_peaks["display_rank"] = shown_peaks["abs_rank"].astype(int)

        if len(shown_peaks):
            ax.scatter(
                shown_peaks["month"], shown_peaks["hlutfall"],
                s=PEAK_MARKER_SIZE, facecolors=PEAK_MARKER_FACE, edgecolors=PEAK_MARKER_EDGE,
                linewidths=1.1, zorder=3
            )

        for _, r in shown_peaks.iterrows():
            ax.annotate(
                f"{int(r.get('display_rank', r['abs_rank']))}",
                xy=(r["month"], r["hlutfall"]),
                xytext=(0, PEAK_RANK_DY),
                textcoords="offset points",
                ha="center", va="bottom",
                fontsize=PEAK_RANK_FONTSIZE,
                color=PEAK_MARKER_EDGE,
                zorder=3.2,
            )

    if len(sub):
        xmin = sub["month"].min() - pd.Timedelta(days=15)
        xmax = sub["month"].max() + pd.Timedelta(days=15)
        ax.set_xlim(mdates.date2num(xmin), mdates.date2num(xmax))

    ymin, ymax = ax.get_ylim()
    ax.set_ylim(ymin, ymax + (ymax - ymin) * 0.05)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    #ax.grid(True, axis="y", color="#e0e0e0", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_title(g, fontsize=17)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))

    ax.xaxis.set_major_locator(YearLocator(2))
    ax.xaxis.set_major_formatter(DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(MonthLocator(bymonth=(4,7,10)))
    ax.tick_params(axis="x", which="minor", length=3, color="0.85")
    ax.tick_params(axis="x", which="major", labelsize=11)
    ax.tick_params(axis="y", which="major", labelsize=11)
    ax.set_ylabel("% of all comments", fontsize=12)
    ax.set_xlabel("Year", fontsize=12)

    #ax_tbl.axis("off")
    #add_event_table(ax_tbl, g, shown_peaks)

    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], color=RAW_COLOR, lw=RAW_LW, label="Mention share"),
        Line2D([0], [0], color=_blend_with_white(HATE_COLOR, HATE_ALPHA), lw=HATE_LW,
               linestyle=(0, (4, 2)), label="Hate speech"),
        Line2D([0], [0], color=_blend_with_white(TOX_COLOR, TOX_ALPHA), lw=TOX_LW,
               linestyle=":", label="Toxicity"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.03), frameon=False, fontsize=12)

    out_path = FIGURE_OUTDIR / f"proportion_of_comments_{_group_slug(g)}.png"
    plt.savefig(out_path, facecolor="white", format="png", bbox_inches="tight")
    plt.close(fig)
    saved_paths.append(str(out_path))

print(
    f"Peak method: k-neighbor local maxima (k={PEAK_NEIGHBOR_K}), min comments={PEAK_MIN_COMMENTS}, rank mode={PEAK_RANK_MODE}, count weight={PEAK_RANK_COUNT_WEIGHT:.2f}, shown top K={PEAK_SHOW_TOP_K}; events listed in right-side peak table"
)

print(f"Saved {len(saved_paths)} per-group figures to: {FIGURE_OUTDIR}")







In [ ]:
def make_composite(_GROUPS, _NROWS, _NCOLS, _FIGSIZE, _OUT_PATH):
    # =========================
    # Composite 2x2 figure: Islam, Women, LGBTQIA+, Foreigners/asylum seekers/refugees
    # Reuses all data and helpers defined in the per-group cell above.
    # =========================


    #BASE_GROUPS_KEEP = ["Women", "Men",
    #COMBINED_GROUPS = {"LGBTQIA+": [
    #        "Transgender identities", "Homosexuality", "Bisexuality",
    #        "Intersex/Gender minorities", "Queer (general/other)"],    
    
    #    "Disability": 
    #         ["Intellectual/Developmental disabilities", 
    #         "Physical disabilities and general disability terms",
    #         "Mental illnesses/disorders", "Addiction disorders"],


    COMPOSITE_GROUPS = _GROUPS


    #COMPOSITE_GROUPS = [
    #    "Islam", 
    #    "Judaism", 
    #    "Christianity", 
    #    "Religion (other)",
    #]


    #COMPOSITE_GROUPS = [
    #    "Foreigners, asylum seekers, refugees (general)",
    #    "Middle East (nationalities)", 
    #    "Eastern Europe (nationalities)", 
    #    "Icelanders",
    #    "Nationalities (other)",
    #    "Ethnicities"
    #]

    TOX_ALPHA = 0.55
    TOX_COLOR = "#2CA89A"
    fig, axes = plt.subplots(
        _NROWS, _NCOLS,
        figsize=_FIGSIZE,
        dpi=300,
        constrained_layout=True,
    )

    for ax, g in zip(axes.flat, COMPOSITE_GROUPS):
        sub = (
            plot_df.loc[plot_df["generalized"] == g, ["month", "hlutfall"]]
            .sort_values("month")
            .merge(monthly_n_lookup, on="month", how="left")
        )
        sub[DEN_COL] = pd.to_numeric(sub[DEN_COL], errors="coerce").fillna(0.0)
        peak_df = detect_local_peaks(sub)

        # faint toxicity overlay
        if tox_plot_df is not None:
            tox_sub = tox_plot_df.loc[tox_plot_df["generalized"] == g, ["month", "tox_share"]].dropna().sort_values("month")
            if len(tox_sub):
                ax.plot(
                    tox_sub["month"], tox_sub["tox_share"],
                    color=_blend_with_white(TOX_COLOR, TOX_ALPHA), lw=TOX_LW,
                    zorder=1, solid_joinstyle="round", linestyle=":"
                )

        # faint hate-speech overlay
        if hate_plot_df is not None:
            hate_sub = hate_plot_df.loc[hate_plot_df["generalized"] == g, ["month", "hate_share"]].dropna().sort_values("month")
            if len(hate_sub):
                ax.plot(
                    hate_sub["month"], hate_sub["hate_share"],
                    color=_blend_with_white(HATE_COLOR, HATE_ALPHA), lw=HATE_LW,
                    zorder=1.1, solid_joinstyle="round", linestyle=(0, (4, 2))
                )

        # main generalization line
        ax.plot(sub["month"], sub["hlutfall"], lw=RAW_LW, color=RAW_COLOR,
                alpha=1.0, antialiased=True, solid_joinstyle="round", zorder=2)

        # peak markers + rank labels
        peaks = peak_df.loc[peak_df["is_peak"]].copy()
        shown_peaks = peaks.dropna(subset=["abs_rank"]).copy()
        if len(peaks):
            if PEAK_SHOW_TOP_K is not None:
                shown_peaks = shown_peaks[shown_peaks["abs_rank"] <= int(PEAK_SHOW_TOP_K)]

            if PEAK_REQUIRE_WINDOW_EVIDENCE and window_evidence_ctx is not None and len(shown_peaks):
                win_counts = _window_counts_for_peaks(
                    group_name=g,
                    peak_months=shown_peaks["month"],
                    ctx=window_evidence_ctx,
                    window_days=int(PEAK_WINDOW_DAYS),
                )
                shown_peaks = shown_peaks.copy()
                shown_peaks["n_comments_in_window"] = shown_peaks["month"].map(
                    lambda m: win_counts.get(pd.to_datetime(m), 0)
                ).fillna(0).astype(int)
                shown_peaks = shown_peaks[shown_peaks["n_comments_in_window"] >= int(PEAK_WINDOW_MIN_COMMENTS)]

            if len(shown_peaks) and PEAK_RENUMBER_SHOWN:
                shown_peaks = shown_peaks.sort_values(["month", "abs_rank"]).copy()
                shown_peaks["display_rank"] = np.arange(1, len(shown_peaks) + 1)
            elif len(shown_peaks):
                shown_peaks = shown_peaks.copy()
                shown_peaks["display_rank"] = shown_peaks["abs_rank"].astype(int)

            if len(shown_peaks):
                ax.scatter(
                    shown_peaks["month"], shown_peaks["hlutfall"],
                    s=PEAK_MARKER_SIZE, facecolors=PEAK_MARKER_FACE, edgecolors=PEAK_MARKER_EDGE,
                    linewidths=1.1, zorder=3
                )

            for _, r in shown_peaks.iterrows():
                ax.annotate(
                    f"{int(r.get('display_rank', r['abs_rank']))}",
                    xy=(r["month"], r["hlutfall"]),
                    xytext=(0, PEAK_RANK_DY),
                    textcoords="offset points",
                    ha="center", va="bottom",
                    fontsize=PEAK_RANK_FONTSIZE,
                    color=PEAK_MARKER_EDGE,
                    zorder=3.2,
                )

        if len(sub):
            xmin = sub["month"].min() - pd.Timedelta(days=15)
            xmax = sub["month"].max() + pd.Timedelta(days=15)
            ax.set_xlim(mdates.date2num(xmin), mdates.date2num(xmax))

        ymin, ymax = ax.get_ylim()
        ax.set_ylim(ymin, ymax + (ymax - ymin) * 0.05)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.spines["bottom"].set_visible(False)
        ax.set_axisbelow(True)
        ax.text(0.5, 1.02, g, transform=ax.transAxes, fontsize=17, va="bottom", ha="center")
        # per-panel labels
        idx = list(axes.flat).index(ax)
        if idx % _NCOLS == 0:
            ax.set_ylabel("% of all comments", fontsize=12)
        if idx >= len(COMPOSITE_GROUPS) - _NCOLS:
            ax.set_xlabel("Year", fontsize=12)
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=1))
        ax.xaxis.set_major_locator(YearLocator(2))
        ax.xaxis.set_major_formatter(DateFormatter("%Y"))
        ax.xaxis.set_minor_locator(MonthLocator(bymonth=(4, 7, 10)))
        ax.tick_params(axis="x", which="minor", length=3, color="0.85")
        ax.tick_params(axis="x", which="major", labelsize=11)
        ax.tick_params(axis="y", which="major", labelsize=11)


    from matplotlib.lines import Line2D
    legend_handles = [
        Line2D([0], [0], color=RAW_COLOR, lw=RAW_LW,
               label="Mention share"),
        Line2D([0], [0], color=_blend_with_white(HATE_COLOR, HATE_ALPHA), lw=HATE_LW,
               linestyle=(0, (4, 2)), label="Hate speech"),
        Line2D([0], [0], color=_blend_with_white(TOX_COLOR, TOX_ALPHA), lw=TOX_LW, linestyle=":",
               label="Toxicity"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3,
               bbox_to_anchor=(0.5, -0.03), frameon=False, fontsize=14)

    out_path = _OUT_PATH
    plt.savefig(out_path, facecolor="white", format="png", bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")



# Figure 3: nationality and ethnicity; Figure 4: religion; Figure 5: gender
make_composite(["Foreigners, asylum seekers, refugees (general)", "Middle East (nationalities)",
                "Eastern Europe (nationalities)", "Icelanders", "Nationalities (other)", "Ethnicities"],
               3, 2, (24, 14), "fig3_composite_nationality_ethnicity.png")
make_composite(["Islam", "Judaism", "Christianity", "Religion (other)"],
               2, 2, (24, 14), "fig4_composite_religion.png")
make_composite(["Women", "Men"], 1, 2, (24, 7), "proportion_composite_women_men.png")


In [ ]:
# =========================
# Peak-day anchoring + +/-15 day extraction windows
# Uses current peak logic from detect_local_peaks(...)
# Exports one CSV per plotted main group (base + combined)
# =========================

import re
from pathlib import Path
import pandas as pd
import numpy as np

WINDOW_DAYS = 15
# Keep export filtering consistent with detection threshold.
MIN_COMMENTS_FOR_USABLE_PEAK = int(globals().get("PEAK_MIN_COMMENTS", 5))
OUTPUT_DIR = Path(f"peak_windows_{WINDOW_DAYS}d_by_group")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

required = ["plot_df", "tt", "groups_keep", "detect_local_peaks", "gen_allgroups", "comments", "DEN_COL"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError(f"Missing required objects: {missing}. Run the peak-detection plotting cell first.")

def _slug(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_").lower()
    return s or "group"

# 1) Rebuild monthly peaks with the current detector + current top-k display rule (if set)
monthly_n_lookup = (
    tt[["month", DEN_COL]]
    .dropna(subset=["month", DEN_COL])
    .drop_duplicates(subset=["month"])
)

peak_rows = []
for g in groups_keep:
    sub = (
        plot_df.loc[plot_df["generalized"] == g, ["month", "hlutfall"]]
        .sort_values("month")
        .merge(monthly_n_lookup, on="month", how="left")
    )
    sub[DEN_COL] = pd.to_numeric(sub[DEN_COL], errors="coerce").fillna(0.0)

    peak_df = detect_local_peaks(sub)
    peaks = peak_df.loc[peak_df["is_peak"], ["month", "hlutfall", "abs_rank"]].copy()

    # Keep extraction aligned with what is shown on charts if top-k is enabled.
    if "PEAK_SHOW_TOP_K" in globals() and PEAK_SHOW_TOP_K is not None:
        peaks = peaks[pd.to_numeric(peaks["abs_rank"], errors="coerce") <= int(PEAK_SHOW_TOP_K)]

    peaks["generalized"] = g
    peak_rows.append(peaks)

peak_catalog = pd.concat(peak_rows, ignore_index=True) if peak_rows else pd.DataFrame()
if peak_catalog.empty:
    raise ValueError("No peaks detected with current settings.")

peak_catalog = peak_catalog.rename(columns={"month": "peak_month", "hlutfall": "peak_value"})
peak_catalog["peak_month"] = pd.to_datetime(peak_catalog["peak_month"], errors="coerce")
peak_catalog["abs_rank"] = pd.to_numeric(peak_catalog["abs_rank"], errors="coerce")
peak_catalog = peak_catalog.dropna(subset=["peak_month", "abs_rank"])
peak_catalog["abs_rank"] = peak_catalog["abs_rank"].astype(int)

# Use rank-based id so IDs match plotted labels.
peak_catalog["peak_id"] = peak_catalog["abs_rank"]
peak_catalog = peak_catalog.sort_values(["generalized", "peak_id", "peak_month"]).reset_index(drop=True)

# Map plotted groups to source labels in gen_allgroups (combined groups -> component labels).
GROUP_TO_SOURCE_LABELS = {
    g: (COMBINED_GROUPS[g] if ("COMBINED_GROUPS" in globals() and g in COMBINED_GROUPS) else [g])
    for g in groups_keep
}

# 2) Build daily group shares and select one peak day inside each peak month
comments_daily = comments.copy()
comments_daily["dt"] = pd.to_datetime(comments_daily["dt"], errors="coerce")
comments_daily = comments_daily.dropna(subset=["dt"])
comments_daily["day"] = comments_daily["dt"].dt.floor("D")

if "comment_id" in comments_daily.columns:
    daily_N = comments_daily.groupby("day", as_index=False)["comment_id"].nunique().rename(columns={"comment_id": "N_day"})
else:
    daily_N = comments_daily.groupby("day", as_index=False).size().rename(columns={"size": "N_day"})

g_daily = gen_allgroups.copy()
g_daily["dt"] = pd.to_datetime(g_daily["dt"], errors="coerce")
g_daily = g_daily.dropna(subset=["dt", "generalized"])
g_daily = g_daily[g_daily["group_generalization_presence"].isin([3, 4])]
g_daily["day"] = g_daily["dt"].dt.floor("D")
g_daily["month"] = g_daily["dt"].dt.to_period("M").dt.to_timestamp()

# Avoid counting same comment multiple times for same group/day.
if "comment_id" in g_daily.columns:
    g_daily = g_daily.drop_duplicates(subset=["comment_id", "generalized", "day"])

if "comment_id" in g_daily.columns:
    daily_group = (
        g_daily.groupby(["generalized", "month", "day"], as_index=False)["comment_id"]
        .nunique()
        .rename(columns={"comment_id": "group_day_cnt"})
    )
else:
    daily_group = (
        g_daily.groupby(["generalized", "month", "day"], as_index=False)
        .size()
        .rename(columns={"size": "group_day_cnt"})
    )

daily_group = daily_group.merge(daily_N, on="day", how="left")
daily_group["day_share"] = (
    pd.to_numeric(daily_group["group_day_cnt"], errors="coerce")
    / pd.to_numeric(daily_group["N_day"], errors="coerce")
).replace([np.inf, -np.inf], np.nan).fillna(0.0)

peak_catalog["peak_day"] = pd.NaT
peak_catalog["peak_day_share"] = np.nan
peak_catalog["peak_day_count"] = np.nan

for idx, row in peak_catalog.iterrows():
    g = row["generalized"]
    m = row["peak_month"]

    source_labels = GROUP_TO_SOURCE_LABELS.get(g, [g])
    cand = daily_group[(daily_group["generalized"].isin(source_labels)) & (daily_group["month"] == m)].copy()

    if cand.empty:
        # Monthly peaks are month-level; when daily rows are missing we anchor to mid-month.
        peak_catalog.at[idx, "peak_day"] = m + pd.offsets.Day(14)
        continue

    # Pick day with highest daily share; tie-break by higher count, then earliest day.
    cand = cand.sort_values(["day_share", "group_day_cnt", "day"], ascending=[False, False, True])
    best = cand.iloc[0]
    peak_catalog.at[idx, "peak_day"] = best["day"]
    peak_catalog.at[idx, "peak_day_share"] = float(best["day_share"])
    peak_catalog.at[idx, "peak_day_count"] = float(best["group_day_cnt"])

# 3) Build +/-15 day windows around each peak day
base_comments = g_daily[[c for c in ["comment_id", "generalized", "dt"] if c in g_daily.columns]].drop_duplicates().copy()

window_rows = []
window_meta_rows = []
for _, row in peak_catalog.iterrows():
    lo = row["peak_day"] - pd.Timedelta(days=WINDOW_DAYS)
    hi = row["peak_day"] + pd.Timedelta(days=WINDOW_DAYS)

    source_labels = GROUP_TO_SOURCE_LABELS.get(row["generalized"], [row["generalized"]])
    m = base_comments[
        (base_comments["generalized"].isin(source_labels)) &
        (base_comments["dt"] >= lo) &
        (base_comments["dt"] <= hi)
    ].copy()

    n_in_window = len(m)
    window_meta_rows.append({
        "generalized": row["generalized"],
        "peak_id": row["peak_id"],
        "peak_month": row["peak_month"],
        "peak_day": row["peak_day"],
        "window_start": lo,
        "window_end": hi,
        "n_comments_in_window": n_in_window,
    })

    if n_in_window > 0:
        m["generalized_source"] = m["generalized"]
        m["generalized"] = row["generalized"]
        m["peak_id"] = row["peak_id"]
        m["peak_month"] = row["peak_month"]
        m["peak_day"] = row["peak_day"]
        m["window_start"] = lo
        m["window_end"] = hi
        m["n_comments_in_window"] = n_in_window
        window_rows.append(m)

peak_windows = pd.DataFrame(window_meta_rows)
peak_catalog = peak_catalog.merge(
    peak_windows[["generalized", "peak_id", "n_comments_in_window", "window_start", "window_end"]],
    on=["generalized", "peak_id"],
    how="left"
)
peak_catalog["n_comments_in_window"] = peak_catalog["n_comments_in_window"].fillna(0).astype(int)

# Keep audit copy, then keep only peaks with comment evidence in window.
detected_peak_catalog = peak_catalog.copy()
peak_catalog = peak_catalog[peak_catalog["n_comments_in_window"] >= MIN_COMMENTS_FOR_USABLE_PEAK].copy()

peak_comments = pd.concat(window_rows, ignore_index=True) if window_rows else pd.DataFrame(columns=[
    "comment_id", "generalized", "dt", "peak_id", "peak_month", "peak_day", "window_start", "window_end", "n_comments_in_window"
])

if not peak_comments.empty and not peak_catalog.empty:
    peak_comments = peak_comments.merge(
        peak_catalog[["generalized", "peak_id"]].drop_duplicates(),
        on=["generalized", "peak_id"],
        how="inner"
    )
elif peak_catalog.empty:
    peak_comments = peak_comments.iloc[0:0].copy()

# Optional enrichment from comments table (align output schema with legacy peak-comments files)
if not peak_comments.empty and "comment_id" in peak_comments.columns:
    comment_cols = [c for c in ["comment_id", "blog_id", "dt", "comment_text", "comment", "text"] if c in comments.columns]
    if comment_cols:
        peak_comments = peak_comments.merge(
            comments[comment_cols].drop_duplicates(subset=["comment_id"]),
            on="comment_id",
            how="left",
            suffixes=("", "_comment")
        )

# Canonical text fields
if not peak_comments.empty:
    # Preserve the original comment timestamp column from comments table.
    if "dt_comment" not in peak_comments.columns:
        for cand in ["dt_comment", "dt_y", "dt_comment_comment"]:
            if cand in peak_comments.columns:
                peak_comments["dt_comment"] = peak_comments[cand]
                break

    # Build single comment_text column with fallback precedence.
    if "comment_text" not in peak_comments.columns:
        peak_comments["comment_text"] = pd.NA
    for col in ["comment_text", "comment", "text", "comment_text_comment", "comment_comment", "text_comment"]:
        if col in peak_comments.columns:
            peak_comments["comment_text"] = peak_comments["comment_text"].fillna(peak_comments[col])

# Add blog_text via in-memory blogs DataFrame if available.
if not peak_comments.empty:
    peak_comments["blog_text"] = pd.NA

if not peak_comments.empty and "blogs" in globals() and "blog_id" in peak_comments.columns:
    b = blogs.copy()
    if "id" in b.columns:
        key = "id"
    elif "blog_id" in b.columns:
        key = "blog_id"
    else:
        key = None

    if key is not None and "blog_content" in b.columns:
        bj = b[[key, "blog_content"]].drop_duplicates(subset=[key]).copy()
        bj["blog_id_join"] = pd.to_numeric(bj[key], errors="coerce")
        peak_comments["blog_id_join"] = pd.to_numeric(peak_comments["blog_id"], errors="coerce")
        peak_comments = peak_comments.merge(
            bj.drop(columns=[key]),
            on="blog_id_join",
            how="left"
        )
        peak_comments["blog_text"] = peak_comments["blog_text"].fillna(peak_comments["blog_content"])
        peak_comments = peak_comments.drop(columns=[c for c in ["blog_id_join", "blog_content"] if c in peak_comments.columns])

# Fallback: fetch blog_content from SQLite by comment_id -> blog_id.
if not peak_comments.empty and peak_comments["blog_text"].isna().all() and "comment_id" in peak_comments.columns:
    import sqlite3
    from pathlib import Path

    db_path = Path(DB_PATH)
    if db_path.exists():
        with sqlite3.connect(str(db_path)) as conn:
            m = pd.read_sql_query(
                """
                SELECT c.id AS comment_id, c.blog_id AS blog_id_db, b.blog_content
                FROM comments c
                LEFT JOIN blogs b ON c.blog_id = b.id
                """,
                conn,
            )
        m["comment_id"] = pd.to_numeric(m["comment_id"], errors="coerce")
        m = m.dropna(subset=["comment_id"]).drop_duplicates(subset=["comment_id"])
        peak_comments["comment_id"] = pd.to_numeric(peak_comments["comment_id"], errors="coerce")
        peak_comments = peak_comments.merge(m, on="comment_id", how="left")
        if "blog_content" in peak_comments.columns:
            peak_comments["blog_text"] = peak_comments["blog_text"].fillna(peak_comments["blog_content"])
        peak_comments = peak_comments.drop(columns=[c for c in ["blog_content", "blog_id_db"] if c in peak_comments.columns])

# Enforce column order similar to legacy exports (keep extras after canonical set).
canonical_cols = [
    "comment_id", "generalized", "dt", "generalized_source", "peak_id", "peak_month", "peak_day",
    "window_start", "window_end", "n_comments_in_window", "blog_id", "dt_comment", "comment_text", "blog_text"
]
for c in canonical_cols:
    if c not in peak_comments.columns:
        peak_comments[c] = pd.NA
extra_cols = [c for c in peak_comments.columns if c not in canonical_cols]
peak_comments = peak_comments[canonical_cols + extra_cols]

# Exports
catalog_path = OUTPUT_DIR / f"peak_catalog_{WINDOW_DAYS}d_all_groups.csv"
comments_path = OUTPUT_DIR / f"peak_comments_{WINDOW_DAYS}d_all_groups.csv"
detected_path = OUTPUT_DIR / f"detected_peak_catalog_{WINDOW_DAYS}d_all_groups.csv"

peak_catalog.to_csv(catalog_path, index=False)
peak_comments.to_csv(comments_path, index=False)
detected_peak_catalog.to_csv(detected_path, index=False)

written = []
for g in groups_keep:
    g_slug = _slug(g)
    g_comments = peak_comments[peak_comments["generalized"] == g].copy()
    g_catalog = peak_catalog[peak_catalog["generalized"] == g].copy()

    g_comments_path = OUTPUT_DIR / f"peak_comments_{WINDOW_DAYS}d_{g_slug}.csv"
    g_catalog_path = OUTPUT_DIR / f"peak_catalog_{WINDOW_DAYS}d_{g_slug}.csv"

    g_comments.to_csv(g_comments_path, index=False)
    g_catalog.to_csv(g_catalog_path, index=False)
    written.append((g, len(g_comments), len(g_catalog), g_comments_path.name, g_catalog_path.name))

print(f"Detected peaks (before evidence filter): {len(detected_peak_catalog)}")
print(f"Usable peaks (>= {MIN_COMMENTS_FOR_USABLE_PEAK} comments in window): {len(peak_catalog)}")
print(f"Windowed peak comments (+/-{WINDOW_DAYS} days): {len(peak_comments)}")
print(f"Export directory: {OUTPUT_DIR}")
print(f"Wrote combined files: {catalog_path.name}, {comments_path.name}, {detected_path.name}")
print("Per-group files:")
for g, n_comments, n_peaks, cfile, pfile in written:
    print(f" - {g}: {n_peaks} peaks, {n_comments} comments -> {pfile}, {cfile}")






In [ ]:
# =============================================================
# DATA UNDERLYING THE FIGURES (numbering as in the manuscript)
# One CSV per figure (group) is written to ../figure_data/
# =============================================================
FIG_DATA = Path(FIGURE_DATA_DIR)
FIG_DATA.mkdir(parents=True, exist_ok=True)


def save_csv(df, name):
    df.to_csv(FIG_DATA / name, index=False)
    print(f"Saved: {name} ({len(df)} rows)")


# Figure 2: weekly means of within-user centred scores (input to the spline fits)
save_csv(
    wk.rename(columns={"vika": "week", "emotion": "dimension", "mu": "mean_centred_score", "n": "n_comments"}),
    "fig2_weekly_centred_means.csv",
)

# Figures 3-7: monthly mention share and the shares also classified as hate speech / toxic
groups_by_figure = {
    3: ["Foreigners, asylum seekers, refugees (general)", "Middle East (nationalities)",
        "Eastern Europe (nationalities)", "Icelanders", "Nationalities (other)", "Ethnicities"],
    4: ["Islam", "Judaism", "Christianity", "Religion (other)"],
    5: ["Women", "Men"],
    6: ["LGBTQIA+"],
    7: ["Disability"],
}
parts = []
for fig_no, groups in groups_by_figure.items():
    for g in groups:
        m = plot_df.loc[plot_df["generalized"] == g, ["month", "hlutfall"]].rename(columns={"hlutfall": "mention_share"})
        m = m.merge(hate_plot_df.loc[hate_plot_df["generalized"] == g, ["month", "hate_share"]], on="month", how="left")
        m = m.merge(
            tox_plot_df.loc[tox_plot_df["generalized"] == g, ["month", "tox_share"]].rename(columns={"tox_share": "toxicity_share"}),
            on="month", how="left",
        )
        m.insert(0, "group", g)
        m.insert(0, "figure", fig_no)
        parts.append(m)
save_csv(pd.concat(parts, ignore_index=True), "fig3-7_monthly_group_shares.csv")

# Detected peaks shown as numbered markers in Figures 3-7 (supplementary tables S3-S16)
save_csv(pd.read_csv("peak_windows_15d_by_group/peak_catalog_15d_all_groups.csv"), "fig3-7_peak_catalog.csv")


def prevalence(gen):
    d = gen.loc[gen["group_generalization_presence"] > 2, ["generalized", "hate_speech_presence", "toxicity"]].copy()
    h = pd.to_numeric(d["hate_speech_presence"], errors="coerce").fillna(0)
    t = pd.to_numeric(d["toxicity"], errors="coerce").fillna(0)
    d = d.assign(hate3=(h == 3), hate4=(h == 4), toxic3=(t == 3), toxic4=(t == 4))
    res = (
        d.groupby("generalized", observed=True)
        .agg(n_total=("hate3", "size"), hate3=("hate3", "sum"), hate4=("hate4", "sum"),
             toxic3=("toxic3", "sum"), toxic4=("toxic4", "sum"))
        .reset_index()
        .rename(columns={"generalized": "group"})
    )
    res["prop_hate"] = (res["hate3"] + res["hate4"]) / res["n_total"]
    res["prop_toxic"] = (res["toxic3"] + res["toxic4"]) / res["n_total"]
    return res.sort_values("n_total", ascending=False)


# Figures 8-9 (protected-trait categories) and S1-S2 (named groups); groups with n < 5 are not plotted
save_csv(prevalence(gen_grouped), "fig8-9_prevalence_protected_traits.csv")
save_csv(prevalence(gen_allgroups), "figS1-S2_prevalence_named_groups.csv")
